Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Step 11 v2：Cox、RSF、RNN、LSTM-v2 开发集 OOF 统一校准与正式比较
=================================================================

目的
----
1. 读取四个模型完全独立的开发集五折 OOF 预测：
   - Super Landmark Cox
   - Pooled Landmark RSF
   - RNN survival
   - LSTM-v2 survival
2. 所有模型采用完全相同的五折交叉拟合离散风险校准：
      logit(h_cal,j) = alpha_j + beta * logit(h_raw,j)
   其中：
      alpha_j = 10 个未来半年区间特异截距
      beta    = 一个共同斜率
3. 在完全相同的患者、Landmark、结局和预测时间上评价：
   - 未来 5 年 Uno C-index
   - 6–60 月动态 AUC 及 iAUC
   - 6–60 月 Brier score 及 IBS
   - 1、3、5 年 AUC / Brier
   - 1、3、5 年总体和十分位校准
   - 1、3、5 年生存 DCA
4. 患者级 paired bootstrap：
   - 同一个 bootstrap 抽样同时用于全部模型、全部 Landmark
   - 默认 1000 次
   - 支持断点续跑
5. 同时报告：
   - 6 个 Landmark 等权平均：0、1、2、3、4、5 年
   - 主文 4 个 Landmark 等权平均：0、1、3、5 年
6. 预设全部必要配对比较：
   - RSF - Cox
   - RNN - Cox
   - LSTM-v2 - Cox
   - RSF - RNN
   - LSTM-v2 - RSF
   - LSTM-v2 - RNN
7. 保存全开发集 OOF 拟合的最终校准器，供后续冻结模型后使用。
8. 全程不读取 9,574 人锁定内部测试集。

重要说明
--------
- 本步骤不训练 Cox / RSF / RNN / LSTM。
- 本步骤只读取已经生成的 OOF 预测并做统一统计评价。
- LSTM-v2 路径对应已经完成的 Step 10E：
  rolling_5y_step10e_lstm_v2_final_oof_selected_existing_trials
- Bootstrap 默认 1000 次；调试时可临时：
    CKD_BOOTSTRAP_REPS=50 python Step11_v2_....py
  正式论文结果仍建议 1000 次。
"""

from __future__ import annotations

import hashlib
import json
import os
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import spearmanr
from sksurv.metrics import (
    brier_score,
    concordance_index_ipcw,
    cumulative_dynamic_auc,
    integrated_brier_score,
)
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.util import Surv


# =============================================================================
# 1. 固定配置
# =============================================================================

PROJECT_DIR = Path(
    os.getenv(
        "CKD_LSTM_PROJECT_DIR",
        "__CKD_WORKDIR__",
    )
)

STEP6_DIR = (
    PROJECT_DIR
    / "rolling_5y_step6_super_landmark_data"
)

COX_DIR = (
    PROJECT_DIR
    / "rolling_5y_step7_unpenalized_cox_fixed95_v5"
)

RSF_DIR = (
    PROJECT_DIR
    / "rolling_5y_step8_pooled_landmark_rsf_resume_v2"
    / "final_oof"
)

RNN_DIR = (
    PROJECT_DIR
    / "rolling_5y_step9b_rnn_tune_resume_v1"
    / "final_oof"
)

LSTM_V2_DIR = (
    PROJECT_DIR
    / "rolling_5y_step10e_lstm_v2_final_oof_selected_existing_trials"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "rolling_5y_step11_v2_four_model_oof_comparison_lstm_v2"
)

FIGURE_DIR = OUTPUT_DIR / "figures"
BOOTSTRAP_DIR = OUTPUT_DIR / "bootstrap_checkpoints"

for directory in [
    OUTPUT_DIR,
    FIGURE_DIR,
    BOOTSTRAP_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

BOOTSTRAP_REPS = int(
    os.getenv(
        "CKD_BOOTSTRAP_REPS",
        "1000",
    )
)

RANDOM_SEED = int(
    os.getenv(
        "CKD_RANDOM_SEED",
        "20260804",
    )
)

BOOTSTRAP_SAVE_EVERY = 10
MINIMUM_VALID_BOOTSTRAP_RATE = 0.80

N_SPLITS = 5
EXPECTED_DEVELOPMENT_N = 22337
EXPECTED_LANDMARK_N = 6
EXPECTED_INTERVAL_N = 10
EXPECTED_LONG_N = 100122

LANDMARK_MONTHS = np.asarray(
    [0, 12, 24, 36, 48, 60],
    dtype=np.float64,
)

FUTURE_END_MONTHS = np.arange(
    6,
    61,
    6,
    dtype=np.float64,
)

# sksurv 要求评价时间严格小于最大随访时间。
# 第 10 个风险值仍表示“未来 60 月风险”，
# 但数值评价时使用 59.999 月。
METRIC_TIMES = np.asarray(
    [
        6,
        12,
        18,
        24,
        30,
        36,
        42,
        48,
        54,
        59.999,
    ],
    dtype=np.float64,
)

PRIMARY_LANDMARK_MONTHS = np.asarray(
    [0, 12, 36, 60],
    dtype=np.float64,
)

PRIMARY_LANDMARK_POSITIONS = np.asarray(
    [0, 1, 3, 5],
    dtype=np.int64,
)

ALL_LANDMARK_POSITIONS = np.arange(
    EXPECTED_LANDMARK_N,
    dtype=np.int64,
)

MODEL_ORDER = [
    "Cox",
    "RSF",
    "RNN",
    "LSTM-v2",
]

MODEL_INDEX = {
    model_name: index
    for index, model_name in enumerate(
        MODEL_ORDER
    )
}

PAIRWISE_COMPARISONS = [
    ("RSF", "Cox"),
    ("RNN", "Cox"),
    ("LSTM-v2", "Cox"),
    ("RSF", "RNN"),
    ("LSTM-v2", "RSF"),
    ("LSTM-v2", "RNN"),
]

SUMMARY_SCOPES = {
    "six_landmark_equal_weight_mean": (
        ALL_LANDMARK_POSITIONS
    ),
    "primary_0_1_3_5y_landmark_equal_weight_mean": (
        PRIMARY_LANDMARK_POSITIONS
    ),
}

DCA_THRESHOLD_RANGES = {
    12.0: (0.002, 0.030, 80),
    36.0: (0.005, 0.100, 80),
    60.0: (0.010, 0.200, 80),
}

CALIBRATION_RIDGE = 1e-6
EPS = 1e-7
DPI = 300

METRIC_NAMES = [
    "uno_c_index_5y",
    "integrated_dynamic_auc",
    "integrated_brier",
]

HIGHER_IS_BETTER = {
    "uno_c_index_5y": True,
    "integrated_dynamic_auc": True,
    "integrated_brier": False,
}

if BOOTSTRAP_REPS < 1:
    raise ValueError(
        "BOOTSTRAP_REPS 必须为正整数。"
    )


# =============================================================================
# 2. 通用工具
# =============================================================================

def format_duration(
    seconds: float,
) -> str:
    seconds = max(
        0,
        int(round(float(seconds))),
    )
    hours, remainder = divmod(
        seconds,
        3600,
    )
    minutes, seconds = divmod(
        remainder,
        60,
    )

    if hours:
        return (
            f"{hours}小时"
            f"{minutes:02d}分"
            f"{seconds:02d}秒"
        )
    if minutes:
        return (
            f"{minutes}分"
            f"{seconds:02d}秒"
        )
    return f"{seconds}秒"


def save_json(
    value: Any,
    path: Path,
) -> None:
    path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )


def require_files(
    paths: list[Path],
) -> None:
    missing = [
        str(path)
        for path in paths
        if not path.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "以下必要输入文件不存在：\n"
            + "\n".join(missing)
        )


def display_table(
    frame: pd.DataFrame,
    title: str,
) -> None:
    print(
        "\n"
        + "=" * 110
    )
    print(title)
    print("=" * 110)

    try:
        from IPython.display import display
        display(frame)
    except ImportError:
        print(
            frame.to_string(
                index=False
            )
        )


def save_figure(
    fig: plt.Figure,
    filename_stem: str,
) -> None:
    png_path = (
        FIGURE_DIR
        / f"{filename_stem}.png"
    )
    pdf_path = (
        FIGURE_DIR
        / f"{filename_stem}.pdf"
    )

    fig.savefig(
        png_path,
        dpi=DPI,
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)


def build_survival_array(
    event: np.ndarray,
    time_month: np.ndarray,
) -> np.ndarray:
    event = np.asarray(
        event,
        dtype=bool,
    )

    time_month = np.asarray(
        time_month,
        dtype=np.float64,
    )

    if event.shape != time_month.shape:
        raise ValueError(
            "事件和随访时间形状不一致。"
        )

    if not np.isfinite(
        time_month
    ).all():
        raise ValueError(
            "随访时间存在 NaN 或无穷值。"
        )

    if np.any(
        time_month <= 0
    ):
        raise ValueError(
            "Landmark 后随访时间必须 > 0。"
        )

    return Surv.from_arrays(
        event=event,
        time=time_month,
    )


def percentile_interval(
    values: np.ndarray,
    expected_n: int | None = None,
) -> tuple[float, float, int]:
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    valid = values[
        np.isfinite(values)
    ]

    if expected_n is None:
        expected_n = len(values)

    required_n = int(
        np.ceil(
            expected_n
            * MINIMUM_VALID_BOOTSTRAP_RATE
        )
    )

    if len(valid) < required_n:
        raise RuntimeError(
            f"仅获得 {len(valid)}/{expected_n} "
            "个有效 Bootstrap 估计。"
        )

    return (
        float(
            np.percentile(
                valid,
                2.5,
            )
        ),
        float(
            np.percentile(
                valid,
                97.5,
            )
        ),
        int(len(valid)),
    )


def favorable_fraction(
    differences: np.ndarray,
    higher_is_better: bool,
) -> float:
    differences = np.asarray(
        differences,
        dtype=np.float64,
    )

    valid = differences[
        np.isfinite(differences)
    ]

    if len(valid) == 0:
        return np.nan

    if higher_is_better:
        return float(
            np.mean(
                valid > 0
            )
        )

    return float(
        np.mean(
            valid < 0
        )
    )


def two_sided_bootstrap_p(
    differences: np.ndarray,
) -> float:
    differences = np.asarray(
        differences,
        dtype=np.float64,
    )

    valid = differences[
        np.isfinite(differences)
    ]

    if len(valid) == 0:
        return np.nan

    p_lower = (
        np.mean(valid <= 0)
    )
    p_upper = (
        np.mean(valid >= 0)
    )

    return float(
        min(
            1.0,
            2.0
            * min(
                p_lower,
                p_upper,
            ),
        )
    )


def complete_row_mean(
    matrix: np.ndarray,
    positions: np.ndarray,
) -> np.ndarray:
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    subset = matrix[
        :,
        positions,
    ]

    valid = np.isfinite(
        subset
    ).all(
        axis=1
    )

    result = np.full(
        subset.shape[0],
        np.nan,
        dtype=np.float64,
    )

    result[
        valid
    ] = subset[
        valid
    ].mean(
        axis=1
    )

    return result


def safe_name(
    model_name: str,
) -> str:
    return (
        model_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


# =============================================================================
# 3. 读取 Step 6 标签、映射和四模型 OOF
# =============================================================================

required_files = [
    STEP6_DIR
    / "development_long_row_index_map.npy",

    STEP6_DIR
    / "development_future_event_long.npy",

    STEP6_DIR
    / "development_future_at_risk_long.npy",

    STEP6_DIR
    / "development_local_patient_idx_long.npy",

    STEP6_DIR
    / "development_fold_id_long.npy",

    STEP6_DIR
    / "development_landmark_index_long.npy",

    STEP6_DIR
    / "development_landmark_month_long.npy",

    STEP6_DIR
    / "development_analysis_time_month.npy",

    STEP6_DIR
    / "development_event_within_60m.npy",

    STEP6_DIR
    / "landmark_months.npy",

    STEP6_DIR
    / "future_end_months.npy",

    COX_DIR
    / "cox_oof_survival_long.npy",

    COX_DIR
    / "cox_oof_risk_long.npy",

    COX_DIR
    / "cox_oof_log_partial_hazard_long.npy",

    RSF_DIR
    / "rsf_oof_survival_long.npy",

    RSF_DIR
    / "rsf_oof_risk_long.npy",

    RNN_DIR
    / "rnn_oof_survival_long.npy",

    RNN_DIR
    / "rnn_oof_risk_long.npy",

    RNN_DIR
    / "rnn_summary.json",

    LSTM_V2_DIR
    / "lstm_v2_oof_survival_long.npy",

    LSTM_V2_DIR
    / "lstm_v2_oof_risk_long.npy",

    LSTM_V2_DIR
    / "lstm_v2_summary.json",
]

require_files(
    required_files
)

row_index_map = np.load(
    STEP6_DIR
    / "development_long_row_index_map.npy"
).astype(
    np.int32
)

future_event_long = np.load(
    STEP6_DIR
    / "development_future_event_long.npy"
).astype(
    np.float32
)

future_at_risk_long = np.load(
    STEP6_DIR
    / "development_future_at_risk_long.npy"
).astype(
    bool
)

local_patient_idx_long = np.load(
    STEP6_DIR
    / "development_local_patient_idx_long.npy"
).astype(
    np.int32
)

fold_id_long = np.load(
    STEP6_DIR
    / "development_fold_id_long.npy"
).astype(
    np.int8
)

landmark_index_long = np.load(
    STEP6_DIR
    / "development_landmark_index_long.npy"
).astype(
    np.int8
)

landmark_month_long = np.load(
    STEP6_DIR
    / "development_landmark_month_long.npy"
).astype(
    np.float64
)

analysis_time_month = np.load(
    STEP6_DIR
    / "development_analysis_time_month.npy"
).astype(
    np.float64
)

event_within_60m = np.load(
    STEP6_DIR
    / "development_event_within_60m.npy"
).astype(
    bool
)

saved_landmarks = np.load(
    STEP6_DIR
    / "landmark_months.npy"
).astype(
    np.float64
)

saved_future_ends = np.load(
    STEP6_DIR
    / "future_end_months.npy"
).astype(
    np.float64
)

model_survival_raw = {
    "Cox": np.load(
        COX_DIR
        / "cox_oof_survival_long.npy"
    ).astype(
        np.float32
    ),

    "RSF": np.load(
        RSF_DIR
        / "rsf_oof_survival_long.npy"
    ).astype(
        np.float32
    ),

    "RNN": np.load(
        RNN_DIR
        / "rnn_oof_survival_long.npy"
    ).astype(
        np.float32
    ),

    "LSTM-v2": np.load(
        LSTM_V2_DIR
        / "lstm_v2_oof_survival_long.npy"
    ).astype(
        np.float32
    ),
}

model_risk_raw = {
    "Cox": np.load(
        COX_DIR
        / "cox_oof_risk_long.npy"
    ).astype(
        np.float32
    ),

    "RSF": np.load(
        RSF_DIR
        / "rsf_oof_risk_long.npy"
    ).astype(
        np.float32
    ),

    "RNN": np.load(
        RNN_DIR
        / "rnn_oof_risk_long.npy"
    ).astype(
        np.float32
    ),

    "LSTM-v2": np.load(
        LSTM_V2_DIR
        / "lstm_v2_oof_risk_long.npy"
    ).astype(
        np.float32
    ),
}

cox_log_partial_hazard_long = np.load(
    COX_DIR
    / "cox_oof_log_partial_hazard_long.npy"
).astype(
    np.float32
)

rnn_summary = json.loads(
    (
        RNN_DIR
        / "rnn_summary.json"
    ).read_text(
        encoding="utf-8"
    )
)

lstm_v2_summary = json.loads(
    (
        LSTM_V2_DIR
        / "lstm_v2_summary.json"
    ).read_text(
        encoding="utf-8"
    )
)

if bool(
    rnn_summary.get(
        "locked_test_used",
        False,
    )
):
    raise ValueError(
        "RNN 正式 OOF 记录显示使用过锁定测试集。"
    )

if bool(
    lstm_v2_summary.get(
        "locked_test_used",
        False,
    )
):
    raise ValueError(
        "LSTM-v2 正式 OOF 记录显示使用过锁定测试集。"
    )


# =============================================================================
# 4. 输入一致性与 OOF 审计
# =============================================================================

expected_shapes = {
    "row_index_map": (
        EXPECTED_DEVELOPMENT_N,
        EXPECTED_LANDMARK_N,
    ),

    "future_event_long": (
        EXPECTED_LONG_N,
        EXPECTED_INTERVAL_N,
    ),

    "future_at_risk_long": (
        EXPECTED_LONG_N,
        EXPECTED_INTERVAL_N,
    ),

    "local_patient_idx_long": (
        EXPECTED_LONG_N,
    ),

    "fold_id_long": (
        EXPECTED_LONG_N,
    ),

    "landmark_index_long": (
        EXPECTED_LONG_N,
    ),

    "landmark_month_long": (
        EXPECTED_LONG_N,
    ),

    "analysis_time_month": (
        EXPECTED_LONG_N,
    ),

    "event_within_60m": (
        EXPECTED_LONG_N,
    ),
}

actual_arrays = {
    "row_index_map": row_index_map,
    "future_event_long": future_event_long,
    "future_at_risk_long": (
        future_at_risk_long
    ),
    "local_patient_idx_long": (
        local_patient_idx_long
    ),
    "fold_id_long": fold_id_long,
    "landmark_index_long": (
        landmark_index_long
    ),
    "landmark_month_long": (
        landmark_month_long
    ),
    "analysis_time_month": (
        analysis_time_month
    ),
    "event_within_60m": (
        event_within_60m
    ),
}

for name, expected_shape in (
    expected_shapes.items()
):
    actual_shape = (
        actual_arrays[name].shape
    )

    if (
        actual_shape
        != expected_shape
    ):
        raise ValueError(
            f"{name} 形状={actual_shape}，"
            f"预期={expected_shape}。"
        )

if not np.array_equal(
    saved_landmarks,
    LANDMARK_MONTHS,
):
    raise ValueError(
        "Step 6 Landmark 配置不一致。"
    )

if not np.array_equal(
    saved_future_ends,
    FUTURE_END_MONTHS,
):
    raise ValueError(
        "Step 6 未来预测区间配置不一致。"
    )

if not np.isin(
    fold_id_long,
    np.arange(
        N_SPLITS
    ),
).all():
    raise ValueError(
        "长格式 fold_id 必须全部为 0～4。"
    )

if not np.array_equal(
    landmark_month_long,
    LANDMARK_MONTHS[
        landmark_index_long
    ],
):
    raise ValueError(
        "Landmark 索引与月份不一致。"
    )

if (
    np.any(
        analysis_time_month
        <= 0
    )
    or not np.isfinite(
        analysis_time_month
    ).all()
):
    raise ValueError(
        "Landmark 后随访时间非法。"
    )

valid_map = (
    row_index_map
    >= 0
)

patient_grid = np.broadcast_to(
    np.arange(
        EXPECTED_DEVELOPMENT_N,
        dtype=np.int32,
    )[:, None],
    row_index_map.shape,
)

landmark_grid = np.broadcast_to(
    np.arange(
        EXPECTED_LANDMARK_N,
        dtype=np.int32,
    )[None, :],
    row_index_map.shape,
)

mapped_rows = row_index_map[
    valid_map
].astype(
    np.int64
)

if not np.array_equal(
    local_patient_idx_long[
        mapped_rows
    ],
    patient_grid[
        valid_map
    ],
):
    raise ValueError(
        "row_index_map 与 "
        "local_patient_idx_long 不一致。"
    )

if not np.array_equal(
    landmark_index_long[
        mapped_rows
    ],
    landmark_grid[
        valid_map
    ],
):
    raise ValueError(
        "row_index_map 与 "
        "landmark_index_long 不一致。"
    )

for model_name in MODEL_ORDER:
    survival = (
        model_survival_raw[
            model_name
        ]
    )

    risk = (
        model_risk_raw[
            model_name
        ]
    )

    expected_shape = (
        EXPECTED_LONG_N,
        EXPECTED_INTERVAL_N,
    )

    if (
        survival.shape
        != expected_shape
    ):
        raise ValueError(
            f"{model_name} survival "
            f"形状={survival.shape}，"
            f"预期={expected_shape}。"
        )

    if (
        risk.shape
        != expected_shape
    ):
        raise ValueError(
            f"{model_name} risk "
            f"形状={risk.shape}，"
            f"预期={expected_shape}。"
        )

    if (
        not np.isfinite(
            survival
        ).all()
        or not np.isfinite(
            risk
        ).all()
    ):
        raise ValueError(
            f"{model_name} OOF "
            "存在 NaN 或无穷值。"
        )

    if np.any(
        (survival < -EPS)
        | (survival > 1 + EPS)
    ):
        raise ValueError(
            f"{model_name} 生存概率超出 0～1。"
        )

    if np.any(
        (risk < -EPS)
        | (risk > 1 + EPS)
    ):
        raise ValueError(
            f"{model_name} 累积风险超出 0～1。"
        )

    survival_violation_n = int(
        np.sum(
            np.diff(
                survival,
                axis=1,
            )
            > 1e-6
        )
    )

    risk_violation_n = int(
        np.sum(
            np.diff(
                risk,
                axis=1,
            )
            < -1e-6
        )
    )

    if (
        survival_violation_n
        != 0
    ):
        raise ValueError(
            f"{model_name} 生存概率存在 "
            f"{survival_violation_n} "
            "处单调性违反。"
        )

    if (
        risk_violation_n
        != 0
    ):
        raise ValueError(
            f"{model_name} 累积风险存在 "
            f"{risk_violation_n} "
            "处单调性违反。"
        )

    if not np.allclose(
        risk,
        1.0 - survival,
        atol=2e-6,
    ):
        max_diff = float(
            np.max(
                np.abs(
                    risk
                    - (
                        1.0
                        - survival
                    )
                )
            )
        )

        raise ValueError(
            f"{model_name} risk != 1-survival，"
            f"最大差值={max_diff:.3e}。"
        )


# =============================================================================
# 5. Cox 排序审计
# =============================================================================

if not np.isfinite(
    cox_log_partial_hazard_long
).all():
    raise ValueError(
        "Cox log partial hazard "
        "存在 NaN 或无穷值。"
    )

cox_audit_rows: list[
    dict[str, Any]
] = []

for fold_id in range(
    N_SPLITS
):
    for (
        landmark_position,
        landmark_month,
    ) in enumerate(
        LANDMARK_MONTHS
    ):
        rows = np.where(
            (
                fold_id_long
                == fold_id
            )
            & (
                landmark_index_long
                == landmark_position
            )
        )[0]

        survival_reference = (
            build_survival_array(
                event_within_60m[
                    rows
                ],
                analysis_time_month[
                    rows
                ],
            )
        )

        risk_5y = (
            model_risk_raw[
                "Cox"
            ][
                rows,
                -1,
            ].astype(
                np.float64
            )
        )

        linear_score = (
            cox_log_partial_hazard_long[
                rows
            ].astype(
                np.float64
            )
        )

        rank_correlation = float(
            spearmanr(
                risk_5y,
                linear_score,
            ).statistic
        )

        c_risk = float(
            concordance_index_ipcw(
                survival_reference,
                survival_reference,
                risk_5y,
                tau=float(
                    METRIC_TIMES[
                        -1
                    ]
                ),
            )[0]
        )

        c_linear = float(
            concordance_index_ipcw(
                survival_reference,
                survival_reference,
                linear_score,
                tau=float(
                    METRIC_TIMES[
                        -1
                    ]
                ),
            )[0]
        )

        cox_audit_rows.append(
            {
                "fold_id": int(
                    fold_id
                ),
                "landmark_month": float(
                    landmark_month
                ),
                "landmark_year": float(
                    landmark_month
                    / 12.0
                ),
                "record_n": int(
                    len(rows)
                ),
                "future_5y_event_n": int(
                    event_within_60m[
                        rows
                    ].sum()
                ),
                "spearman_risk_vs_linear_score": (
                    rank_correlation
                ),
                "uno_c_from_5y_risk": (
                    c_risk
                ),
                "uno_c_from_linear_score": (
                    c_linear
                ),
                "absolute_c_difference": (
                    abs(
                        c_risk
                        - c_linear
                    )
                ),
            }
        )

cox_audit = pd.DataFrame(
    cox_audit_rows
)

cox_audit.to_csv(
    OUTPUT_DIR
    / "cox_prediction_alignment_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

if (
    cox_audit[
        "spearman_risk_vs_linear_score"
    ].min()
    < 0.999
):
    raise ValueError(
        "Cox 累积风险与线性风险评分"
        "排序审计未通过。"
    )

if (
    cox_audit[
        "absolute_c_difference"
    ].max()
    > 1e-3
):
    raise ValueError(
        "Cox 累积风险与线性评分"
        "Uno C 审计未通过。"
    )


# =============================================================================
# 6. 生存概率 ↔ 条件风险
# =============================================================================

def survival_to_hazard(
    survival: np.ndarray,
) -> np.ndarray:
    survival = np.clip(
        np.asarray(
            survival,
            dtype=np.float64,
        ),
        EPS,
        1.0,
    )

    previous_survival = (
        np.concatenate(
            [
                np.ones(
                    (
                        survival.shape[
                            0
                        ],
                        1,
                    ),
                    dtype=np.float64,
                ),
                survival[
                    :,
                    :-1,
                ],
            ],
            axis=1,
        )
    )

    hazard = (
        1.0
        - (
            survival
            / np.clip(
                previous_survival,
                EPS,
                1.0,
            )
        )
    )

    return np.clip(
        hazard,
        EPS,
        1.0 - EPS,
    ).astype(
        np.float32
    )


def hazard_to_risk(
    hazard: np.ndarray,
) -> np.ndarray:
    hazard = np.clip(
        np.asarray(
            hazard,
            dtype=np.float64,
        ),
        EPS,
        1.0 - EPS,
    )

    survival = np.cumprod(
        1.0 - hazard,
        axis=1,
    )

    return np.clip(
        1.0 - survival,
        0.0,
        1.0,
    ).astype(
        np.float32
    )


def probability_logit(
    probability: np.ndarray,
) -> np.ndarray:
    probability = np.clip(
        np.asarray(
            probability,
            dtype=np.float64,
        ),
        EPS,
        1.0 - EPS,
    )

    return (
        np.log(
            probability
        )
        - np.log1p(
            -probability
        )
    )


model_hazard_raw = {
    model_name: (
        survival_to_hazard(
            model_survival_raw[
                model_name
            ]
        )
    )
    for model_name
    in MODEL_ORDER
}

for model_name in MODEL_ORDER:
    reconstructed_risk = (
        hazard_to_risk(
            model_hazard_raw[
                model_name
            ]
        )
    )

    if not np.allclose(
        reconstructed_risk,
        model_risk_raw[
            model_name
        ],
        atol=2e-5,
    ):
        max_diff = float(
            np.max(
                np.abs(
                    reconstructed_risk
                    - model_risk_raw[
                        model_name
                    ]
                )
            )
        )

        raise ValueError(
            f"{model_name} 条件风险重建"
            "累积风险失败，"
            f"最大差值={max_diff:.3e}。"
        )


# =============================================================================
# 7. 五折交叉拟合 hazard 校准
# =============================================================================

def fit_interval_hazard_calibrator(
    hazard: np.ndarray,
    target: np.ndarray,
    mask: np.ndarray,
) -> dict[str, Any]:
    """
    模型：
      logit(h_cal,j)
        = alpha_j
        + beta * logit(h_raw,j)

    alpha_j：
      10 个区间特异截距

    beta：
      一个共同斜率
    """

    hazard = np.asarray(
        hazard,
        dtype=np.float64,
    )

    target = np.asarray(
        target,
        dtype=np.float64,
    )

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    if (
        hazard.shape
        != target.shape
        or hazard.shape
        != mask.shape
    ):
        raise ValueError(
            "校准输入形状不一致。"
        )

    raw_logit = (
        probability_logit(
            hazard
        )
    )

    interval_matrix = (
        np.broadcast_to(
            np.arange(
                EXPECTED_INTERVAL_N,
                dtype=np.int16,
            ),
            hazard.shape,
        )
    )

    x = raw_logit[
        mask
    ]

    y = target[
        mask
    ]

    interval_index = (
        interval_matrix[
            mask
        ]
    )

    if len(y) == 0:
        raise ValueError(
            "校准训练数据没有有效风险区间。"
        )

    if float(
        y.sum()
    ) <= 0:
        raise ValueError(
            "校准训练数据没有事件。"
        )

    initial = np.concatenate(
        [
            np.zeros(
                EXPECTED_INTERVAL_N,
                dtype=np.float64,
            ),
            np.asarray(
                [1.0],
                dtype=np.float64,
            ),
        ]
    )

    def objective_and_gradient(
        parameters: np.ndarray,
    ) -> tuple[
        float,
        np.ndarray,
    ]:
        alpha = parameters[
            :EXPECTED_INTERVAL_N
        ]

        beta = parameters[
            -1
        ]

        eta = (
            alpha[
                interval_index
            ]
            + beta
            * x
        )

        loss = np.mean(
            np.logaddexp(
                0.0,
                eta,
            )
            - y
            * eta
        )

        residual = (
            expit(
                eta
            )
            - y
        ) / len(y)

        gradient_alpha = (
            np.bincount(
                interval_index,
                weights=residual,
                minlength=(
                    EXPECTED_INTERVAL_N
                ),
            ).astype(
                np.float64
            )
        )

        gradient_beta = float(
            np.dot(
                residual,
                x,
            )
        )

        if (
            CALIBRATION_RIDGE
            > 0
        ):
            loss += (
                0.5
                * CALIBRATION_RIDGE
                * (
                    float(
                        np.mean(
                            alpha
                            ** 2
                        )
                    )
                    + float(
                        (
                            beta
                            - 1.0
                        )
                        ** 2
                    )
                )
            )

            gradient_alpha += (
                CALIBRATION_RIDGE
                * alpha
                / EXPECTED_INTERVAL_N
            )

            gradient_beta += (
                CALIBRATION_RIDGE
                * (
                    beta
                    - 1.0
                )
            )

        gradient = np.concatenate(
            [
                gradient_alpha,
                np.asarray(
                    [
                        gradient_beta
                    ],
                    dtype=np.float64,
                ),
            ]
        )

        return (
            float(loss),
            gradient,
        )

    bounds = (
        [
            (-8.0, 8.0)
        ]
        * EXPECTED_INTERVAL_N
        + [
            (0.05, 5.0)
        ]
    )

    result = minimize(
        fun=(
            objective_and_gradient
        ),
        x0=initial,
        method="L-BFGS-B",
        jac=True,
        bounds=bounds,
        options={
            "maxiter": 300,
            "ftol": 1e-12,
            "gtol": 1e-8,
            "maxls": 50,
        },
    )

    if not result.success:
        raise RuntimeError(
            "离散风险校准失败："
            + str(
                result.message
            )
        )

    return {
        "alpha": (
            result.x[
                :EXPECTED_INTERVAL_N
            ].astype(
                np.float64
            )
        ),
        "beta": float(
            result.x[
                -1
            ]
        ),
        "valid_interval_n": int(
            len(y)
        ),
        "event_interval_n": int(
            y.sum()
        ),
        "objective": float(
            result.fun
        ),
        "iterations": int(
            result.nit
        ),
    }


def apply_interval_hazard_calibrator(
    hazard: np.ndarray,
    fit: dict[str, Any],
) -> np.ndarray:
    raw_logit = (
        probability_logit(
            hazard
        )
    )

    alpha = np.asarray(
        fit[
            "alpha"
        ],
        dtype=np.float64,
    )

    beta = float(
        fit[
            "beta"
        ]
    )

    calibrated = expit(
        alpha[
            None,
            :,
        ]
        + beta
        * raw_logit
    )

    return np.clip(
        calibrated,
        EPS,
        1.0 - EPS,
    ).astype(
        np.float32
    )


model_hazard_calibrated: dict[
    str,
    np.ndarray,
] = {}

model_risk_calibrated: dict[
    str,
    np.ndarray,
] = {}

model_survival_calibrated: dict[
    str,
    np.ndarray,
] = {}

calibration_parameter_rows: list[
    dict[str, Any]
] = []

final_calibrators: dict[
    str,
    dict[str, Any],
] = {}

for model_name in MODEL_ORDER:
    print(
        "\n"
        + "-" * 110
    )
    print(
        f"{model_name}："
        "开始五折交叉拟合 hazard 校准"
    )
    print(
        "-" * 110
    )

    raw_hazard = (
        model_hazard_raw[
            model_name
        ]
    )

    calibrated_hazard = (
        np.full_like(
            raw_hazard,
            np.nan,
            dtype=np.float32,
        )
    )

    final_calibrators[
        model_name
    ] = {}

    for (
        landmark_position,
        landmark_month,
    ) in enumerate(
        LANDMARK_MONTHS
    ):
        landmark_mask = (
            landmark_index_long
            == landmark_position
        )

        for heldout_fold in range(
            N_SPLITS
        ):
            train_mask = (
                landmark_mask
                & (
                    fold_id_long
                    != heldout_fold
                )
            )

            valid_mask = (
                landmark_mask
                & (
                    fold_id_long
                    == heldout_fold
                )
            )

            fit = (
                fit_interval_hazard_calibrator(
                    raw_hazard[
                        train_mask
                    ],
                    future_event_long[
                        train_mask
                    ],
                    future_at_risk_long[
                        train_mask
                    ],
                )
            )

            calibrated_hazard[
                valid_mask
            ] = (
                apply_interval_hazard_calibrator(
                    raw_hazard[
                        valid_mask
                    ],
                    fit,
                )
            )

            for (
                interval_position,
                interval_month,
            ) in enumerate(
                FUTURE_END_MONTHS
            ):
                calibration_parameter_rows.append(
                    {
                        "model": model_name,
                        "fit_scope": (
                            "crossfit"
                        ),
                        "heldout_fold": int(
                            heldout_fold
                        ),
                        "landmark_month": float(
                            landmark_month
                        ),
                        "landmark_year": float(
                            landmark_month
                            / 12.0
                        ),
                        "interval_position": int(
                            interval_position
                        ),
                        "interval_end_month": float(
                            interval_month
                        ),
                        "alpha_interval": float(
                            fit[
                                "alpha"
                            ][
                                interval_position
                            ]
                        ),
                        "beta_common_slope": float(
                            fit[
                                "beta"
                            ]
                        ),
                        "valid_interval_n": int(
                            fit[
                                "valid_interval_n"
                            ]
                        ),
                        "event_interval_n": int(
                            fit[
                                "event_interval_n"
                            ]
                        ),
                        "objective": float(
                            fit[
                                "objective"
                            ]
                        ),
                        "optimizer_iterations": int(
                            fit[
                                "iterations"
                            ]
                        ),
                    }
                )

            print(
                f"{model_name} | "
                f"Landmark "
                f"{landmark_month / 12:.0f} 年 | "
                f"held-out fold "
                f"{heldout_fold} | "
                f"beta="
                f"{fit['beta']:.4f}"
            )

        # 最终校准器：
        # 仅使用全开发集 OOF 预测 + 开发集真实结局。
        final_fit = (
            fit_interval_hazard_calibrator(
                raw_hazard[
                    landmark_mask
                ],
                future_event_long[
                    landmark_mask
                ],
                future_at_risk_long[
                    landmark_mask
                ],
            )
        )

        final_calibrators[
            model_name
        ][
            str(
                int(
                    landmark_month
                )
            )
        ] = {
            "landmark_month": float(
                landmark_month
            ),
            "alpha": (
                final_fit[
                    "alpha"
                ].tolist()
            ),
            "beta": float(
                final_fit[
                    "beta"
                ]
            ),
            "valid_interval_n": int(
                final_fit[
                    "valid_interval_n"
                ]
            ),
            "event_interval_n": int(
                final_fit[
                    "event_interval_n"
                ]
            ),
        }

        for (
            interval_position,
            interval_month,
        ) in enumerate(
            FUTURE_END_MONTHS
        ):
            calibration_parameter_rows.append(
                {
                    "model": model_name,
                    "fit_scope": (
                        "final_all_development_oof"
                    ),
                    "heldout_fold": -1,
                    "landmark_month": float(
                        landmark_month
                    ),
                    "landmark_year": float(
                        landmark_month
                        / 12.0
                    ),
                    "interval_position": int(
                        interval_position
                    ),
                    "interval_end_month": float(
                        interval_month
                    ),
                    "alpha_interval": float(
                        final_fit[
                            "alpha"
                        ][
                            interval_position
                        ]
                    ),
                    "beta_common_slope": float(
                        final_fit[
                            "beta"
                        ]
                    ),
                    "valid_interval_n": int(
                        final_fit[
                            "valid_interval_n"
                        ]
                    ),
                    "event_interval_n": int(
                        final_fit[
                            "event_interval_n"
                        ]
                    ),
                    "objective": float(
                        final_fit[
                            "objective"
                        ]
                    ),
                    "optimizer_iterations": int(
                        final_fit[
                            "iterations"
                        ]
                    ),
                }
            )

    if not np.isfinite(
        calibrated_hazard
    ).all():
        raise ValueError(
            f"{model_name} 交叉拟合校准后"
            "仍存在缺失值。"
        )

    calibrated_risk = (
        hazard_to_risk(
            calibrated_hazard
        )
    )

    calibrated_survival = (
        1.0
        - calibrated_risk
    ).astype(
        np.float32
    )

    if int(
        np.sum(
            np.diff(
                calibrated_risk,
                axis=1,
            )
            < -1e-7
        )
    ) != 0:
        raise ValueError(
            f"{model_name} 校准后"
            "累积风险不单调。"
        )

    model_hazard_calibrated[
        model_name
    ] = calibrated_hazard

    model_risk_calibrated[
        model_name
    ] = calibrated_risk

    model_survival_calibrated[
        model_name
    ] = calibrated_survival

    model_file_name = (
        safe_name(
            model_name
        )
    )

    np.save(
        OUTPUT_DIR
        / (
            f"{model_file_name}_crossfit_"
            "calibrated_oof_hazard_long.npy"
        ),
        calibrated_hazard,
    )

    np.save(
        OUTPUT_DIR
        / (
            f"{model_file_name}_crossfit_"
            "calibrated_oof_risk_long.npy"
        ),
        calibrated_risk,
    )

    np.save(
        OUTPUT_DIR
        / (
            f"{model_file_name}_crossfit_"
            "calibrated_oof_survival_long.npy"
        ),
        calibrated_survival,
    )

calibration_parameters = pd.DataFrame(
    calibration_parameter_rows
)

calibration_parameters.to_csv(
    OUTPUT_DIR
    / "four_model_hazard_calibration_parameters.csv",
    index=False,
    encoding="utf-8-sig",
)

save_json(
    final_calibrators,
    OUTPUT_DIR
    / "four_model_final_development_oof_calibrators.json",
)


# =============================================================================
# 8. 指标计算函数
# =============================================================================

def evaluate_survival_predictions(
    survival_train: np.ndarray,
    survival_test: np.ndarray,
    risk_matrix: np.ndarray,
) -> dict[str, Any]:
    risk_matrix = np.asarray(
        risk_matrix,
        dtype=np.float64,
    )

    if (
        risk_matrix.ndim
        != 2
        or risk_matrix.shape[
            1
        ]
        != EXPECTED_INTERVAL_N
    ):
        raise ValueError(
            "risk_matrix 形状非法。"
        )

    survival_matrix = (
        1.0
        - risk_matrix
    )

    c_index = float(
        concordance_index_ipcw(
            survival_train,
            survival_test,
            risk_matrix[
                :,
                -1,
            ],
            tau=float(
                METRIC_TIMES[
                    -1
                ]
            ),
        )[0]
    )

    dynamic_auc, mean_auc = (
        cumulative_dynamic_auc(
            survival_train,
            survival_test,
            risk_matrix,
            METRIC_TIMES,
        )
    )

    _, brier_values = (
        brier_score(
            survival_train,
            survival_test,
            survival_matrix,
            METRIC_TIMES,
        )
    )

    ibs = float(
        integrated_brier_score(
            survival_train,
            survival_test,
            survival_matrix,
            METRIC_TIMES,
        )
    )

    return {
        "uno_c_index_5y": (
            c_index
        ),
        "dynamic_auc": (
            np.asarray(
                dynamic_auc,
                dtype=np.float64,
            )
        ),
        "integrated_dynamic_auc": (
            float(
                mean_auc
            )
        ),
        "brier": (
            np.asarray(
                brier_values,
                dtype=np.float64,
            )
        ),
        "integrated_brier": (
            ibs
        ),
    }


def safe_evaluate_survival_predictions(
    survival_train: np.ndarray,
    survival_test: np.ndarray,
    risk_matrix: np.ndarray,
) -> dict[str, Any] | None:
    try:
        return (
            evaluate_survival_predictions(
                survival_train,
                survival_test,
                risk_matrix,
            )
        )
    except (
        ValueError,
        ArithmeticError,
        ZeroDivisionError,
        FloatingPointError,
    ):
        return None


# =============================================================================
# 9. Raw 和 cross-fitted calibrated point performance
# =============================================================================

def calculate_point_tables(
    model_risks: dict[
        str,
        np.ndarray,
    ],
    prediction_stage: str,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    landmark_rows: list[
        dict[str, Any]
    ] = []

    horizon_rows: list[
        dict[str, Any]
    ] = []

    selected_horizon_rows: list[
        dict[str, Any]
    ] = []

    for (
        landmark_position,
        landmark_month,
    ) in enumerate(
        LANDMARK_MONTHS
    ):
        rows = np.where(
            landmark_index_long
            == landmark_position
        )[0]

        survival_reference = (
            build_survival_array(
                event_within_60m[
                    rows
                ],
                analysis_time_month[
                    rows
                ],
            )
        )

        for model_name in MODEL_ORDER:
            risk_matrix = (
                model_risks[
                    model_name
                ][
                    rows,
                    :,
                ]
            )

            metrics = (
                evaluate_survival_predictions(
                    survival_reference,
                    survival_reference,
                    risk_matrix,
                )
            )

            landmark_rows.append(
                {
                    "prediction_stage": (
                        prediction_stage
                    ),
                    "model": (
                        model_name
                    ),
                    "landmark_position": int(
                        landmark_position
                    ),
                    "landmark_month": float(
                        landmark_month
                    ),
                    "landmark_year": float(
                        landmark_month
                        / 12.0
                    ),
                    "risk_set_n": int(
                        len(rows)
                    ),
                    "future_5y_event_n": int(
                        event_within_60m[
                            rows
                        ].sum()
                    ),
                    "uno_c_index_5y": float(
                        metrics[
                            "uno_c_index_5y"
                        ]
                    ),
                    "integrated_dynamic_auc": float(
                        metrics[
                            "integrated_dynamic_auc"
                        ]
                    ),
                    "integrated_brier": float(
                        metrics[
                            "integrated_brier"
                        ]
                    ),
                }
            )

            for (
                horizon_position,
                horizon_month,
            ) in enumerate(
                FUTURE_END_MONTHS
            ):
                row = {
                    "prediction_stage": (
                        prediction_stage
                    ),
                    "model": (
                        model_name
                    ),
                    "landmark_position": int(
                        landmark_position
                    ),
                    "landmark_month": float(
                        landmark_month
                    ),
                    "landmark_year": float(
                        landmark_month
                        / 12.0
                    ),
                    "horizon_position": int(
                        horizon_position
                    ),
                    "horizon_month": float(
                        horizon_month
                    ),
                    "horizon_year": float(
                        horizon_month
                        / 12.0
                    ),
                    "dynamic_auc": float(
                        metrics[
                            "dynamic_auc"
                        ][
                            horizon_position
                        ]
                    ),
                    "brier_score": float(
                        metrics[
                            "brier"
                        ][
                            horizon_position
                        ]
                    ),
                }

                horizon_rows.append(
                    row
                )

                if horizon_month in {
                    12.0,
                    36.0,
                    60.0,
                }:
                    selected_horizon_rows.append(
                        row.copy()
                    )

    return (
        pd.DataFrame(
            landmark_rows
        ),
        pd.DataFrame(
            horizon_rows
        ),
        pd.DataFrame(
            selected_horizon_rows
        ),
    )


(
    raw_landmark_metrics,
    raw_horizon_metrics,
    raw_1y_3y_5y_metrics,
) = calculate_point_tables(
    model_risk_raw,
    prediction_stage="raw_oof",
)

(
    calibrated_landmark_metrics,
    calibrated_horizon_metrics,
    calibrated_1y_3y_5y_metrics,
) = calculate_point_tables(
    model_risk_calibrated,
    prediction_stage=(
        "crossfit_calibrated_oof"
    ),
)

raw_landmark_metrics.to_csv(
    OUTPUT_DIR
    / "four_model_raw_landmark_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

raw_horizon_metrics.to_csv(
    OUTPUT_DIR
    / "four_model_raw_horizon_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

raw_1y_3y_5y_metrics.to_csv(
    OUTPUT_DIR
    / "four_model_raw_1y_3y_5y_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

calibrated_landmark_metrics.to_csv(
    OUTPUT_DIR
    / "four_model_crossfit_calibrated_landmark_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

calibrated_horizon_metrics.to_csv(
    OUTPUT_DIR
    / "four_model_crossfit_calibrated_horizon_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

calibrated_1y_3y_5y_metrics.to_csv(
    OUTPUT_DIR
    / "four_model_crossfit_calibrated_1y_3y_5y_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 10. 患者级 paired bootstrap
# =============================================================================

BOOTSTRAP_METRIC_INDEX = {
    metric_name: index
    for index, metric_name in enumerate(
        METRIC_NAMES
    )
}

checkpoint_file = (
    BOOTSTRAP_DIR
    / (
        f"paired_bootstrap_"
        f"{BOOTSTRAP_REPS}_replicates.npz"
    )
)

bootstrap_shape = (
    BOOTSTRAP_REPS,
    len(
        MODEL_ORDER
    ),
    EXPECTED_LANDMARK_N,
    len(
        METRIC_NAMES
    ),
)

if checkpoint_file.exists():
    checkpoint = np.load(
        checkpoint_file,
        allow_pickle=False,
    )

    bootstrap_metrics = (
        checkpoint[
            "bootstrap_metrics"
        ].astype(
            np.float64
        )
    )

    completed_replicates = (
        checkpoint[
            "completed_replicates"
        ].astype(
            bool
        )
    )

    if (
        bootstrap_metrics.shape
        != bootstrap_shape
    ):
        raise ValueError(
            "已有 Bootstrap checkpoint "
            "形状与当前配置不一致。"
        )

    if (
        completed_replicates.shape
        != (
            BOOTSTRAP_REPS,
        )
    ):
        raise ValueError(
            "已有 Bootstrap checkpoint "
            "完成标记形状不一致。"
        )

    print(
        "\n检测到 Bootstrap 断点："
        f"{int(completed_replicates.sum())}/"
        f"{BOOTSTRAP_REPS} 已完成。"
    )

else:
    bootstrap_metrics = np.full(
        bootstrap_shape,
        np.nan,
        dtype=np.float64,
    )

    completed_replicates = np.zeros(
        BOOTSTRAP_REPS,
        dtype=bool,
    )


# 每个 Landmark 的完整原始风险集，
# 作为 IPCW 的 survival_train/reference。
landmark_reference_rows: list[
    np.ndarray
] = []

landmark_survival_reference: list[
    np.ndarray
] = []

for landmark_position in range(
    EXPECTED_LANDMARK_N
):
    rows = np.where(
        landmark_index_long
        == landmark_position
    )[0]

    landmark_reference_rows.append(
        rows
    )

    landmark_survival_reference.append(
        build_survival_array(
            event_within_60m[
                rows
            ],
            analysis_time_month[
                rows
            ],
        )
    )


bootstrap_start = time.time()

for bootstrap_index in range(
    BOOTSTRAP_REPS
):
    if completed_replicates[
        bootstrap_index
    ]:
        continue

    # 每个 replicate 使用独立、可复现的 seed，
    # 因此断点续跑不会改变结果。
    rng = np.random.default_rng(
        RANDOM_SEED
        + bootstrap_index
        * 1009
    )

    sampled_patients = (
        rng.integers(
            0,
            EXPECTED_DEVELOPMENT_N,
            size=EXPECTED_DEVELOPMENT_N,
            endpoint=False,
        )
    )

    for landmark_position in range(
        EXPECTED_LANDMARK_N
    ):
        sampled_rows = (
            row_index_map[
                sampled_patients,
                landmark_position,
            ]
        )

        sampled_rows = sampled_rows[
            sampled_rows
            >= 0
        ].astype(
            np.int64
        )

        if (
            len(sampled_rows)
            < 50
        ):
            continue

        sampled_event = (
            event_within_60m[
                sampled_rows
            ]
        )

        if int(
            sampled_event.sum()
        ) < 5:
            continue

        survival_test = (
            build_survival_array(
                sampled_event,
                analysis_time_month[
                    sampled_rows
                ],
            )
        )

        survival_train = (
            landmark_survival_reference[
                landmark_position
            ]
        )

        for (
            model_index,
            model_name,
        ) in enumerate(
            MODEL_ORDER
        ):
            metrics = (
                safe_evaluate_survival_predictions(
                    survival_train,
                    survival_test,
                    model_risk_calibrated[
                        model_name
                    ][
                        sampled_rows,
                        :,
                    ],
                )
            )

            if metrics is None:
                continue

            bootstrap_metrics[
                bootstrap_index,
                model_index,
                landmark_position,
                BOOTSTRAP_METRIC_INDEX[
                    "uno_c_index_5y"
                ],
            ] = (
                metrics[
                    "uno_c_index_5y"
                ]
            )

            bootstrap_metrics[
                bootstrap_index,
                model_index,
                landmark_position,
                BOOTSTRAP_METRIC_INDEX[
                    "integrated_dynamic_auc"
                ],
            ] = (
                metrics[
                    "integrated_dynamic_auc"
                ]
            )

            bootstrap_metrics[
                bootstrap_index,
                model_index,
                landmark_position,
                BOOTSTRAP_METRIC_INDEX[
                    "integrated_brier"
                ],
            ] = (
                metrics[
                    "integrated_brier"
                ]
            )

    completed_replicates[
        bootstrap_index
    ] = True

    completed_n = int(
        completed_replicates.sum()
    )

    if (
        completed_n
        % BOOTSTRAP_SAVE_EVERY
        == 0
        or completed_n
        == BOOTSTRAP_REPS
    ):
        np.savez_compressed(
            checkpoint_file,
            bootstrap_metrics=(
                bootstrap_metrics
            ),
            completed_replicates=(
                completed_replicates
            ),
        )

    if (
        completed_n
        % 25
        == 0
        or completed_n
        == BOOTSTRAP_REPS
    ):
        print(
            "Paired bootstrap："
            f"{completed_n}/"
            f"{BOOTSTRAP_REPS} | "
            "本次已运行 "
            f"{format_duration(time.time() - bootstrap_start)}"
        )


# =============================================================================
# 11. Bootstrap：Landmark 级 CI
# =============================================================================

landmark_bootstrap_rows: list[
    dict[str, Any]
] = []

point_metric_lookup: dict[
    tuple[
        str,
        int,
        str,
    ],
    float,
] = {}

for _, row in (
    calibrated_landmark_metrics.iterrows()
):
    model_name = str(
        row[
            "model"
        ]
    )

    landmark_position = int(
        row[
            "landmark_position"
        ]
    )

    for metric_name in (
        METRIC_NAMES
    ):
        point_metric_lookup[
            (
                model_name,
                landmark_position,
                metric_name,
            )
        ] = float(
            row[
                metric_name
            ]
        )


for (
    model_index,
    model_name,
) in enumerate(
    MODEL_ORDER
):
    for landmark_position in range(
        EXPECTED_LANDMARK_N
    ):
        landmark_month = (
            LANDMARK_MONTHS[
                landmark_position
            ]
        )

        for metric_name in (
            METRIC_NAMES
        ):
            metric_index = (
                BOOTSTRAP_METRIC_INDEX[
                    metric_name
                ]
            )

            values = (
                bootstrap_metrics[
                    :,
                    model_index,
                    landmark_position,
                    metric_index,
                ]
            )

            (
                lower,
                upper,
                valid_n,
            ) = percentile_interval(
                values,
                expected_n=BOOTSTRAP_REPS,
            )

            landmark_bootstrap_rows.append(
                {
                    "model": (
                        model_name
                    ),
                    "landmark_position": int(
                        landmark_position
                    ),
                    "landmark_month": float(
                        landmark_month
                    ),
                    "landmark_year": float(
                        landmark_month
                        / 12.0
                    ),
                    "metric": (
                        metric_name
                    ),
                    "point_estimate": float(
                        point_metric_lookup[
                            (
                                model_name,
                                landmark_position,
                                metric_name,
                            )
                        ]
                    ),
                    "lower_95": (
                        lower
                    ),
                    "upper_95": (
                        upper
                    ),
                    "valid_bootstrap_n": int(
                        valid_n
                    ),
                }
            )

landmark_bootstrap = pd.DataFrame(
    landmark_bootstrap_rows
)

landmark_bootstrap.to_csv(
    OUTPUT_DIR
    / "four_model_landmark_bootstrap_performance.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 12. 跨 Landmark 等权平均 + Bootstrap CI
# =============================================================================

model_summary_rows: list[
    dict[str, Any]
] = []

scope_bootstrap_cache: dict[
    tuple[
        str,
        str,
        str,
    ],
    np.ndarray,
] = {}

for (
    scope_name,
    landmark_positions,
) in SUMMARY_SCOPES.items():
    for (
        model_index,
        model_name,
    ) in enumerate(
        MODEL_ORDER
    ):
        point_subset = (
            calibrated_landmark_metrics.loc[
                (
                    calibrated_landmark_metrics[
                        "model"
                    ]
                    == model_name
                )
                & (
                    calibrated_landmark_metrics[
                        "landmark_position"
                    ].isin(
                        landmark_positions
                    )
                )
            ]
            .sort_values(
                "landmark_position"
            )
        )

        if (
            len(
                point_subset
            )
            != len(
                landmark_positions
            )
        ):
            raise ValueError(
                f"{scope_name} | "
                f"{model_name} "
                "Landmark 数量不完整。"
            )

        row = {
            "scope": scope_name,
            "model": model_name,
            "landmark_positions": ",".join(
                str(
                    int(x)
                )
                for x in (
                    landmark_positions
                )
            ),
            "landmark_years": ",".join(
                str(
                    int(
                        LANDMARK_MONTHS[
                            x
                        ]
                        / 12
                    )
                )
                for x in (
                    landmark_positions
                )
            ),
        }

        for metric_name in (
            METRIC_NAMES
        ):
            metric_index = (
                BOOTSTRAP_METRIC_INDEX[
                    metric_name
                ]
            )

            point_value = float(
                point_subset[
                    metric_name
                ].mean()
            )

            replicate_by_landmark = (
                bootstrap_metrics[
                    :,
                    model_index,
                    :,
                    metric_index,
                ]
            )

            replicate_mean = (
                complete_row_mean(
                    replicate_by_landmark,
                    landmark_positions,
                )
            )

            scope_bootstrap_cache[
                (
                    scope_name,
                    model_name,
                    metric_name,
                )
            ] = replicate_mean

            (
                lower,
                upper,
                valid_n,
            ) = percentile_interval(
                replicate_mean,
                expected_n=BOOTSTRAP_REPS,
            )

            prefix = {
                "uno_c_index_5y": (
                    "mean_uno_c_index_5y"
                ),
                "integrated_dynamic_auc": (
                    "mean_integrated_dynamic_auc"
                ),
                "integrated_brier": (
                    "mean_integrated_brier"
                ),
            }[
                metric_name
            ]

            row[
                prefix
            ] = point_value

            row[
                (
                    prefix
                    + "_lower_95"
                )
            ] = lower

            row[
                (
                    prefix
                    + "_upper_95"
                )
            ] = upper

            row[
                (
                    prefix
                    + "_valid_bootstrap_n"
                )
            ] = valid_n

        model_summary_rows.append(
            row
        )

model_summary = pd.DataFrame(
    model_summary_rows
)

model_summary.to_csv(
    OUTPUT_DIR
    / "four_model_equal_weight_mean_performance.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 13. 预设患者级配对 Bootstrap 模型差值
# =============================================================================

pairwise_rows: list[
    dict[str, Any]
] = []

for (
    scope_name,
    landmark_positions,
) in SUMMARY_SCOPES.items():
    for (
        model_a,
        model_b,
    ) in PAIRWISE_COMPARISONS:
        for metric_name in (
            METRIC_NAMES
        ):
            a_boot = (
                scope_bootstrap_cache[
                    (
                        scope_name,
                        model_a,
                        metric_name,
                    )
                ]
            )

            b_boot = (
                scope_bootstrap_cache[
                    (
                        scope_name,
                        model_b,
                        metric_name,
                    )
                ]
            )

            difference = (
                a_boot
                - b_boot
            )

            (
                lower,
                upper,
                valid_n,
            ) = percentile_interval(
                difference,
                expected_n=BOOTSTRAP_REPS,
            )

            point_a = float(
                model_summary.loc[
                    (
                        model_summary[
                            "scope"
                        ]
                        == scope_name
                    )
                    & (
                        model_summary[
                            "model"
                        ]
                        == model_a
                    ),
                    {
                        "uno_c_index_5y": (
                            "mean_uno_c_index_5y"
                        ),
                        "integrated_dynamic_auc": (
                            "mean_integrated_dynamic_auc"
                        ),
                        "integrated_brier": (
                            "mean_integrated_brier"
                        ),
                    }[
                        metric_name
                    ],
                ].iloc[
                    0
                ]
            )

            point_b = float(
                model_summary.loc[
                    (
                        model_summary[
                            "scope"
                        ]
                        == scope_name
                    )
                    & (
                        model_summary[
                            "model"
                        ]
                        == model_b
                    ),
                    {
                        "uno_c_index_5y": (
                            "mean_uno_c_index_5y"
                        ),
                        "integrated_dynamic_auc": (
                            "mean_integrated_dynamic_auc"
                        ),
                        "integrated_brier": (
                            "mean_integrated_brier"
                        ),
                    }[
                        metric_name
                    ],
                ].iloc[
                    0
                ]
            )

            higher = (
                HIGHER_IS_BETTER[
                    metric_name
                ]
            )

            pairwise_rows.append(
                {
                    "scope": (
                        scope_name
                    ),
                    "model_a": (
                        model_a
                    ),
                    "model_b": (
                        model_b
                    ),
                    "comparison_label": (
                        f"{model_a} - {model_b}"
                    ),
                    "metric": (
                        metric_name
                    ),
                    "higher_is_better": bool(
                        higher
                    ),
                    "model_a_value": (
                        point_a
                    ),
                    "model_b_value": (
                        point_b
                    ),
                    "difference_a_minus_b": float(
                        point_a
                        - point_b
                    ),
                    "difference_lower_95": (
                        lower
                    ),
                    "difference_upper_95": (
                        upper
                    ),
                    "valid_bootstrap_n": int(
                        valid_n
                    ),
                    "model_a_favorable_fraction": (
                        favorable_fraction(
                            difference,
                            higher_is_better=(
                                higher
                            ),
                        )
                    ),
                    "bootstrap_two_sided_p": (
                        two_sided_bootstrap_p(
                            difference
                        )
                    ),
                }
            )

pairwise_comparison = pd.DataFrame(
    pairwise_rows
)

pairwise_comparison.to_csv(
    OUTPUT_DIR
    / "four_model_paired_bootstrap_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 14. Kaplan–Meier 校准
# =============================================================================

def km_event_risk(
    time_month: np.ndarray,
    event: np.ndarray,
    horizon_month: float,
) -> tuple[
    float,
    float,
    float,
]:
    time_month = np.asarray(
        time_month,
        dtype=np.float64,
    )

    event = np.asarray(
        event,
        dtype=bool,
    )

    if len(
        time_month
    ) == 0:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    try:
        (
            km_time,
            km_survival,
            km_ci,
        ) = kaplan_meier_estimator(
            event,
            time_month,
            conf_type="log-log",
        )

        eligible = np.where(
            km_time
            <= horizon_month
        )[0]

        if len(
            eligible
        ) == 0:
            survival = 1.0
            lower_survival = 1.0
            upper_survival = 1.0
        else:
            index = int(
                eligible[
                    -1
                ]
            )

            survival = float(
                km_survival[
                    index
                ]
            )

            lower_survival = float(
                km_ci[
                    0,
                    index,
                ]
            )

            upper_survival = float(
                km_ci[
                    1,
                    index,
                ]
            )

        return (
            float(
                1.0
                - survival
            ),
            float(
                1.0
                - upper_survival
            ),
            float(
                1.0
                - lower_survival
            ),
        )

    except TypeError:
        # 兼容旧版 sksurv：
        # 无 conf_type 参数时仍给出点估计。
        (
            km_time,
            km_survival,
        ) = kaplan_meier_estimator(
            event,
            time_month,
        )

        eligible = np.where(
            km_time
            <= horizon_month
        )[0]

        if len(
            eligible
        ) == 0:
            survival = 1.0
        else:
            survival = float(
                km_survival[
                    eligible[
                        -1
                    ]
                ]
            )

        risk = float(
            1.0
            - survival
        )

        return (
            risk,
            np.nan,
            np.nan,
        )


overall_calibration_rows: list[
    dict[str, Any]
] = []

decile_calibration_rows: list[
    dict[str, Any]
] = []

for (
    landmark_position,
    landmark_month,
) in enumerate(
    LANDMARK_MONTHS
):
    rows = np.where(
        landmark_index_long
        == landmark_position
    )[0]

    residual_time = (
        analysis_time_month[
            rows
        ]
    )

    event_indicator = (
        event_within_60m[
            rows
        ]
    )

    for model_name in MODEL_ORDER:
        risk_matrix = (
            model_risk_calibrated[
                model_name
            ][
                rows,
                :,
            ]
        )

        for horizon_month in [
            12.0,
            36.0,
            60.0,
        ]:
            horizon_position = int(
                np.where(
                    np.isclose(
                        FUTURE_END_MONTHS,
                        horizon_month,
                    )
                )[0][0]
            )

            predicted_risk = (
                risk_matrix[
                    :,
                    horizon_position,
                ]
            )

            (
                observed_risk,
                observed_lower,
                observed_upper,
            ) = km_event_risk(
                residual_time,
                event_indicator,
                horizon_month,
            )

            overall_calibration_rows.append(
                {
                    "model": (
                        model_name
                    ),
                    "landmark_month": float(
                        landmark_month
                    ),
                    "landmark_year": float(
                        landmark_month
                        / 12.0
                    ),
                    "horizon_month": float(
                        horizon_month
                    ),
                    "horizon_year": float(
                        horizon_month
                        / 12.0
                    ),
                    "risk_set_n": int(
                        len(rows)
                    ),
                    "mean_predicted_risk": float(
                        np.mean(
                            predicted_risk
                        )
                    ),
                    "km_observed_risk": (
                        observed_risk
                    ),
                    "km_lower_95": (
                        observed_lower
                    ),
                    "km_upper_95": (
                        observed_upper
                    ),
                    "calibration_difference": float(
                        np.mean(
                            predicted_risk
                        )
                        - observed_risk
                    ),
                }
            )

            risk_groups = (
                pd.qcut(
                    pd.Series(
                        predicted_risk
                    ),
                    q=10,
                    labels=False,
                    duplicates="drop",
                ).to_numpy()
            )

            valid_groups = (
                risk_groups[
                    ~pd.isna(
                        risk_groups
                    )
                ]
            )

            if len(
                valid_groups
            ) == 0:
                raise ValueError(
                    f"{model_name} | "
                    f"Landmark "
                    f"{landmark_month / 12:.0f} 年 | "
                    f"未来 "
                    f"{horizon_month / 12:.0f} 年："
                    "无法形成有效风险分组。"
                )

            for risk_group in np.sort(
                np.unique(
                    valid_groups
                )
            ):
                group_mask = (
                    risk_groups
                    == risk_group
                )

                (
                    group_risk,
                    group_lower,
                    group_upper,
                ) = km_event_risk(
                    residual_time[
                        group_mask
                    ],
                    event_indicator[
                        group_mask
                    ],
                    horizon_month,
                )

                decile_calibration_rows.append(
                    {
                        "model": (
                            model_name
                        ),
                        "landmark_month": float(
                            landmark_month
                        ),
                        "landmark_year": float(
                            landmark_month
                            / 12.0
                        ),
                        "horizon_month": float(
                            horizon_month
                        ),
                        "horizon_year": float(
                            horizon_month
                            / 12.0
                        ),
                        "risk_group": int(
                            risk_group
                        )
                        + 1,
                        "patient_n": int(
                            group_mask.sum()
                        ),
                        "mean_predicted_risk": float(
                            np.mean(
                                predicted_risk[
                                    group_mask
                                ]
                            )
                        ),
                        "km_observed_risk": (
                            group_risk
                        ),
                        "km_lower_95": (
                            group_lower
                        ),
                        "km_upper_95": (
                            group_upper
                        ),
                    }
                )

overall_calibration = pd.DataFrame(
    overall_calibration_rows
)

decile_calibration = pd.DataFrame(
    decile_calibration_rows
)

overall_calibration.to_csv(
    OUTPUT_DIR
    / "four_model_calibration_overall.csv",
    index=False,
    encoding="utf-8-sig",
)

decile_calibration.to_csv(
    OUTPUT_DIR
    / "four_model_calibration_deciles.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 15. 生存 DCA：IPCW net benefit
# =============================================================================

def step_function_value(
    time_grid: np.ndarray,
    probability: np.ndarray,
    query_time: np.ndarray | float,
) -> np.ndarray:
    time_grid = np.asarray(
        time_grid,
        dtype=np.float64,
    )

    probability = np.asarray(
        probability,
        dtype=np.float64,
    )

    query = np.asarray(
        query_time,
        dtype=np.float64,
    )

    indices = np.searchsorted(
        time_grid,
        query,
        side="right",
    ) - 1

    result = np.ones_like(
        query,
        dtype=np.float64,
    )

    valid = (
        indices
        >= 0
    )

    result[
        valid
    ] = probability[
        indices[
            valid
        ]
    ]

    return result


def ipcw_dca_weights(
    event: np.ndarray,
    time_month: np.ndarray,
    horizon_month: float,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:
    event = np.asarray(
        event,
        dtype=bool,
    )

    time_month = np.asarray(
        time_month,
        dtype=np.float64,
    )

    # reverse KM：
    # 估计 censoring survival G(t)
    (
        censor_time,
        censor_survival,
    ) = kaplan_meier_estimator(
        event,
        time_month,
        reverse=True,
    )

    event_by_horizon = (
        event
        & (
            time_month
            <= horizon_month
        )
    )

    known_non_event = (
        time_month
        > horizon_month
    )

    weights = np.zeros(
        len(
            time_month
        ),
        dtype=np.float64,
    )

    if np.any(
        event_by_horizon
    ):
        query_event_time = (
            np.nextafter(
                time_month[
                    event_by_horizon
                ],
                -np.inf,
            )
        )

        g_event = (
            step_function_value(
                censor_time,
                censor_survival,
                query_event_time,
            )
        )

        weights[
            event_by_horizon
        ] = (
            1.0
            / np.clip(
                g_event,
                1e-6,
                None,
            )
        )

    if np.any(
        known_non_event
    ):
        g_horizon = float(
            step_function_value(
                censor_time,
                censor_survival,
                float(
                    horizon_month
                ),
            )
        )

        weights[
            known_non_event
        ] = (
            1.0
            / max(
                g_horizon,
                1e-6,
            )
        )

    observed_binary = (
        event_by_horizon
        .astype(
            np.int8
        )
    )

    known_mask = (
        event_by_horizon
        | known_non_event
    )

    return (
        observed_binary,
        weights,
        known_mask,
    )


def survival_net_benefit(
    predicted_risk: np.ndarray,
    event: np.ndarray,
    time_month: np.ndarray,
    horizon_month: float,
    thresholds: np.ndarray,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:
    predicted_risk = np.asarray(
        predicted_risk,
        dtype=np.float64,
    )

    thresholds = np.asarray(
        thresholds,
        dtype=np.float64,
    )

    (
        observed_binary,
        weights,
        known_mask,
    ) = ipcw_dca_weights(
        event,
        time_month,
        horizon_month,
    )

    n_total = float(
        len(
            predicted_risk
        )
    )

    weighted_event_total = float(
        np.sum(
            weights[
                known_mask
            ]
            * observed_binary[
                known_mask
            ]
        )
    )

    weighted_nonevent_total = float(
        np.sum(
            weights[
                known_mask
            ]
            * (
                1
                - observed_binary[
                    known_mask
                ]
            )
        )
    )

    treat_all = np.empty(
        len(
            thresholds
        ),
        dtype=np.float64,
    )

    treat_none = np.zeros(
        len(
            thresholds
        ),
        dtype=np.float64,
    )

    model_nb = np.empty(
        len(
            thresholds
        ),
        dtype=np.float64,
    )

    for index, threshold in enumerate(
        thresholds
    ):
        odds = (
            threshold
            / (
                1.0
                - threshold
            )
        )

        positive = (
            predicted_risk
            >= threshold
        )

        true_positive_weight = float(
            np.sum(
                weights[
                    positive
                    & known_mask
                ]
                * observed_binary[
                    positive
                    & known_mask
                ]
            )
        )

        false_positive_weight = float(
            np.sum(
                weights[
                    positive
                    & known_mask
                ]
                * (
                    1
                    - observed_binary[
                        positive
                        & known_mask
                    ]
                )
            )
        )

        model_nb[
            index
        ] = (
            true_positive_weight
            / n_total
            - (
                false_positive_weight
                / n_total
            )
            * odds
        )

        treat_all[
            index
        ] = (
            weighted_event_total
            / n_total
            - (
                weighted_nonevent_total
                / n_total
            )
            * odds
        )

    return (
        model_nb,
        treat_all,
        treat_none,
    )


dca_rows: list[
    dict[str, Any]
] = []

for landmark_month in (
    PRIMARY_LANDMARK_MONTHS
):
    landmark_position = int(
        np.where(
            np.isclose(
                LANDMARK_MONTHS,
                landmark_month,
            )
        )[0][0]
    )

    rows = np.where(
        landmark_index_long
        == landmark_position
    )[0]

    event_indicator = (
        event_within_60m[
            rows
        ]
    )

    residual_time = (
        analysis_time_month[
            rows
        ]
    )

    for horizon_month in [
        12.0,
        36.0,
        60.0,
    ]:
        (
            threshold_min,
            threshold_max,
            threshold_n,
        ) = (
            DCA_THRESHOLD_RANGES[
                horizon_month
            ]
        )

        thresholds = np.linspace(
            threshold_min,
            threshold_max,
            int(
                threshold_n
            ),
        )

        horizon_position = int(
            np.where(
                np.isclose(
                    FUTURE_END_MONTHS,
                    horizon_month,
                )
            )[0][0]
        )

        reference_all = None
        reference_none = None

        for model_name in MODEL_ORDER:
            predicted_risk = (
                model_risk_calibrated[
                    model_name
                ][
                    rows,
                    horizon_position,
                ]
            )

            (
                model_nb,
                treat_all,
                treat_none,
            ) = survival_net_benefit(
                predicted_risk,
                event_indicator,
                residual_time,
                horizon_month,
                thresholds,
            )

            if reference_all is None:
                reference_all = (
                    treat_all
                )
                reference_none = (
                    treat_none
                )

            for (
                threshold,
                net_benefit,
            ) in zip(
                thresholds,
                model_nb,
            ):
                dca_rows.append(
                    {
                        "model": (
                            model_name
                        ),
                        "landmark_month": float(
                            landmark_month
                        ),
                        "landmark_year": float(
                            landmark_month
                            / 12.0
                        ),
                        "horizon_month": float(
                            horizon_month
                        ),
                        "horizon_year": float(
                            horizon_month
                            / 12.0
                        ),
                        "threshold_probability": float(
                            threshold
                        ),
                        "net_benefit": float(
                            net_benefit
                        ),
                    }
                )

        for (
            threshold,
            all_nb,
            none_nb,
        ) in zip(
            thresholds,
            reference_all,
            reference_none,
        ):
            dca_rows.append(
                {
                    "model": (
                        "Treat all"
                    ),
                    "landmark_month": float(
                        landmark_month
                    ),
                    "landmark_year": float(
                        landmark_month
                        / 12.0
                    ),
                    "horizon_month": float(
                        horizon_month
                    ),
                    "horizon_year": float(
                        horizon_month
                        / 12.0
                    ),
                    "threshold_probability": float(
                        threshold
                    ),
                    "net_benefit": float(
                        all_nb
                    ),
                }
            )

            dca_rows.append(
                {
                    "model": (
                        "Treat none"
                    ),
                    "landmark_month": float(
                        landmark_month
                    ),
                    "landmark_year": float(
                        landmark_month
                        / 12.0
                    ),
                    "horizon_month": float(
                        horizon_month
                    ),
                    "horizon_year": float(
                        horizon_month
                        / 12.0
                    ),
                    "threshold_probability": float(
                        threshold
                    ),
                    "net_benefit": float(
                        none_nb
                    ),
                }
            )

dca_table = pd.DataFrame(
    dca_rows
)

dca_table.to_csv(
    OUTPUT_DIR
    / "four_model_survival_dca.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 16. 图形
# =============================================================================

# 16.1 主文 0/1/3/5 年 Landmark：5 年 Uno C
primary_c = (
    landmark_bootstrap.loc[
        (
            landmark_bootstrap[
                "metric"
            ]
            == "uno_c_index_5y"
        )
        & (
            landmark_bootstrap[
                "landmark_month"
            ].isin(
                PRIMARY_LANDMARK_MONTHS
            )
        )
    ]
    .copy()
)

x_positions = np.arange(
    len(
        PRIMARY_LANDMARK_MONTHS
    ),
    dtype=float,
)

bar_width = 0.18

fig, ax = plt.subplots(
    figsize=(
        11.0,
        6.2,
    )
)

for (
    model_offset,
    model_name,
) in enumerate(
    MODEL_ORDER
):
    subset = (
        primary_c.loc[
            primary_c[
                "model"
            ]
            == model_name
        ]
        .sort_values(
            "landmark_month"
        )
    )

    values = (
        subset[
            "point_estimate"
        ].to_numpy(
            dtype=float
        )
    )

    lower = (
        subset[
            "lower_95"
        ].to_numpy(
            dtype=float
        )
    )

    upper = (
        subset[
            "upper_95"
        ].to_numpy(
            dtype=float
        )
    )

    positions = (
        x_positions
        + (
            model_offset
            - 1.5
        )
        * bar_width
    )

    ax.bar(
        positions,
        values,
        width=bar_width,
        label=model_name,
        yerr=np.vstack(
            [
                values
                - lower,
                upper
                - values,
            ]
        ),
        capsize=3,
    )

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    [
        f"ART year "
        f"{int(month / 12)}"
        for month in (
            PRIMARY_LANDMARK_MONTHS
        )
    ]
)

ax.set_xlabel(
    "Prediction landmark"
)

ax.set_ylabel(
    "Uno C-index for future 5-year CKD risk"
)

ax.set_title(
    "Cross-fitted OOF discrimination"
)

ax.grid(
    axis="y",
    alpha=0.20,
)

ax.legend(
    frameon=False
)

fig.tight_layout()

save_figure(
    fig,
    "figure_1_primary_landmark_5y_uno_c",
)


# 16.2 主文 scope 的模型平均性能
primary_summary = (
    model_summary.loc[
        model_summary[
            "scope"
        ]
        == (
            "primary_0_1_3_5y_"
            "landmark_equal_weight_mean"
        )
    ]
    .copy()
)

for (
    metric_column,
    lower_column,
    upper_column,
    ylabel,
    filename,
) in [
    (
        "mean_uno_c_index_5y",
        "mean_uno_c_index_5y_lower_95",
        "mean_uno_c_index_5y_upper_95",
        "Mean Uno C-index",
        "figure_2_primary_mean_uno_c",
    ),
    (
        "mean_integrated_dynamic_auc",
        "mean_integrated_dynamic_auc_lower_95",
        "mean_integrated_dynamic_auc_upper_95",
        "Mean integrated dynamic AUC",
        "figure_3_primary_mean_iauc",
    ),
    (
        "mean_integrated_brier",
        "mean_integrated_brier_lower_95",
        "mean_integrated_brier_upper_95",
        "Mean integrated Brier score",
        "figure_4_primary_mean_ibs",
    ),
]:
    values = (
        primary_summary[
            metric_column
        ].to_numpy(
            dtype=float
        )
    )

    lower = (
        primary_summary[
            lower_column
        ].to_numpy(
            dtype=float
        )
    )

    upper = (
        primary_summary[
            upper_column
        ].to_numpy(
            dtype=float
        )
    )

    x = np.arange(
        len(
            primary_summary
        )
    )

    fig, ax = plt.subplots(
        figsize=(
            8.5,
            5.5,
        )
    )

    ax.bar(
        x,
        values,
        yerr=np.vstack(
            [
                values
                - lower,
                upper
                - values,
            ]
        ),
        capsize=4,
    )

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        primary_summary[
            "model"
        ]
    )

    ax.set_ylabel(
        ylabel
    )

    ax.set_title(
        "Equal-weight mean across ART years 0, 1, 3, and 5"
    )

    ax.grid(
        axis="y",
        alpha=0.20,
    )

    fig.tight_layout()

    save_figure(
        fig,
        filename,
    )


# 16.3 动态 AUC 曲线：每个主要 Landmark 单独一张图
for landmark_month in (
    PRIMARY_LANDMARK_MONTHS
):
    subset = (
        calibrated_horizon_metrics.loc[
            calibrated_horizon_metrics[
                "landmark_month"
            ]
            == landmark_month
        ]
    )

    fig, ax = plt.subplots(
        figsize=(
            8.5,
            5.8,
        )
    )

    for model_name in (
        MODEL_ORDER
    ):
        model_subset = (
            subset.loc[
                subset[
                    "model"
                ]
                == model_name
            ]
            .sort_values(
                "horizon_month"
            )
        )

        ax.plot(
            model_subset[
                "horizon_year"
            ],
            model_subset[
                "dynamic_auc"
            ],
            marker="o",
            label=model_name,
        )

    ax.set_xlabel(
        "Years after landmark"
    )

    ax.set_ylabel(
        "Time-dependent AUC"
    )

    ax.set_title(
        "Dynamic AUC | "
        f"ART year "
        f"{int(landmark_month / 12)}"
    )

    ax.grid(
        alpha=0.20
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    save_figure(
        fig,
        (
            "figure_dynamic_auc_"
            f"landmark_{int(landmark_month)}m"
        ),
    )


# 16.4 Brier 曲线：每个主要 Landmark 单独一张图
for landmark_month in (
    PRIMARY_LANDMARK_MONTHS
):
    subset = (
        calibrated_horizon_metrics.loc[
            calibrated_horizon_metrics[
                "landmark_month"
            ]
            == landmark_month
        ]
    )

    fig, ax = plt.subplots(
        figsize=(
            8.5,
            5.8,
        )
    )

    for model_name in (
        MODEL_ORDER
    ):
        model_subset = (
            subset.loc[
                subset[
                    "model"
                ]
                == model_name
            ]
            .sort_values(
                "horizon_month"
            )
        )

        ax.plot(
            model_subset[
                "horizon_year"
            ],
            model_subset[
                "brier_score"
            ],
            marker="o",
            label=model_name,
        )

    ax.set_xlabel(
        "Years after landmark"
    )

    ax.set_ylabel(
        "Brier score"
    )

    ax.set_title(
        "Brier score | "
        f"ART year "
        f"{int(landmark_month / 12)}"
    )

    ax.grid(
        alpha=0.20
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    save_figure(
        fig,
        (
            "figure_brier_"
            f"landmark_{int(landmark_month)}m"
        ),
    )


# 16.5 校准图：1/3/5 年，每个 Landmark 单独输出
for landmark_month in (
    PRIMARY_LANDMARK_MONTHS
):
    for horizon_month in [
        12.0,
        36.0,
        60.0,
    ]:
        subset = (
            decile_calibration.loc[
                (
                    decile_calibration[
                        "landmark_month"
                    ]
                    == landmark_month
                )
                & (
                    decile_calibration[
                        "horizon_month"
                    ]
                    == horizon_month
                )
            ]
        )

        fig, ax = plt.subplots(
            figsize=(
                6.5,
                6.2,
            )
        )

        for model_name in (
            MODEL_ORDER
        ):
            model_subset = (
                subset.loc[
                    subset[
                        "model"
                    ]
                    == model_name
                ]
                .sort_values(
                    "mean_predicted_risk"
                )
            )

            ax.plot(
                model_subset[
                    "mean_predicted_risk"
                ],
                model_subset[
                    "km_observed_risk"
                ],
                marker="o",
                label=model_name,
            )

        axis_limit = float(
            max(
                subset[
                    "mean_predicted_risk"
                ].max(),
                subset[
                    "km_observed_risk"
                ].max(),
                0.01,
            )
            * 1.10
        )

        ax.plot(
            [
                0.0,
                axis_limit,
            ],
            [
                0.0,
                axis_limit,
            ],
            linestyle="--",
            linewidth=1.0,
        )

        ax.set_xlim(
            0.0,
            axis_limit,
        )

        ax.set_ylim(
            0.0,
            axis_limit,
        )

        ax.set_xlabel(
            "Mean predicted risk"
        )

        ax.set_ylabel(
            "Kaplan–Meier observed risk"
        )

        ax.set_title(
            f"Calibration | "
            f"ART year "
            f"{int(landmark_month / 12)} | "
            f"future "
            f"{int(horizon_month / 12)} year"
        )

        ax.grid(
            alpha=0.20
        )

        ax.legend(
            frameon=False
        )

        fig.tight_layout()

        save_figure(
            fig,
            (
                "figure_calibration_"
                f"landmark_{int(landmark_month)}m_"
                f"horizon_{int(horizon_month)}m"
            ),
        )


# 16.6 DCA：1/3/5 年，每个主要 Landmark 单独输出
for landmark_month in (
    PRIMARY_LANDMARK_MONTHS
):
    for horizon_month in [
        12.0,
        36.0,
        60.0,
    ]:
        subset = (
            dca_table.loc[
                (
                    dca_table[
                        "landmark_month"
                    ]
                    == landmark_month
                )
                & (
                    dca_table[
                        "horizon_month"
                    ]
                    == horizon_month
                )
            ]
        )

        fig, ax = plt.subplots(
            figsize=(
                8.5,
                5.8,
            )
        )

        for model_name in (
            MODEL_ORDER
            + [
                "Treat all",
                "Treat none",
            ]
        ):
            model_subset = (
                subset.loc[
                    subset[
                        "model"
                    ]
                    == model_name
                ]
                .sort_values(
                    "threshold_probability"
                )
            )

            if len(
                model_subset
            ) == 0:
                continue

            ax.plot(
                model_subset[
                    "threshold_probability"
                ],
                model_subset[
                    "net_benefit"
                ],
                label=model_name,
            )

        ax.xaxis.set_major_formatter(
            PercentFormatter(
                1.0
            )
        )

        ax.set_xlabel(
            "Threshold probability"
        )

        ax.set_ylabel(
            "Net benefit"
        )

        ax.set_title(
            f"Survival DCA | "
            f"ART year "
            f"{int(landmark_month / 12)} | "
            f"future "
            f"{int(horizon_month / 12)} year"
        )

        ax.grid(
            alpha=0.20
        )

        ax.legend(
            frameon=False
        )

        fig.tight_layout()

        save_figure(
            fig,
            (
                "figure_dca_"
                f"landmark_{int(landmark_month)}m_"
                f"horizon_{int(horizon_month)}m"
            ),
        )


# 16.7 主文 scope：配对差值 forest，每个指标一张
primary_pairwise = (
    pairwise_comparison.loc[
        pairwise_comparison[
            "scope"
        ]
        == (
            "primary_0_1_3_5y_"
            "landmark_equal_weight_mean"
        )
    ]
)

for metric_name in (
    METRIC_NAMES
):
    subset = (
        primary_pairwise.loc[
            primary_pairwise[
                "metric"
            ]
            == metric_name
        ]
        .copy()
    )

    values = (
        subset[
            "difference_a_minus_b"
        ].to_numpy(
            dtype=float
        )
    )

    lower = (
        subset[
            "difference_lower_95"
        ].to_numpy(
            dtype=float
        )
    )

    upper = (
        subset[
            "difference_upper_95"
        ].to_numpy(
            dtype=float
        )
    )

    y = np.arange(
        len(
            subset
        )
    )

    fig, ax = plt.subplots(
        figsize=(
            9.0,
            6.0,
        )
    )

    ax.errorbar(
        values,
        y,
        xerr=np.vstack(
            [
                values
                - lower,
                upper
                - values,
            ]
        ),
        fmt="o",
        capsize=4,
    )

    ax.axvline(
        0.0,
        linestyle="--",
        linewidth=1.0,
    )

    ax.set_yticks(
        y
    )

    ax.set_yticklabels(
        subset[
            "comparison_label"
        ]
    )

    ax.set_xlabel(
        (
            "Difference "
            "(model A - model B)"
        )
    )

    ax.set_title(
        "Patient-level paired bootstrap | "
        + metric_name
    )

    ax.grid(
        axis="x",
        alpha=0.20,
    )

    fig.tight_layout()

    save_figure(
        fig,
        (
            "figure_pairwise_forest_"
            + metric_name
        ),
    )


# =============================================================================
# 17. 结果显示
# =============================================================================

display_table(
    model_summary.round(
        6
    ),
    "四模型跨 Landmark 等权平均性能",
)

display_table(
    pairwise_comparison.loc[
        pairwise_comparison[
            "scope"
        ]
        == (
            "primary_0_1_3_5y_"
            "landmark_equal_weight_mean"
        )
    ].round(
        6
    ),
    "主文 0/1/3/5 年 Landmark："
    "患者级配对 Bootstrap 模型差值",
)

display_table(
    calibrated_landmark_metrics.loc[
        calibrated_landmark_metrics[
            "landmark_month"
        ].isin(
            PRIMARY_LANDMARK_MONTHS
        )
    ].round(
        6
    ),
    "0、1、3、5 年 Landmark "
    "cross-fitted calibrated performance",
)


# =============================================================================
# 18. 保存元数据和最终摘要
# =============================================================================

input_fingerprint = hashlib.sha256()

for path in required_files:
    stat = path.stat()

    input_fingerprint.update(
        str(
            path
        ).encode(
            "utf-8"
        )
    )

    input_fingerprint.update(
        str(
            stat.st_size
        ).encode(
            "utf-8"
        )
    )

    input_fingerprint.update(
        str(
            stat.st_mtime_ns
        ).encode(
            "utf-8"
        )
    )

metadata = {
    "step": "11_v2",
    "analysis_name": (
        "Four-model cross-fitted OOF calibration "
        "and paired comparison with LSTM-v2"
    ),
    "models": (
        MODEL_ORDER
    ),
    "pairwise_comparisons": (
        PAIRWISE_COMPARISONS
    ),
    "project_dir": str(
        PROJECT_DIR
    ),
    "step6_dir": str(
        STEP6_DIR
    ),
    "cox_dir": str(
        COX_DIR
    ),
    "rsf_dir": str(
        RSF_DIR
    ),
    "rnn_dir": str(
        RNN_DIR
    ),
    "lstm_v2_dir": str(
        LSTM_V2_DIR
    ),
    "output_dir": str(
        OUTPUT_DIR
    ),
    "bootstrap_reps": int(
        BOOTSTRAP_REPS
    ),
    "bootstrap_unit": (
        "development patient"
    ),
    "bootstrap_pairing": (
        "same sampled development patients "
        "used for all models and all landmarks"
    ),
    "bootstrap_seed": int(
        RANDOM_SEED
    ),
    "calibration_method": (
        "five-fold cross-fitted discrete hazard "
        "recalibration with interval-specific "
        "intercepts and one common slope per "
        "model and landmark"
    ),
    "calibration_ridge": float(
        CALIBRATION_RIDGE
    ),
    "landmark_months": (
        LANDMARK_MONTHS.tolist()
    ),
    "future_end_months": (
        FUTURE_END_MONTHS.tolist()
    ),
    "metric_times": (
        METRIC_TIMES.tolist()
    ),
    "summary_scopes": {
        key: value.tolist()
        for (
            key,
            value,
        ) in SUMMARY_SCOPES.items()
    },
    "lstm_v2_selected_trial": (
        lstm_v2_summary.get(
            "selected_trial_number"
        )
    ),
    "lstm_v2_seed_ensemble_n_per_fold": (
        lstm_v2_summary.get(
            "seed_ensemble_n_per_fold"
        )
    ),
    "input_fingerprint_sha256": (
        input_fingerprint.hexdigest()
    ),
    "locked_test_used": False,
    "warning": (
        "This is development-cohort OOF performance. "
        "It must not be interpreted as locked internal-test "
        "or geographic external-validation performance."
    ),
}

save_json(
    metadata,
    OUTPUT_DIR
    / "step11_v2_metadata.json",
)

primary_summary_records = (
    model_summary.loc[
        model_summary[
            "scope"
        ]
        == (
            "primary_0_1_3_5y_"
            "landmark_equal_weight_mean"
        )
    ]
    .to_dict(
        orient="records"
    )
)

primary_pairwise_records = (
    pairwise_comparison.loc[
        pairwise_comparison[
            "scope"
        ]
        == (
            "primary_0_1_3_5y_"
            "landmark_equal_weight_mean"
        )
    ]
    .to_dict(
        orient="records"
    )
)

summary = {
    "analysis": (
        "development_oof_four_model_comparison"
    ),
    "models": (
        MODEL_ORDER
    ),
    "primary_scope": (
        "ART years 0, 1, 3, 5 equal-weight mean"
    ),
    "primary_model_performance": (
        primary_summary_records
    ),
    "primary_pairwise_comparison": (
        primary_pairwise_records
    ),
    "bootstrap_reps": int(
        BOOTSTRAP_REPS
    ),
    "locked_test_used": False,
    "output_dir": str(
        OUTPUT_DIR
    ),
}

save_json(
    summary,
    OUTPUT_DIR
    / "step11_v2_summary.json",
)


# =============================================================================
# 19. 完成
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "Step 11 v2 完成："
    "Cox / RSF / RNN / LSTM-v2 "
    "开发集 OOF 统一比较"
)

print(
    "=" * 110
)

print(
    "模型：",
    MODEL_ORDER,
)

print(
    "Bootstrap：",
    BOOTSTRAP_REPS,
)

print(
    "LSTM-v2 Trial：",
    lstm_v2_summary.get(
        "selected_trial_number"
    ),
)

print(
    "锁定测试集：未读取"
)

print(
    "输出目录：",
    OUTPUT_DIR,
)

print(
    "\n最重要结果文件："
)

for filename in [
    "four_model_equal_weight_mean_performance.csv",
    "four_model_paired_bootstrap_comparison.csv",
    "four_model_crossfit_calibrated_landmark_metrics.csv",
    "four_model_crossfit_calibrated_horizon_metrics.csv",
    "four_model_crossfit_calibrated_1y_3y_5y_metrics.csv",
    "four_model_calibration_overall.csv",
    "four_model_calibration_deciles.csv",
    "four_model_survival_dca.csv",
    "step11_v2_summary.json",
]:
    print(
        " -",
        OUTPUT_DIR
        / filename,
    )

print(
    "=" * 110
)


Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Step 12C：LSTM-v2 Grouped Clinical-Domain Reference Occlusion 五折OOF分析

研究目的
--------
1. 不重新训练模型，不读取锁定内部测试集。
2. 使用已经冻结的Step10E LSTM-v2五折OOF模型和Step11 cross-fit calibrator。
3. 在主Landmark（0、1、3、5年）中，以预先定义、互斥且覆盖全部模型输入的临床域为单位，
   对患者特异性输入进行reference occlusion，量化各临床信息域对预测性能的条件性贡献。
4. Group occlusion原则：
   - 对动态变量：在该Landmark所有有效历史时间点，将指定组相关动态通道替换为
     held-out fold之外训练风险集的Landmark×历史时间点reference；
   - 对实验室变量：value、observed、time-since-last、delta四个表示一起替换，
     避免只遮挡原始值却保留该变量派生表示；
   - 对静态变量：替换为该fold训练风险集Landmark-specific静态reference；
   - 对Age：替换为该fold训练风险集Landmark-specific标准化年龄reference；
   - row_mask、history length、Landmark context保持不变；
   - 原Step11 cross-fit calibrator冻结，不针对遮挡后预测重新校准。
5. 预先固定10个互斥临床域：
   renal function；HIV disease；hematologic；metabolic biochemistry；
   liver/viral coinfection；cardiometabolic comorbidities；metabolic medications；
   current ART regimen；cumulative ART exposure；demographic/anthropometric。
6. 主要结果：
   - ΔiAUC = baseline - occluded；
   - ΔIBS = occluded - baseline；
   - ΔUno C = baseline - occluded；
   - 5年风险绝对变化；
   - 6–60月dynamic AUC/Brier变化。
7. 全程保持OOF原则：每个患者只由其held-out fold模型预测。

方法学解释
----------
- 这是冻结模型后的post-hoc grouped input reference occlusion，不是重新训练后的ablation。
- 正值的ΔiAUC/ΔUno C/ΔIBS统一表示遮挡该组后预测性能变差。
- 不同组的occlusion效应不能简单相加，因为神经网络存在非线性和变量交互。
- 该步骤用于解释冻结LSTM对不同临床信息域的依赖；它不能替代后续
  Baseline-only / Current-only / Full-history重新训练的纵向增量价值分析。
"""

from __future__ import annotations

import gc
import json
import math
import os
import random
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn

import matplotlib.pyplot as plt
from scipy.special import expit
from sksurv.metrics import (
    brier_score,
    concordance_index_ipcw,
    cumulative_dynamic_auc,
    integrated_brier_score,
)
from sksurv.util import Surv


# =============================================================================
# 1. 固定路径与研究配置
# =============================================================================

PROJECT_DIR = Path(
    os.getenv("CKD_LSTM_PROJECT_DIR", "__CKD_WORKDIR__")
)

STEP1_DIR = PROJECT_DIR / "rolling_5y_step1_new_split"
STEP2_DIR = PROJECT_DIR / "rolling_5y_step2_folds"
STEP3_DIR = PROJECT_DIR / "rolling_5y_step3_raw_features"
STEP4_DIR = PROJECT_DIR / "rolling_5y_step4_preprocessed"
STEP6_DIR = PROJECT_DIR / "rolling_5y_step6_super_landmark_data"

STEP10E_DIR = (
    PROJECT_DIR
    / "rolling_5y_step10e_lstm_v2_final_oof_selected_existing_trials"
)

STEP11_DIR = (
    PROJECT_DIR
    / "rolling_5y_step11_v2_four_model_oof_comparison_lstm_v2"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "rolling_5y_step12c_lstm_v2_grouped_reference_occlusion_v1"
)
CHECKPOINT_DIR = OUTPUT_DIR / "prediction_checkpoints"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUTPUT_DIR / "step12c_live_progress.log"

EXPECTED_TOTAL_N = 31911
EXPECTED_DEVELOPMENT_N = 22337
EXPECTED_HISTORY_STEP_N = 11
EXPECTED_BASE_FEATURE_N = 57
EXPECTED_STATIC_N = 16
EXPECTED_BASE_DYNAMIC_N = 40
EXPECTED_LAB_N = 15
EXPECTED_EXTRA_DYNAMIC_N = EXPECTED_LAB_N * 3
EXPECTED_ENHANCED_DYNAMIC_N = EXPECTED_BASE_DYNAMIC_N + EXPECTED_EXTRA_DYNAMIC_N
EXPECTED_LANDMARK_N = 6
EXPECTED_FUTURE_INTERVAL_N = 10
EXPECTED_DEVELOPMENT_VALID_ORIGIN_N = 100122
N_SPLITS = 5
EPS = 1e-7

LANDMARK_MONTHS = np.asarray([0, 12, 24, 36, 48, 60], dtype=np.int32)
LANDMARK_BINS = (LANDMARK_MONTHS // 6).astype(np.int64)
TIME_STEP_YEARS = np.arange(EXPECTED_HISTORY_STEP_N, dtype=np.float32) * 0.5

PRIMARY_LANDMARK_INDICES = np.asarray([0, 1, 3, 5], dtype=np.int64)
PRIMARY_LANDMARK_MONTHS = LANDMARK_MONTHS[PRIMARY_LANDMARK_INDICES]

FUTURE_END_MONTHS = np.arange(6, 61, 6, dtype=np.float64)
METRIC_TIMES = np.asarray(
    [6, 12, 18, 24, 30, 36, 42, 48, 54, 59.999],
    dtype=np.float64,
)

# 解释分析采用确定性的全精度FP32前向。
# 与Step12A IG一致：正式AMP保存预测只用于审计；perturbation前后必须使用相同数值路径。
INFERENCE_BATCH_SIZE_OVERRIDE = int(
    os.getenv("CKD_GROUP_OCCLUSION_BATCH_SIZE", "0")
)
RESUME = os.getenv("CKD_GROUP_OCCLUSION_RESUME", "1") != "0"
RANDOM_SEED = int(os.getenv("CKD_GROUP_OCCLUSION_RANDOM_SEED", "20260812"))

# FP32 baseline与正式Step11 AMP保存风险之间允许极小数值漂移。
# 判据使用mean/p99，不让单个极端浮点值误触发。
FP32_BASELINE_MEAN_HARD_TOL = 1e-4
FP32_BASELINE_P99_HARD_TOL = 5e-4

PIPELINE_VERSION = "step12c_grouped_reference_occlusion_v1"

CALIBRATION_FILE = (
    STEP11_DIR / "four_model_hazard_calibration_parameters.csv"
)
CALIBRATED_RISK_FILE = (
    STEP11_DIR / "lstm_v2_crossfit_calibrated_oof_risk_long.npy"
)


# =============================================================================
# 2. 模型输入特征定义——与Step10E完全一致
# =============================================================================

LAB_FEATURES = [
    "HIVRNA_log10",
    "CD4",
    "CD8",
    "Urea",
    "WBC",
    "PLT",
    "HB",
    "TC",
    "TG",
    "HDL",
    "LDL",
    "GLU",
    "ALT",
    "AST",
    "eGFR",
]

PERSISTENT_STATUS_FEATURES = [
    "CVD_status",
    "diabetes_status",
    "hypertension_status",
    "hypercholesterolemia_status",
    "HBV_status",
    "HCV_status",
]

METABOLIC_MED_FEATURES = [
    "antidiabetic_med",
    "antihypertensive_med",
    "antilipid_med",
]

CURRENT_ART_FEATURES = [
    "current_TDF_NNRTI_3TC_FTC",
    "current_TDF_PI_3TC_FTC",
    "current_nonTDF_PI",
    "current_BIC_FTC_TAF",
    "current_EVGc_FTC_TAF",
    "current_TDF_INSTI_3TC_FTC",
    "current_nonTDF_DTG",
    "current_nonTDF_traditional_NNRTI",
]

CUMULATIVE_ART_FEATURES = [
    "TDF_NNRTI_3TC_FTC_cum_month",
    "TDF_PI_3TC_FTC_cum_month",
    "nonTDF_PI_cum_month",
    "BIC_FTC_TAF_cum_month",
    "EVGc_FTC_TAF_cum_month",
    "TDF_INSTI_3TC_FTC_cum_month",
    "nonTDF_DTG_cum_month",
    "nonTDF_traditional_NNRTI_cum_month",
]

DYNAMIC_FEATURES = (
    LAB_FEATURES
    + PERSISTENT_STATUS_FEATURES
    + METABOLIC_MED_FEATURES
    + CURRENT_ART_FEATURES
    + CUMULATIVE_ART_FEATURES
)


# =============================================================================
# 3. 通用函数
# =============================================================================

def set_random_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def require_files(paths: list[Path]) -> None:
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "以下必要文件不存在：\n" + "\n".join(missing)
        )


def save_json(value: Any, path: Path) -> None:
    path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )


def format_duration(seconds: float) -> str:
    seconds = max(0, int(round(float(seconds))))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    if hours:
        return f"{hours}小时{minutes:02d}分{seconds:02d}秒"
    if minutes:
        return f"{minutes}分{seconds:02d}秒"
    return f"{seconds}秒"


def progress_print(message: str) -> None:
    line = (
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        f"{message}"
    )
    print(line, flush=True)
    with LOG_FILE.open("a", encoding="utf-8") as file:
        file.write(line + "\n")
        file.flush()


def torch_load_full(
    path: Path,
    map_location: str | torch.device = "cpu",
):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


def default_device() -> torch.device:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "未检测到CUDA；Step12C建议在GPU环境运行。"
        )
    device = torch.device("cuda")
    torch.set_num_threads(min(8, os.cpu_count() or 1))
    torch.cuda.empty_cache()
    return device


# =============================================================================
# 4. 共同数据
# =============================================================================

@dataclass
class CommonData:
    development_idx: np.ndarray
    development_fold_id: np.ndarray
    sequence_row_mask: np.ndarray
    prediction_origin_mask: np.ndarray
    future_event_matrix: np.ndarray
    future_at_risk_mask: np.ndarray
    continuous_raw: np.ndarray
    continuous_vars: list[str]
    age_continuous_index: int
    lab_continuous_indices: np.ndarray
    long_row_index_map: np.ndarray


def load_common_data() -> CommonData:
    required = [
        STEP1_DIR / "development_idx.npy",
        STEP1_DIR / "prediction_origin_mask.npy",
        STEP1_DIR / "future_event_matrix.npy",
        STEP1_DIR / "future_at_risk_mask.npy",
        STEP1_DIR / "landmark_months.npy",
        STEP1_DIR / "landmark_bins.npy",
        STEP2_DIR / "development_fold_id.npy",
        STEP3_DIR / "continuous_raw_0_60.npy",
        STEP3_DIR / "feature_groups.json",
        STEP4_DIR / "development_idx.npy",
        STEP4_DIR / "development_fold_id.npy",
        STEP4_DIR / "sequence_row_mask_development.npy",
        STEP6_DIR / "development_long_row_index_map.npy",
    ]
    for fold_id in range(N_SPLITS):
        fold_dir = STEP4_DIR / f"fold_{fold_id}"
        required.extend(
            [
                fold_dir / "X_development.npy",
                fold_dir / "preprocessor.joblib",
                fold_dir / "feature_names.csv",
            ]
        )
    require_files(required)

    development_idx_step1 = np.load(STEP1_DIR / "development_idx.npy").astype(np.int32)
    development_idx = np.load(STEP4_DIR / "development_idx.npy").astype(np.int32)
    fold_step2 = np.load(STEP2_DIR / "development_fold_id.npy").astype(np.int8)
    development_fold_id = np.load(STEP4_DIR / "development_fold_id.npy").astype(np.int8)

    if not np.array_equal(development_idx_step1, development_idx):
        raise ValueError("Step1与Step4开发集顺序不一致。")
    if not np.array_equal(fold_step2, development_fold_id):
        raise ValueError("Step2与Step4固定五折不一致。")

    sequence_row_mask = np.load(
        STEP4_DIR / "sequence_row_mask_development.npy"
    ).astype(bool)

    prediction_origin_mask_all = np.load(
        STEP1_DIR / "prediction_origin_mask.npy"
    ).astype(bool)
    future_event_all = np.load(STEP1_DIR / "future_event_matrix.npy").astype(np.float32)
    future_at_risk_all = np.load(
        STEP1_DIR / "future_at_risk_mask.npy"
    ).astype(bool)

    landmark_months_file = np.load(STEP1_DIR / "landmark_months.npy").astype(np.int32)
    landmark_bins_file = np.load(STEP1_DIR / "landmark_bins.npy").astype(np.int64)
    if not np.array_equal(landmark_months_file, LANDMARK_MONTHS):
        raise ValueError("Landmark月份配置与Step10E不一致。")
    if not np.array_equal(landmark_bins_file, LANDMARK_BINS):
        raise ValueError("Landmark时间行配置与Step10E不一致。")

    continuous_raw = np.load(
        STEP3_DIR / "continuous_raw_0_60.npy",
        mmap_mode="r",
    )
    feature_groups = json.loads(
        (STEP3_DIR / "feature_groups.json").read_text(encoding="utf-8")
    )
    continuous_vars = list(feature_groups["continuous_vars"])
    if "Age" not in continuous_vars:
        raise ValueError("连续变量中缺少Age。")
    missing_labs = sorted(set(LAB_FEATURES) - set(continuous_vars))
    if missing_labs:
        raise ValueError(f"连续变量中缺少实验室指标：{missing_labs}")

    age_continuous_index = continuous_vars.index("Age")
    lab_continuous_indices = np.asarray(
        [continuous_vars.index(name) for name in LAB_FEATURES],
        dtype=np.int64,
    )

    long_row_index_map = np.load(
        STEP6_DIR / "development_long_row_index_map.npy"
    ).astype(np.int32)

    prediction_origin_mask = prediction_origin_mask_all[development_idx]
    future_event_matrix = future_event_all[development_idx]
    future_at_risk_mask = future_at_risk_all[development_idx]

    checks = [
        (development_idx.shape, (EXPECTED_DEVELOPMENT_N,), "development_idx"),
        (development_fold_id.shape, (EXPECTED_DEVELOPMENT_N,), "development_fold_id"),
        (
            sequence_row_mask.shape,
            (EXPECTED_DEVELOPMENT_N, EXPECTED_HISTORY_STEP_N),
            "sequence_row_mask",
        ),
        (
            prediction_origin_mask.shape,
            (EXPECTED_DEVELOPMENT_N, EXPECTED_LANDMARK_N),
            "prediction_origin_mask",
        ),
        (
            future_event_matrix.shape,
            (
                EXPECTED_DEVELOPMENT_N,
                EXPECTED_LANDMARK_N,
                EXPECTED_FUTURE_INTERVAL_N,
            ),
            "future_event_matrix",
        ),
        (
            long_row_index_map.shape,
            (EXPECTED_DEVELOPMENT_N, EXPECTED_LANDMARK_N),
            "long_row_index_map",
        ),
    ]
    for actual, expected, name in checks:
        if actual != expected:
            raise ValueError(f"{name}形状={actual}，预期={expected}。")

    if int(prediction_origin_mask.sum()) != EXPECTED_DEVELOPMENT_VALID_ORIGIN_N:
        raise ValueError("有效患者-Landmark记录数不是100122。")
    if not np.array_equal(prediction_origin_mask, long_row_index_map >= 0):
        raise ValueError("有效Landmark与长格式映射不一致。")

    return CommonData(
        development_idx=development_idx,
        development_fold_id=development_fold_id,
        sequence_row_mask=sequence_row_mask,
        prediction_origin_mask=prediction_origin_mask,
        future_event_matrix=future_event_matrix,
        future_at_risk_mask=future_at_risk_mask,
        continuous_raw=continuous_raw,
        continuous_vars=continuous_vars,
        age_continuous_index=age_continuous_index,
        lab_continuous_indices=lab_continuous_indices,
        long_row_index_map=long_row_index_map,
    )


# =============================================================================
# 5. 强化纵向特征——与Step10E完全一致
# =============================================================================

def build_enhanced_dynamic_array(
    base_dynamic: np.ndarray,
    lab_raw_development: np.ndarray,
    sequence_row_mask: np.ndarray,
) -> tuple[np.ndarray, list[str]]:
    base_dynamic = np.asarray(base_dynamic, dtype=np.float32)
    lab_raw_development = np.asarray(lab_raw_development, dtype=np.float32)
    sequence_row_mask = np.asarray(sequence_row_mask, dtype=bool)

    expected_base_shape = (
        EXPECTED_DEVELOPMENT_N,
        EXPECTED_HISTORY_STEP_N,
        EXPECTED_BASE_DYNAMIC_N,
    )
    expected_lab_shape = (
        EXPECTED_DEVELOPMENT_N,
        EXPECTED_HISTORY_STEP_N,
        EXPECTED_LAB_N,
    )
    if base_dynamic.shape != expected_base_shape:
        raise ValueError(f"基础动态特征形状={base_dynamic.shape}，预期={expected_base_shape}。")
    if lab_raw_development.shape != expected_lab_shape:
        raise ValueError(f"原始实验室形状={lab_raw_development.shape}，预期={expected_lab_shape}。")

    lab_observed = (
        np.isfinite(lab_raw_development)
        & sequence_row_mask[:, :, None]
    )
    lab_observed_float = lab_observed.astype(np.float32)
    time_since = np.zeros_like(lab_observed_float, dtype=np.float32)
    lab_delta = np.zeros_like(lab_observed_float, dtype=np.float32)

    standardized_labs = base_dynamic[:, :, :EXPECTED_LAB_N]
    last_seen_step = np.full(
        (EXPECTED_DEVELOPMENT_N, EXPECTED_LAB_N),
        -1,
        dtype=np.int16,
    )
    last_seen_value = np.zeros(
        (EXPECTED_DEVELOPMENT_N, EXPECTED_LAB_N),
        dtype=np.float32,
    )
    has_seen = np.zeros(
        (EXPECTED_DEVELOPMENT_N, EXPECTED_LAB_N),
        dtype=bool,
    )

    for step_index in range(EXPECTED_HISTORY_STEP_N):
        active_row = sequence_row_mask[:, step_index][:, None]
        observed_now = lab_observed[:, step_index, :]
        current_value = standardized_labs[:, step_index, :]

        elapsed_years = (
            step_index - last_seen_step
        ).astype(np.float32) * 0.5
        elapsed_years = np.clip(elapsed_years, 0.0, 5.0)
        elapsed_years[~has_seen] = min((step_index + 1) * 0.5, 5.0)

        time_since[:, step_index, :] = np.where(
            active_row,
            np.where(observed_now, 0.0, elapsed_years / 5.0),
            0.0,
        )

        delta_now = current_value - last_seen_value
        lab_delta[:, step_index, :] = np.where(
            observed_now & has_seen,
            delta_now,
            0.0,
        )

        last_seen_value = np.where(observed_now, current_value, last_seen_value)
        last_seen_step = np.where(observed_now, step_index, last_seen_step)
        has_seen |= observed_now

    enhanced = np.concatenate(
        [base_dynamic, lab_observed_float, time_since, lab_delta],
        axis=2,
    ).astype(np.float32)
    enhanced[~sequence_row_mask] = 0.0

    if enhanced.shape != (
        EXPECTED_DEVELOPMENT_N,
        EXPECTED_HISTORY_STEP_N,
        EXPECTED_ENHANCED_DYNAMIC_N,
    ):
        raise ValueError("强化动态特征形状错误。")
    if not np.isfinite(enhanced).all():
        raise ValueError("强化动态特征存在NaN或无穷值。")

    names = (
        DYNAMIC_FEATURES
        + [f"{name}_observed" for name in LAB_FEATURES]
        + [f"{name}_time_since_last" for name in LAB_FEATURES]
        + [f"{name}_delta_last_observed" for name in LAB_FEATURES]
    )
    if len(names) != EXPECTED_ENHANCED_DYNAMIC_N:
        raise ValueError("强化动态特征名称数错误。")
    return enhanced, names


# =============================================================================
# 6. Fold输入数据
# =============================================================================

@dataclass
class FoldArrays:
    fold_id: int
    enhanced_dynamic: np.ndarray
    enhanced_dynamic_names: list[str]
    static_baseline: np.ndarray
    static_names: list[str]
    baseline_age_raw: np.ndarray
    age_mean: float
    age_scale: float
    # Reference按Landmark训练风险集分别构建。
    dynamic_reference_by_landmark: dict[int, np.ndarray]
    static_reference_by_landmark: dict[int, np.ndarray]
    age_reference_by_landmark: dict[int, float]


def prepare_fold_arrays(common: CommonData, fold_id: int) -> FoldArrays:
    fold_dir = STEP4_DIR / f"fold_{fold_id}"
    x_development = np.load(fold_dir / "X_development.npy", mmap_mode="r")
    feature_names = (
        pd.read_csv(fold_dir / "feature_names.csv", encoding="utf-8-sig")["feature_name"]
        .astype(str)
        .tolist()
    )
    preprocessor = joblib.load(fold_dir / "preprocessor.joblib")

    if x_development.shape != (
        EXPECTED_DEVELOPMENT_N,
        EXPECTED_HISTORY_STEP_N,
        EXPECTED_BASE_FEATURE_N,
    ):
        raise ValueError(f"第{fold_id}折预处理张量形状错误：{x_development.shape}")
    if len(feature_names) != EXPECTED_BASE_FEATURE_N:
        raise ValueError(f"第{fold_id}折预处理特征数不是57。")

    feature_to_index = {name: index for index, name in enumerate(feature_names)}
    onehot_static = [
        name
        for name in feature_names
        if (
            name.startswith("Sex_")
            or name.startswith("Marriage_")
            or name.startswith("Course_")
            or name.startswith("WHOstage_")
        )
    ]
    static_names = ["BMI", "Oppinfection", *onehot_static]
    if len(static_names) != EXPECTED_STATIC_N:
        raise ValueError(
            f"第{fold_id}折静态特征数={len(static_names)}，应为16。"
        )

    required_names = {"Age", *static_names, *DYNAMIC_FEATURES}
    missing = sorted(required_names - set(feature_names))
    if missing:
        raise ValueError(f"第{fold_id}折缺少模型输入特征：{missing}")

    static_indices = np.asarray(
        [feature_to_index[name] for name in static_names],
        dtype=np.int64,
    )
    dynamic_indices = np.asarray(
        [feature_to_index[name] for name in DYNAMIC_FEATURES],
        dtype=np.int64,
    )

    first_observed_step = np.argmax(common.sequence_row_mask, axis=1).astype(np.int64)
    if (~common.sequence_row_mask.any(axis=1)).any():
        raise ValueError("部分开发集患者没有任何历史时间行。")

    patient_local = np.arange(EXPECTED_DEVELOPMENT_N, dtype=np.int64)
    static_baseline = np.asarray(
        x_development[
            patient_local,
            first_observed_step,
            :,
        ][:, static_indices],
        dtype=np.float32,
    )

    age_first = np.asarray(
        common.continuous_raw[
            common.development_idx,
            first_observed_step,
            common.age_continuous_index,
        ],
        dtype=np.float32,
    )
    baseline_age_raw = (
        age_first - TIME_STEP_YEARS[first_observed_step]
    ).astype(np.float32)
    if not np.isfinite(baseline_age_raw).all():
        raise ValueError("基线年龄存在缺失或非法值。")

    age_mean = float(preprocessor["scaler"].mean_[common.age_continuous_index])
    age_scale = float(preprocessor["scaler"].scale_[common.age_continuous_index])
    if not np.isfinite(age_mean) or not np.isfinite(age_scale) or age_scale <= 0:
        raise ValueError("年龄标准化参数无效。")

    base_dynamic = np.asarray(
        x_development[:, :, dynamic_indices],
        dtype=np.float32,
    )
    lab_raw_dev = np.asarray(
        common.continuous_raw[
            common.development_idx,
            :,
            :,
        ][:, :, common.lab_continuous_indices],
        dtype=np.float32,
    )
    enhanced_dynamic, enhanced_names = build_enhanced_dynamic_array(
        base_dynamic=base_dynamic,
        lab_raw_development=lab_raw_dev,
        sequence_row_mask=common.sequence_row_mask,
    )

    # -------------------------------------------------------------------------
    # Fold-specific reference：
    # - 只能使用该fold训练患者；
    # - 并且必须限定在“当前Landmark仍处于风险集”的训练患者。
    # 这样reference代表该预测时点的典型训练风险集患者，而不会混入已经
    # 在更早时间发生CKD/离开风险集的患者。
    # -------------------------------------------------------------------------
    train_patient_mask = common.development_fold_id != fold_id

    dynamic_reference_by_landmark: dict[int, np.ndarray] = {}
    static_reference_by_landmark: dict[int, np.ndarray] = {}
    age_reference_by_landmark: dict[int, float] = {}

    for landmark_index in range(EXPECTED_LANDMARK_N):
        train_origin = (
            train_patient_mask
            & common.prediction_origin_mask[:, landmark_index]
        )
        if not np.any(train_origin):
            raise ValueError(
                f"第{fold_id}折Landmark {LANDMARK_MONTHS[landmark_index]}月"
                "没有训练风险集。"
            )

        landmark_reference = np.zeros(
            (EXPECTED_HISTORY_STEP_N, EXPECTED_ENHANCED_DYNAMIC_N),
            dtype=np.float32,
        )
        max_step = int(LANDMARK_BINS[landmark_index])

        for step_index in range(max_step + 1):
            active_train_origin = (
                train_origin
                & common.sequence_row_mask[:, step_index]
            )
            if not np.any(active_train_origin):
                raise ValueError(
                    f"第{fold_id}折Landmark {LANDMARK_MONTHS[landmark_index]}月、"
                    f"历史时间步{step_index}没有训练参考记录。"
                )
            landmark_reference[step_index] = np.mean(
                enhanced_dynamic[
                    active_train_origin,
                    step_index,
                    :,
                ],
                axis=0,
                dtype=np.float64,
            ).astype(np.float32)

        dynamic_reference_by_landmark[landmark_index] = landmark_reference

        static_reference_by_landmark[landmark_index] = np.mean(
            static_baseline[train_origin],
            axis=0,
            dtype=np.float64,
        ).astype(np.float32)

        age_raw = (
            baseline_age_raw[train_origin]
            + float(LANDMARK_MONTHS[landmark_index]) / 12.0
        )
        age_standardized = (age_raw - age_mean) / age_scale
        age_reference_by_landmark[landmark_index] = float(
            np.mean(age_standardized, dtype=np.float64)
        )

    return FoldArrays(
        fold_id=int(fold_id),
        enhanced_dynamic=enhanced_dynamic,
        enhanced_dynamic_names=enhanced_names,
        static_baseline=static_baseline,
        static_names=static_names,
        baseline_age_raw=baseline_age_raw,
        age_mean=age_mean,
        age_scale=age_scale,
        dynamic_reference_by_landmark=dynamic_reference_by_landmark,
        static_reference_by_landmark=static_reference_by_landmark,
        age_reference_by_landmark=age_reference_by_landmark,
    )


# =============================================================================
# 7. Hybrid Attention LSTM——与Step10E完全一致
# =============================================================================

class HybridAttentionLSTMSurvival(nn.Module):
    def __init__(
        self,
        dynamic_n: int,
        static_n: int,
        hidden_size: int,
        num_layers: int,
        dropout: float,
        projection_size: int,
        bidirectional: bool,
        pooling_mode: str,
        static_hidden: int,
        summary_hidden: int,
        horizon_embed_dim: int,
        future_n: int,
    ):
        super().__init__()
        self.dynamic_n = int(dynamic_n)
        self.static_n = int(static_n)
        self.hidden_size = int(hidden_size)
        self.num_layers = int(num_layers)
        self.bidirectional = bool(bidirectional)
        self.pooling_mode = str(pooling_mode)
        self.future_n = int(future_n)
        self.direction_n = 2 if self.bidirectional else 1
        self.representation_n = self.hidden_size * self.direction_n

        if self.pooling_mode not in {"last_attention", "last_attention_mean"}:
            raise ValueError("pooling_mode无效。")

        self.input_encoder = nn.Sequential(
            nn.LayerNorm(self.dynamic_n + 2),
            nn.Linear(self.dynamic_n + 2, int(projection_size)),
            nn.SiLU(),
            nn.Dropout(float(dropout)),
        )

        self.forward_cells = nn.ModuleList(
            [
                nn.LSTMCell(
                    input_size=(
                        int(projection_size)
                        if layer_index == 0
                        else self.hidden_size
                    ),
                    hidden_size=self.hidden_size,
                )
                for layer_index in range(self.num_layers)
            ]
        )
        if self.bidirectional:
            self.backward_cells = nn.ModuleList(
                [
                    nn.LSTMCell(
                        input_size=(
                            int(projection_size)
                            if layer_index == 0
                            else self.hidden_size
                        ),
                        hidden_size=self.hidden_size,
                    )
                    for layer_index in range(self.num_layers)
                ]
            )
        else:
            self.backward_cells = None

        self.recurrent_dropout = nn.Dropout(float(dropout))
        self.attention_hidden = nn.Linear(
            self.representation_n,
            self.representation_n,
            bias=False,
        )
        self.attention_query = nn.Linear(
            self.representation_n,
            self.representation_n,
            bias=False,
        )
        self.attention_score = nn.Linear(
            self.representation_n,
            1,
            bias=False,
        )

        self.static_encoder = nn.Sequential(
            nn.LayerNorm(self.static_n),
            nn.Linear(self.static_n, int(static_hidden)),
            nn.SiLU(),
            nn.Dropout(float(dropout)),
        )
        self.dynamic_summary_encoder = nn.Sequential(
            nn.LayerNorm(self.dynamic_n * 2),
            nn.Linear(self.dynamic_n * 2, int(summary_hidden)),
            nn.SiLU(),
            nn.Dropout(float(dropout)),
        )

        recurrent_context_n = self.representation_n * 2
        if self.pooling_mode == "last_attention_mean":
            recurrent_context_n += self.representation_n

        context_n = (
            recurrent_context_n
            + int(static_hidden)
            + int(summary_hidden)
            + 2
        )
        self.context_encoder = nn.Sequential(
            nn.LayerNorm(context_n),
            nn.Linear(context_n, self.hidden_size),
            nn.SiLU(),
            nn.Dropout(float(dropout)),
        )

        self.horizon_embedding = nn.Embedding(
            self.future_n,
            int(horizon_embed_dim),
        )
        head_input_n = self.hidden_size + int(horizon_embed_dim)
        head_hidden_n = max(32, self.hidden_size // 2)
        self.hazard_head = nn.Sequential(
            nn.LayerNorm(head_input_n),
            nn.Linear(head_input_n, head_hidden_n),
            nn.SiLU(),
            nn.Dropout(float(dropout)),
            nn.Linear(head_hidden_n, 1),
        )
        self.interval_bias = nn.Parameter(
            torch.zeros(self.future_n, dtype=torch.float32)
        )

    def _run_direction(
        self,
        encoded_sequence: torch.Tensor,
        active_mask: torch.Tensor,
        cells: nn.ModuleList,
        reverse: bool,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch_n, time_n, _ = encoded_sequence.shape
        hidden = [
            torch.zeros(
                batch_n,
                self.hidden_size,
                dtype=encoded_sequence.dtype,
                device=encoded_sequence.device,
            )
            for _ in range(self.num_layers)
        ]
        cell = [torch.zeros_like(hidden[0]) for _ in range(self.num_layers)]
        history = [None] * time_n

        indices = range(time_n - 1, -1, -1) if reverse else range(time_n)
        for step_index in indices:
            step_active = active_mask[:, step_index].unsqueeze(1)
            layer_input = encoded_sequence[:, step_index, :]
            for layer_index, recurrent_cell in enumerate(cells):
                candidate_hidden, candidate_cell = recurrent_cell(
                    layer_input,
                    (hidden[layer_index], cell[layer_index]),
                )
                hidden[layer_index] = torch.where(
                    step_active,
                    candidate_hidden,
                    hidden[layer_index],
                )
                cell[layer_index] = torch.where(
                    step_active,
                    candidate_cell,
                    cell[layer_index],
                )
                layer_input = hidden[layer_index]
                if layer_index < self.num_layers - 1:
                    layer_input = self.recurrent_dropout(layer_input)
            history[step_index] = hidden[-1]
        return torch.stack(history, dim=1), hidden[-1]

    def forward(
        self,
        dynamic_sequence: torch.Tensor,
        row_mask: torch.Tensor,
        static_baseline: torch.Tensor,
        age_at_landmark: torch.Tensor,
        landmark_normalized: torch.Tensor,
        landmark_bin: torch.Tensor,
    ) -> torch.Tensor:
        batch_n, time_n, dynamic_n = dynamic_sequence.shape
        if dynamic_n != self.dynamic_n:
            raise ValueError("模型收到的动态特征数不正确。")

        step_indices = torch.arange(time_n, device=dynamic_sequence.device)
        active_mask = (
            row_mask
            & (step_indices[None, :] <= landmark_bin[:, None])
        )
        if (~active_mask.any(dim=1)).any():
            raise ValueError("部分样本在Landmark前没有有效历史行。")

        time_normalized = (
            step_indices.float() / float(max(time_n - 1, 1))
        ).expand(batch_n, time_n)

        previous_active = torch.full(
            (batch_n,),
            -1,
            dtype=torch.long,
            device=dynamic_sequence.device,
        )
        gap_values = []
        for step_index in range(time_n):
            current_active = active_mask[:, step_index]
            gap = torch.where(
                current_active & (previous_active >= 0),
                (step_index - previous_active).float()
                / float(max(time_n - 1, 1)),
                torch.zeros(
                    batch_n,
                    dtype=dynamic_sequence.dtype,
                    device=dynamic_sequence.device,
                ),
            )
            gap_values.append(gap)
            previous_active = torch.where(
                current_active,
                torch.full_like(previous_active, step_index),
                previous_active,
            )
        gap_normalized = torch.stack(gap_values, dim=1)

        encoded_sequence = self.input_encoder(
            torch.cat(
                [
                    dynamic_sequence,
                    time_normalized.unsqueeze(2),
                    gap_normalized.unsqueeze(2),
                ],
                dim=2,
            )
        )

        forward_history, forward_final = self._run_direction(
            encoded_sequence,
            active_mask,
            self.forward_cells,
            reverse=False,
        )

        if self.bidirectional:
            backward_history, backward_final = self._run_direction(
                encoded_sequence,
                active_mask,
                self.backward_cells,
                reverse=True,
            )
            recurrent_history = torch.cat(
                [forward_history, backward_history],
                dim=2,
            )
            final_hidden = torch.cat([forward_final, backward_final], dim=1)
        else:
            recurrent_history = forward_history
            final_hidden = forward_final

        attention_logits = self.attention_score(
            torch.tanh(
                self.attention_hidden(recurrent_history)
                + self.attention_query(final_hidden).unsqueeze(1)
            )
        ).squeeze(2)
        attention_logits = attention_logits.masked_fill(~active_mask, -1e4)
        attention_weight = torch.softmax(attention_logits, dim=1)
        attention_pool = torch.sum(
            recurrent_history * attention_weight.unsqueeze(2),
            dim=1,
        )

        active_float = active_mask.unsqueeze(2).to(dynamic_sequence.dtype)
        active_count = active_float.sum(dim=1).clamp_min(1.0)
        recurrent_mean = (
            recurrent_history * active_float
        ).sum(dim=1) / active_count
        dynamic_mean = (
            dynamic_sequence * active_float
        ).sum(dim=1) / active_count

        last_position = (
            active_mask.long() * (step_indices[None, :] + 1)
        ).argmax(dim=1)
        batch_index = torch.arange(batch_n, device=dynamic_sequence.device)
        dynamic_last = dynamic_sequence[
            batch_index,
            last_position,
            :,
        ]

        static_encoded = self.static_encoder(static_baseline)
        summary_encoded = self.dynamic_summary_encoder(
            torch.cat([dynamic_last, dynamic_mean], dim=1)
        )

        recurrent_parts = [final_hidden, attention_pool]
        if self.pooling_mode == "last_attention_mean":
            recurrent_parts.append(recurrent_mean)

        context = self.context_encoder(
            torch.cat(
                [
                    *recurrent_parts,
                    static_encoded,
                    summary_encoded,
                    age_at_landmark.unsqueeze(1),
                    landmark_normalized.unsqueeze(1),
                ],
                dim=1,
            )
        )

        horizon_index = torch.arange(self.future_n, device=dynamic_sequence.device)
        horizon_embedding = self.horizon_embedding(
            horizon_index
        ).unsqueeze(0).expand(batch_n, -1, -1)
        context_expanded = context.unsqueeze(1).expand(-1, self.future_n, -1)
        head_input = torch.cat([context_expanded, horizon_embedding], dim=2)
        logits = self.hazard_head(head_input).squeeze(2)
        return logits + self.interval_bias.unsqueeze(0)


def build_model(parameters: dict[str, Any], device: torch.device) -> HybridAttentionLSTMSurvival:
    return HybridAttentionLSTMSurvival(
        dynamic_n=EXPECTED_ENHANCED_DYNAMIC_N,
        static_n=EXPECTED_STATIC_N,
        hidden_size=int(parameters["hidden_size"]),
        num_layers=int(parameters["num_layers"]),
        dropout=float(parameters["dropout"]),
        projection_size=int(parameters["projection_size"]),
        bidirectional=bool(parameters["bidirectional"]),
        pooling_mode=str(parameters["pooling_mode"]),
        static_hidden=int(parameters["static_hidden"]),
        summary_hidden=int(parameters["summary_hidden"]),
        horizon_embed_dim=int(parameters["horizon_embed_dim"]),
        future_n=EXPECTED_FUTURE_INTERVAL_N,
    ).to(device)




def load_crossfit_calibrator(
    calibration_table: pd.DataFrame,
    fold_id: int,
    landmark_month: int,
) -> tuple[np.ndarray, float]:
    sub = calibration_table.loc[
        (calibration_table["model"] == "LSTM-v2")
        & (calibration_table["fit_scope"] == "crossfit")
        & (calibration_table["heldout_fold"] == fold_id)
        & np.isclose(calibration_table["landmark_month"], float(landmark_month))
    ].sort_values("interval_position")

    if len(sub) != EXPECTED_FUTURE_INTERVAL_N:
        raise ValueError(
            f"Fold {fold_id}, Landmark {landmark_month}月校准参数不是10行。"
        )
    expected_positions = np.arange(EXPECTED_FUTURE_INTERVAL_N)
    if not np.array_equal(sub["interval_position"].to_numpy(int), expected_positions):
        raise ValueError("校准区间顺序错误。")

    beta_values = sub["beta_common_slope"].to_numpy(float)
    if not np.allclose(beta_values, beta_values[0], rtol=0, atol=1e-10):
        raise ValueError("同一fold-landmark的校准beta不一致。")

    alpha = sub["alpha_interval"].to_numpy(dtype=np.float64)
    beta = float(beta_values[0])
    return alpha, beta


def build_model_inputs_for_sample_pairs(
    common: CommonData,
    fold_arrays: FoldArrays,
    sample_pairs: np.ndarray,
) -> dict[str, np.ndarray]:
    """
    为任意[patient_local, landmark_index]组合重建Step10E模型输入。
    用于严格的fold级模型重建审计；不构造IG baseline。
    """
    sample_pairs = np.asarray(sample_pairs, dtype=np.int64)
    if sample_pairs.ndim != 2 or sample_pairs.shape[1] != 2:
        raise ValueError("sample_pairs必须为[n,2]。")

    patient_local = sample_pairs[:, 0]
    landmark_index = sample_pairs[:, 1]
    if np.any((landmark_index < 0) | (landmark_index >= EXPECTED_LANDMARK_N)):
        raise ValueError("sample_pairs存在非法landmark_index。")

    landmark_month = LANDMARK_MONTHS[landmark_index].astype(np.float32)

    dynamic = np.asarray(
        fold_arrays.enhanced_dynamic[patient_local],
        dtype=np.float32,
    )
    row_mask = np.asarray(
        common.sequence_row_mask[patient_local],
        dtype=bool,
    )
    static = np.asarray(
        fold_arrays.static_baseline[patient_local],
        dtype=np.float32,
    )
    age_raw = (
        fold_arrays.baseline_age_raw[patient_local]
        + landmark_month / 12.0
    )
    age = (
        (age_raw - fold_arrays.age_mean)
        / fold_arrays.age_scale
    ).astype(np.float32)

    landmark_norm = (
        landmark_month / 60.0
    ).astype(np.float32)
    landmark_bin = LANDMARK_BINS[
        landmark_index
    ].astype(np.int64)

    return {
        "dynamic": dynamic,
        "row_mask": row_mask,
        "static": static,
        "age": age,
        "landmark_norm": landmark_norm,
        "landmark_bin": landmark_bin,
    }


def np_step10e_hazard_to_survival(hazard: np.ndarray) -> np.ndarray:
    """与Step10E hazards_to_survival()一致。"""
    hazard64 = np.clip(
        np.asarray(hazard, dtype=np.float64),
        EPS,
        1.0 - EPS,
    )
    return np.cumprod(
        1.0 - hazard64,
        axis=1,
    ).astype(np.float32)


def np_step11_survival_to_hazard(survival: np.ndarray) -> np.ndarray:
    """与Step11 survival_to_hazard()一致。"""
    survival64 = np.clip(
        np.asarray(survival, dtype=np.float64),
        EPS,
        1.0,
    )
    previous_survival = np.concatenate(
        [
            np.ones(
                (survival64.shape[0], 1),
                dtype=np.float64,
            ),
            survival64[:, :-1],
        ],
        axis=1,
    )
    hazard = (
        1.0
        - survival64
        / np.clip(previous_survival, EPS, 1.0)
    )
    return np.clip(
        hazard,
        EPS,
        1.0 - EPS,
    ).astype(np.float32)


def np_step11_apply_calibrator(
    hazard: np.ndarray,
    alpha: np.ndarray,
    beta: float,
) -> np.ndarray:
    """与Step11 apply_interval_hazard_calibrator()一致。"""
    probability = np.clip(
        np.asarray(hazard, dtype=np.float64),
        EPS,
        1.0 - EPS,
    )
    raw_logit = (
        np.log(probability)
        - np.log1p(-probability)
    )
    alpha64 = np.asarray(alpha, dtype=np.float64)
    calibrated = expit(
        alpha64[None, :]
        + float(beta) * raw_logit
    )
    return np.clip(
        calibrated,
        EPS,
        1.0 - EPS,
    ).astype(np.float32)


def np_step11_hazard_to_risk(hazard: np.ndarray) -> np.ndarray:
    """与Step11 hazard_to_risk()一致。"""
    hazard64 = np.clip(
        np.asarray(hazard, dtype=np.float64),
        EPS,
        1.0 - EPS,
    )
    survival = np.cumprod(
        1.0 - hazard64,
        axis=1,
    )
    return np.clip(
        1.0 - survival,
        0.0,
        1.0,
    ).astype(np.float32)




# =============================================================================
# 10. Step6结局与Step11基准预测
# =============================================================================

@dataclass
class EvaluationData:
    local_patient_idx_long: np.ndarray
    fold_id_long: np.ndarray
    landmark_index_long: np.ndarray
    landmark_month_long: np.ndarray
    analysis_time_month: np.ndarray
    event_within_60m: np.ndarray
    calibrated_risk_saved: np.ndarray


def load_evaluation_data(common: CommonData) -> EvaluationData:
    required = [
        STEP6_DIR / "development_local_patient_idx_long.npy",
        STEP6_DIR / "development_fold_id_long.npy",
        STEP6_DIR / "development_landmark_index_long.npy",
        STEP6_DIR / "development_landmark_month_long.npy",
        STEP6_DIR / "development_analysis_time_month.npy",
        STEP6_DIR / "development_event_within_60m.npy",
        CALIBRATION_FILE,
        CALIBRATED_RISK_FILE,
    ]
    require_files(required)

    local_patient_idx_long = np.load(
        STEP6_DIR / "development_local_patient_idx_long.npy"
    ).astype(np.int32)
    fold_id_long = np.load(
        STEP6_DIR / "development_fold_id_long.npy"
    ).astype(np.int8)
    landmark_index_long = np.load(
        STEP6_DIR / "development_landmark_index_long.npy"
    ).astype(np.int8)
    landmark_month_long = np.load(
        STEP6_DIR / "development_landmark_month_long.npy"
    ).astype(np.float64)
    analysis_time_month = np.load(
        STEP6_DIR / "development_analysis_time_month.npy"
    ).astype(np.float64)
    event_within_60m = np.load(
        STEP6_DIR / "development_event_within_60m.npy"
    ).astype(bool)
    calibrated_risk_saved = np.load(
        CALIBRATED_RISK_FILE,
        mmap_mode="r",
    )

    expected_long_n = EXPECTED_DEVELOPMENT_VALID_ORIGIN_N
    expected_shapes = {
        "local_patient_idx_long": (expected_long_n,),
        "fold_id_long": (expected_long_n,),
        "landmark_index_long": (expected_long_n,),
        "landmark_month_long": (expected_long_n,),
        "analysis_time_month": (expected_long_n,),
        "event_within_60m": (expected_long_n,),
        "calibrated_risk_saved": (
            expected_long_n,
            EXPECTED_FUTURE_INTERVAL_N,
        ),
    }
    actual = {
        "local_patient_idx_long": local_patient_idx_long.shape,
        "fold_id_long": fold_id_long.shape,
        "landmark_index_long": landmark_index_long.shape,
        "landmark_month_long": landmark_month_long.shape,
        "analysis_time_month": analysis_time_month.shape,
        "event_within_60m": event_within_60m.shape,
        "calibrated_risk_saved": calibrated_risk_saved.shape,
    }
    for name, expected in expected_shapes.items():
        if actual[name] != expected:
            raise ValueError(
                f"{name}形状={actual[name]}，预期={expected}。"
            )

    valid_map = common.long_row_index_map >= 0
    landmark_grid = np.broadcast_to(
        np.arange(EXPECTED_LANDMARK_N, dtype=np.int32)[None, :],
        common.long_row_index_map.shape,
    )
    mapped_rows = common.long_row_index_map[valid_map].astype(np.int64)
    if not np.array_equal(
        landmark_index_long[mapped_rows],
        landmark_grid[valid_map],
    ):
        raise ValueError(
            "Step6 landmark_index_long与development_long_row_index_map不一致。"
        )
    if not np.array_equal(
        landmark_month_long.astype(np.int32),
        LANDMARK_MONTHS[landmark_index_long],
    ):
        raise ValueError(
            "Step6 landmark月份与索引不一致。"
        )
    if not np.array_equal(
        local_patient_idx_long[mapped_rows],
        np.broadcast_to(
            np.arange(EXPECTED_DEVELOPMENT_N, dtype=np.int32)[:, None],
            common.long_row_index_map.shape,
        )[valid_map],
    ):
        raise ValueError(
            "Step6 local_patient_idx_long与development_long_row_index_map不一致。"
        )
    if not np.array_equal(
        fold_id_long,
        common.development_fold_id[local_patient_idx_long],
    ):
        raise ValueError(
            "Step6 fold_id_long与开发集固定五折不一致。"
        )

    if not np.isfinite(analysis_time_month).all():
        raise ValueError(
            "Step6 Landmark后随访时间存在NaN或无穷值。"
        )
    if np.any(analysis_time_month <= 0):
        raise ValueError(
            "Step6 Landmark后随访时间必须>0。"
        )

    return EvaluationData(
        local_patient_idx_long=local_patient_idx_long,
        fold_id_long=fold_id_long,
        landmark_index_long=landmark_index_long,
        landmark_month_long=landmark_month_long,
        analysis_time_month=analysis_time_month,
        event_within_60m=event_within_60m,
        calibrated_risk_saved=calibrated_risk_saved,
    )


# =============================================================================
# 11. 冻结模型ensemble读取
# =============================================================================

@dataclass
class SeedCheckpointPayload:
    seed: int
    parameters: dict[str, Any]
    snapshot_states: list[dict[str, torch.Tensor]]


@dataclass
class FoldCheckpointBundle:
    fold_id: int
    parameters: dict[str, Any]
    dynamic_names: list[str]
    static_names: list[str]
    seeds: list[SeedCheckpointPayload]


def load_fold_checkpoint_bundle(
    fold_id: int,
    fold_arrays: FoldArrays,
) -> FoldCheckpointBundle:
    fold_dir = STEP10E_DIR / f"fold_{fold_id}"
    checkpoint_paths = sorted(
        fold_dir.glob("seed_*_snapshot_ensemble.pt")
    )
    if not checkpoint_paths:
        raise FileNotFoundError(
            f"Fold {fold_id}没有找到seed snapshot ensemble检查点。"
        )

    reference_parameters = None
    reference_dynamic_names = None
    reference_static_names = None
    seed_payloads = []

    for checkpoint_path in checkpoint_paths:
        checkpoint = torch_load_full(
            checkpoint_path,
            map_location="cpu",
        )
        if int(checkpoint["fold_id"]) != int(fold_id):
            raise ValueError(
                f"Fold {fold_id}检查点fold_id错误：{checkpoint_path}"
            )

        parameters = dict(checkpoint["parameters"])
        dynamic_names = list(
            checkpoint["feature_names"]["enhanced_dynamic"]
        )
        static_names = list(
            checkpoint["feature_names"]["static"]
        )
        snapshot_states = list(
            checkpoint["snapshot_state_dicts"]
        )

        if not snapshot_states:
            raise ValueError(
                f"Fold {fold_id}检查点没有snapshot state：{checkpoint_path}"
            )

        if reference_parameters is None:
            reference_parameters = parameters
            reference_dynamic_names = dynamic_names
            reference_static_names = static_names
        else:
            if parameters != reference_parameters:
                raise ValueError(
                    f"Fold {fold_id}不同seed模型参数不一致。"
                )
            if dynamic_names != reference_dynamic_names:
                raise ValueError(
                    f"Fold {fold_id}不同seed动态特征顺序不一致。"
                )
            if static_names != reference_static_names:
                raise ValueError(
                    f"Fold {fold_id}不同seed静态特征顺序不一致。"
                )

        seed_payloads.append(
            SeedCheckpointPayload(
                seed=int(checkpoint["seed"]),
                parameters=parameters,
                snapshot_states=snapshot_states,
            )
        )

    if reference_dynamic_names != fold_arrays.enhanced_dynamic_names:
        raise ValueError(
            f"Fold {fold_id}检查点动态特征与当前重建数据不一致。"
        )
    if reference_static_names != fold_arrays.static_names:
        raise ValueError(
            f"Fold {fold_id}检查点静态特征与当前重建数据不一致。"
        )

    if len(reference_dynamic_names) != EXPECTED_ENHANCED_DYNAMIC_N:
        raise ValueError(
            f"Fold {fold_id}动态特征数不是85。"
        )
    if len(reference_static_names) != EXPECTED_STATIC_N:
        raise ValueError(
            f"Fold {fold_id}静态特征数不是16。"
        )

    return FoldCheckpointBundle(
        fold_id=int(fold_id),
        parameters=reference_parameters,
        dynamic_names=reference_dynamic_names,
        static_names=reference_static_names,
        seeds=seed_payloads,
    )

# =============================================================================
# 12. 预先定义的临床域与严格分区审计
# =============================================================================

@dataclass(frozen=True)
class GroupSpec:
    group_key: str
    group_label: str
    dynamic_clinical_features: tuple[str, ...] = tuple()
    static_exact_features: tuple[str, ...] = tuple()
    static_prefixes: tuple[str, ...] = tuple()
    include_age: bool = False


@dataclass(frozen=True)
class ResolvedGroupSpec:
    group_key: str
    group_label: str
    dynamic_indices: tuple[int, ...]
    dynamic_names: tuple[str, ...]
    static_indices: tuple[int, ...]
    static_names: tuple[str, ...]
    include_age: bool
    clinical_features: tuple[str, ...]


GROUP_SPECS: tuple[GroupSpec, ...] = (
    GroupSpec(
        group_key="renal_function",
        group_label="Renal function",
        dynamic_clinical_features=(
            "Urea",
            "eGFR",
        ),
    ),
    GroupSpec(
        group_key="hiv_disease",
        group_label="HIV disease",
        dynamic_clinical_features=(
            "HIVRNA_log10",
            "CD4",
            "CD8",
        ),
        static_exact_features=(
            "Oppinfection",
        ),
        static_prefixes=(
            "WHOstage_",
        ),
    ),
    GroupSpec(
        group_key="hematologic",
        group_label="Hematologic",
        dynamic_clinical_features=(
            "WBC",
            "PLT",
            "HB",
        ),
    ),
    GroupSpec(
        group_key="metabolic_biochemistry",
        group_label="Metabolic biochemistry",
        dynamic_clinical_features=(
            "TC",
            "TG",
            "HDL",
            "LDL",
            "GLU",
        ),
    ),
    GroupSpec(
        group_key="liver_viral_coinfection",
        group_label="Liver / viral coinfection",
        dynamic_clinical_features=(
            "ALT",
            "AST",
            "HBV_status",
            "HCV_status",
        ),
    ),
    GroupSpec(
        group_key="cardiometabolic_comorbidities",
        group_label="Cardiometabolic comorbidities",
        dynamic_clinical_features=(
            "CVD_status",
            "diabetes_status",
            "hypertension_status",
            "hypercholesterolemia_status",
        ),
    ),
    GroupSpec(
        group_key="metabolic_medications",
        group_label="Metabolic medications",
        dynamic_clinical_features=tuple(
            METABOLIC_MED_FEATURES
        ),
    ),
    GroupSpec(
        group_key="current_art_regimen",
        group_label="Current ART regimen",
        dynamic_clinical_features=tuple(
            CURRENT_ART_FEATURES
        ),
    ),
    GroupSpec(
        group_key="cumulative_art_exposure",
        group_label="Cumulative ART exposure",
        dynamic_clinical_features=tuple(
            CUMULATIVE_ART_FEATURES
        ),
    ),
    GroupSpec(
        group_key="demographic_anthropometric",
        group_label="Demographic / anthropometric",
        static_exact_features=(
            "BMI",
        ),
        static_prefixes=(
            "Sex_",
            "Marriage_",
            "Course_",
        ),
        include_age=True,
    ),
)


def _expanded_dynamic_names_for_clinical_feature(
    feature_name: str,
) -> tuple[str, ...]:
    """
    实验室变量必须同时遮挡4种表示；
    其他动态变量只有基础动态通道。
    """
    feature_name = str(
        feature_name
    )

    if feature_name in LAB_FEATURES:
        return (
            feature_name,
            f"{feature_name}_observed",
            f"{feature_name}_time_since_last",
            f"{feature_name}_delta_last_observed",
        )

    if feature_name in DYNAMIC_FEATURES:
        return (
            feature_name,
        )

    raise ValueError(
        f"未识别的动态临床变量：{feature_name}"
    )


def resolve_group_specs(
    fold_arrays: FoldArrays,
) -> tuple[
    ResolvedGroupSpec,
    ...,
]:
    dynamic_names = list(
        fold_arrays.enhanced_dynamic_names
    )
    static_names = list(
        fold_arrays.static_names
    )

    dynamic_lookup = {
        name: index
        for index, name in enumerate(
            dynamic_names
        )
    }
    static_lookup = {
        name: index
        for index, name in enumerate(
            static_names
        )
    }

    resolved: list[
        ResolvedGroupSpec
    ] = []

    dynamic_owner: dict[
        int,
        str,
    ] = {}
    static_owner: dict[
        int,
        str,
    ] = {}
    age_owner: list[str] = []

    for spec in GROUP_SPECS:
        selected_dynamic_names: list[
            str
        ] = []

        for clinical_name in (
            spec.dynamic_clinical_features
        ):
            expanded = (
                _expanded_dynamic_names_for_clinical_feature(
                    clinical_name
                )
            )
            selected_dynamic_names.extend(
                expanded
            )

        missing_dynamic = [
            name
            for name in selected_dynamic_names
            if name not in dynamic_lookup
        ]
        if missing_dynamic:
            raise ValueError(
                f"组{spec.group_key}缺少动态通道："
                f"{missing_dynamic}"
            )

        selected_dynamic_indices = tuple(
            dynamic_lookup[
                name
            ]
            for name in selected_dynamic_names
        )

        selected_static_names: list[
            str
        ] = []

        for exact_name in (
            spec.static_exact_features
        ):
            if exact_name not in static_lookup:
                raise ValueError(
                    f"组{spec.group_key}缺少静态特征："
                    f"{exact_name}"
                )
            selected_static_names.append(
                exact_name
            )

        for prefix_name in (
            spec.static_prefixes
        ):
            matched = [
                name
                for name in static_names
                if name.startswith(
                    prefix_name
                )
            ]
            if not matched:
                raise ValueError(
                    f"组{spec.group_key}静态前缀"
                    f"{prefix_name}没有匹配特征。"
                )
            selected_static_names.extend(
                matched
            )

        # 保持模型实际静态输入顺序，并去重。
        selected_static_names = [
            name
            for name in static_names
            if name in set(
                selected_static_names
            )
        ]

        selected_static_indices = tuple(
            static_lookup[
                name
            ]
            for name in selected_static_names
        )

        for index in selected_dynamic_indices:
            if index in dynamic_owner:
                raise ValueError(
                    "动态通道被多个group重复覆盖："
                    f"{dynamic_names[index]} -> "
                    f"{dynamic_owner[index]} / "
                    f"{spec.group_key}"
                )
            dynamic_owner[
                index
            ] = spec.group_key

        for index in selected_static_indices:
            if index in static_owner:
                raise ValueError(
                    "静态通道被多个group重复覆盖："
                    f"{static_names[index]} -> "
                    f"{static_owner[index]} / "
                    f"{spec.group_key}"
                )
            static_owner[
                index
            ] = spec.group_key

        if spec.include_age:
            age_owner.append(
                spec.group_key
            )

        clinical_features = tuple(
            [
                *spec.dynamic_clinical_features,
                *spec.static_exact_features,
                *spec.static_prefixes,
                *(
                    ("Age",)
                    if spec.include_age
                    else tuple()
                ),
            ]
        )

        resolved.append(
            ResolvedGroupSpec(
                group_key=spec.group_key,
                group_label=spec.group_label,
                dynamic_indices=tuple(
                    int(x)
                    for x in selected_dynamic_indices
                ),
                dynamic_names=tuple(
                    selected_dynamic_names
                ),
                static_indices=tuple(
                    int(x)
                    for x in selected_static_indices
                ),
                static_names=tuple(
                    selected_static_names
                ),
                include_age=bool(
                    spec.include_age
                ),
                clinical_features=(
                    clinical_features
                ),
            )
        )

    expected_dynamic = set(
        range(
            EXPECTED_ENHANCED_DYNAMIC_N
        )
    )
    observed_dynamic = set(
        dynamic_owner.keys()
    )

    if observed_dynamic != expected_dynamic:
        missing_indices = sorted(
            expected_dynamic
            - observed_dynamic
        )
        extra_indices = sorted(
            observed_dynamic
            - expected_dynamic
        )
        raise ValueError(
            "Group定义未完整且互斥覆盖85维动态输入。"
            f" missing="
            f"{[dynamic_names[i] for i in missing_indices]},"
            f" extra={extra_indices}"
        )

    expected_static = set(
        range(
            EXPECTED_STATIC_N
        )
    )
    observed_static = set(
        static_owner.keys()
    )

    if observed_static != expected_static:
        missing_indices = sorted(
            expected_static
            - observed_static
        )
        extra_indices = sorted(
            observed_static
            - expected_static
        )
        raise ValueError(
            "Group定义未完整且互斥覆盖16维静态输入。"
            f" missing="
            f"{[static_names[i] for i in missing_indices]},"
            f" extra={extra_indices}"
        )

    if age_owner != [
        "demographic_anthropometric"
    ]:
        raise ValueError(
            "Age必须且只能属于"
            "demographic_anthropometric组。"
        )

    if len(
        resolved
    ) != 10:
        raise ValueError(
            "预设临床域数不是10。"
        )

    return tuple(
        resolved
    )


def group_mapping_frame(
    resolved_groups: tuple[
        ResolvedGroupSpec,
        ...,
    ],
) -> pd.DataFrame:
    rows: list[
        dict[str, Any]
    ] = []

    for group in resolved_groups:
        for index, name in zip(
            group.dynamic_indices,
            group.dynamic_names,
        ):
            rows.append(
                {
                    "group_key": (
                        group.group_key
                    ),
                    "group_label": (
                        group.group_label
                    ),
                    "input_branch": (
                        "dynamic"
                    ),
                    "channel_index": int(
                        index
                    ),
                    "channel_name": (
                        name
                    ),
                }
            )

        for index, name in zip(
            group.static_indices,
            group.static_names,
        ):
            rows.append(
                {
                    "group_key": (
                        group.group_key
                    ),
                    "group_label": (
                        group.group_label
                    ),
                    "input_branch": (
                        "static"
                    ),
                    "channel_index": int(
                        index
                    ),
                    "channel_name": (
                        name
                    ),
                }
            )

        if group.include_age:
            rows.append(
                {
                    "group_key": (
                        group.group_key
                    ),
                    "group_label": (
                        group.group_label
                    ),
                    "input_branch": (
                        "age"
                    ),
                    "channel_index": (
                        0
                    ),
                    "channel_name": (
                        "Age"
                    ),
                }
            )

    return pd.DataFrame(
        rows
    )


# =============================================================================
# 13. Grouped reference occlusion前向
# =============================================================================

def apply_group_reference_occlusion(
    inputs: dict[str, np.ndarray],
    fold_arrays: FoldArrays,
    landmark_index: int,
    group: ResolvedGroupSpec | None,
) -> dict[str, np.ndarray]:
    """
    group=None表示FP32 baseline，不做任何改变。

    动态组：
      在所有真实存在且位于Landmark之前/之内的历史时间点，
      将该组动态通道替换为同fold训练风险集reference。

    静态组：
      将该组静态通道替换为Landmark-specific训练风险集reference。

    Age：
      若该组包含Age，则替换为Landmark-specific训练风险集平均标准化年龄。

    row_mask/landmark/history length始终保持不变。
    """
    if group is None:
        return inputs

    result = dict(
        inputs
    )

    landmark_index = int(
        landmark_index
    )

    if group.dynamic_indices:
        dynamic = np.asarray(
            inputs[
                "dynamic"
            ],
            dtype=np.float32,
        ).copy()

        row_mask = np.asarray(
            inputs[
                "row_mask"
            ],
            dtype=bool,
        )

        landmark_bin = np.asarray(
            inputs[
                "landmark_bin"
            ],
            dtype=np.int64,
        )

        reference = (
            fold_arrays
            .dynamic_reference_by_landmark[
                landmark_index
            ]
        )

        dynamic_indices = np.asarray(
            group.dynamic_indices,
            dtype=np.int64,
        )

        max_step = int(
            LANDMARK_BINS[
                landmark_index
            ]
        )

        for step_index in range(
            max_step + 1
        ):
            active = (
                row_mask[
                    :,
                    step_index,
                ]
                & (
                    step_index
                    <= landmark_bin
                )
            )

            active_indices = np.where(
                active
            )[0].astype(
                np.int64
            )

            if len(
                active_indices
            ) == 0:
                continue

            dynamic[
                np.ix_(
                    active_indices,
                    np.asarray(
                        [
                            step_index
                        ],
                        dtype=np.int64,
                    ),
                    dynamic_indices,
                )
            ] = (
                reference[
                    step_index,
                    dynamic_indices,
                ][
                    None,
                    None,
                    :
                ]
            )

        result[
            "dynamic"
        ] = dynamic

    if group.static_indices:
        static = np.asarray(
            inputs[
                "static"
            ],
            dtype=np.float32,
        ).copy()

        static_indices = np.asarray(
            group.static_indices,
            dtype=np.int64,
        )

        static_reference = (
            fold_arrays
            .static_reference_by_landmark[
                landmark_index
            ]
        )

        static[
            :,
            static_indices,
        ] = static_reference[
            static_indices
        ][
            None,
            :
        ]

        result[
            "static"
        ] = static

    if group.include_age:
        age = np.asarray(
            inputs[
                "age"
            ],
            dtype=np.float32,
        ).copy()

        age_reference = float(
            fold_arrays
            .age_reference_by_landmark[
                landmark_index
            ]
        )

        age[
            ...
        ] = age_reference

        result[
            "age"
        ] = age

    return result


def predict_one_snapshot_fp32(
    model: HybridAttentionLSTMSurvival,
    common: CommonData,
    fold_arrays: FoldArrays,
    sample_pairs: np.ndarray,
    landmark_index: int,
    group: ResolvedGroupSpec | None,
    batch_size: int,
    device: torch.device,
) -> np.ndarray:
    model.eval()
    hazards = []

    with torch.no_grad():
        for start in range(
            0,
            len(
                sample_pairs
            ),
            int(
                batch_size
            ),
        ):
            end = min(
                start
                + int(
                    batch_size
                ),
                len(
                    sample_pairs
                ),
            )

            batch_pairs = (
                sample_pairs[
                    start:end
                ]
            )

            inputs = (
                build_model_inputs_for_sample_pairs(
                    common=common,
                    fold_arrays=fold_arrays,
                    sample_pairs=batch_pairs,
                )
            )

            inputs = (
                apply_group_reference_occlusion(
                    inputs=inputs,
                    fold_arrays=fold_arrays,
                    landmark_index=(
                        landmark_index
                    ),
                    group=group,
                )
            )

            dynamic = torch.as_tensor(
                inputs[
                    "dynamic"
                ],
                dtype=torch.float32,
                device=device,
            )

            row_mask = torch.as_tensor(
                inputs[
                    "row_mask"
                ],
                dtype=torch.bool,
                device=device,
            )

            static = torch.as_tensor(
                inputs[
                    "static"
                ],
                dtype=torch.float32,
                device=device,
            )

            age = torch.as_tensor(
                inputs[
                    "age"
                ],
                dtype=torch.float32,
                device=device,
            )

            landmark_norm = torch.as_tensor(
                inputs[
                    "landmark_norm"
                ],
                dtype=torch.float32,
                device=device,
            )

            landmark_bin = torch.as_tensor(
                inputs[
                    "landmark_bin"
                ],
                dtype=torch.long,
                device=device,
            )

            logits = model(
                dynamic_sequence=dynamic,
                row_mask=row_mask,
                static_baseline=static,
                age_at_landmark=age,
                landmark_normalized=(
                    landmark_norm
                ),
                landmark_bin=(
                    landmark_bin
                ),
            )

            hazard = (
                torch.sigmoid(
                    logits.float()
                )
                .cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            hazards.append(
                hazard
            )

            del (
                dynamic,
                row_mask,
                static,
                age,
                landmark_norm,
                landmark_bin,
                logits,
            )

    return np.concatenate(
        hazards,
        axis=0,
    ).astype(
        np.float32
    )


def predict_fold_group_fp32(
    bundle: FoldCheckpointBundle,
    common: CommonData,
    fold_arrays: FoldArrays,
    sample_pairs: np.ndarray,
    landmark_index: int,
    group: ResolvedGroupSpec | None,
    calibration_alpha: np.ndarray,
    calibration_beta: float,
    device: torch.device,
) -> np.ndarray:
    """
    snapshot内和seed间均使用NumPy float32 mean，
    与Step10E正式ensemble规则保持一致。

    baseline与所有group occlusion均使用相同FP32前向路径。
    返回未来10个半年区间的累计CKD风险。
    """
    if len(
        sample_pairs
    ) == 0:
        raise ValueError(
            "sample_pairs为空。"
        )

    if not np.all(
        sample_pairs[
            :,
            1,
        ]
        == int(
            landmark_index
        )
    ):
        raise ValueError(
            "一个predict_fold_group_fp32"
            "调用只能包含一个Landmark。"
        )

    formal_batch_size = int(
        bundle.parameters[
            "batch_size"
        ]
    )

    batch_size = (
        INFERENCE_BATCH_SIZE_OVERRIDE
        if INFERENCE_BATCH_SIZE_OVERRIDE
        > 0
        else formal_batch_size
    )

    seed_hazards = []

    for seed_payload in (
        bundle.seeds
    ):
        if (
            seed_payload.parameters
            != bundle.parameters
        ):
            raise ValueError(
                "seed参数与bundle参数不一致。"
            )

        model = build_model(
            bundle.parameters,
            device,
        )

        snapshot_hazards = []

        for state in (
            seed_payload.snapshot_states
        ):
            model.load_state_dict(
                state,
                strict=True,
            )

            model.eval()

            snapshot_hazard = (
                predict_one_snapshot_fp32(
                    model=model,
                    common=common,
                    fold_arrays=(
                        fold_arrays
                    ),
                    sample_pairs=(
                        sample_pairs
                    ),
                    landmark_index=(
                        landmark_index
                    ),
                    group=group,
                    batch_size=(
                        batch_size
                    ),
                    device=device,
                )
            )

            snapshot_hazards.append(
                snapshot_hazard
            )

        seed_hazard = np.mean(
            np.stack(
                snapshot_hazards,
                axis=0,
            ),
            axis=0,
        ).astype(
            np.float32
        )

        seed_hazards.append(
            seed_hazard
        )

        del model
        gc.collect()
        torch.cuda.empty_cache()

    ensemble_hazard = np.mean(
        np.stack(
            seed_hazards,
            axis=0,
        ),
        axis=0,
    ).astype(
        np.float32
    )

    survival_saved_like = (
        np_step10e_hazard_to_survival(
            ensemble_hazard
        )
    )

    step11_raw_hazard = (
        np_step11_survival_to_hazard(
            survival_saved_like
        )
    )

    calibrated_hazard = (
        np_step11_apply_calibrator(
            hazard=step11_raw_hazard,
            alpha=calibration_alpha,
            beta=calibration_beta,
        )
    )

    calibrated_risk = (
        np_step11_hazard_to_risk(
            calibrated_hazard
        )
    )

    if calibrated_risk.shape != (
        len(
            sample_pairs
        ),
        EXPECTED_FUTURE_INTERVAL_N,
    ):
        raise ValueError(
            "Group occlusion累计风险形状错误。"
        )

    if not np.isfinite(
        calibrated_risk
    ).all():
        raise ValueError(
            "Group occlusion累计风险出现NaN或无穷值。"
        )

    if np.any(
        (
            calibrated_risk
            < -EPS
        )
        | (
            calibrated_risk
            > 1.0
            + EPS
        )
    ):
        raise ValueError(
            "Group occlusion累计风险超出0～1。"
        )

    if np.any(
        np.diff(
            calibrated_risk,
            axis=1,
        )
        < -1e-7
    ):
        raise ValueError(
            "Group occlusion累计风险不单调。"
        )

    return calibrated_risk.astype(
        np.float32
    )


# =============================================================================
# 14. checkpoint / resume
# =============================================================================

def initialize_checkpoint_metadata() -> None:
    metadata_path = (
        CHECKPOINT_DIR
        / "checkpoint_metadata.json"
    )

    expected = {
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "primary_landmark_months": (
            PRIMARY_LANDMARK_MONTHS.tolist()
        ),
        "group_keys": [
            spec.group_key
            for spec in GROUP_SPECS
        ],
        "inference_precision": (
            "FP32"
        ),
        "dynamic_reference_scope": (
            "heldout-fold-excluded training risk set; "
            "landmark-specific and history-step-specific"
        ),
        "static_age_reference_scope": (
            "heldout-fold-excluded training risk set; "
            "landmark-specific"
        ),
        "lab_representation_rule": (
            "value + observed + time_since_last + "
            "delta_last_observed occluded together"
        ),
        "row_mask_held_fixed": True,
        "landmark_context_held_fixed": True,
        "calibrator_refit_after_occlusion": False,
    }

    if metadata_path.exists():
        observed = json.loads(
            metadata_path.read_text(
                encoding="utf-8"
            )
        )

        if observed != expected:
            raise ValueError(
                "已有Step12C checkpoint metadata"
                "与当前代码不一致。"
                "请不要混用不同版本的group occlusion预测。"
            )
        return

    existing_predictions = list(
        CHECKPOINT_DIR.rglob(
            "*_calibrated_risk.npy"
        )
    )

    if existing_predictions:
        raise ValueError(
            "检测到已有Step12C预测checkpoint"
            "但缺少版本metadata；"
            "为避免混用旧结果，请先核对/移动旧输出目录。"
        )

    save_json(
        expected,
        metadata_path,
    )


def landmark_checkpoint_dir(
    fold_id: int,
    landmark_month: int,
) -> Path:
    path = (
        CHECKPOINT_DIR
        / f"fold_{int(fold_id)}"
        / f"landmark_{int(landmark_month)}m"
    )

    path.mkdir(
        parents=True,
        exist_ok=True,
    )

    return path


def group_prediction_path(
    fold_id: int,
    landmark_month: int,
    group_key: str | None,
) -> Path:
    filename = (
        "baseline_calibrated_risk.npy"
        if group_key is None
        else (
            f"group_{group_key}"
            "_calibrated_risk.npy"
        )
    )

    return (
        landmark_checkpoint_dir(
            fold_id,
            landmark_month,
        )
        / filename
    )


def get_or_predict_group(
    bundle: FoldCheckpointBundle,
    common: CommonData,
    fold_arrays: FoldArrays,
    sample_pairs: np.ndarray,
    patient_local: np.ndarray,
    landmark_index: int,
    group: ResolvedGroupSpec | None,
    calibration_alpha: np.ndarray,
    calibration_beta: float,
    device: torch.device,
) -> np.ndarray:
    landmark_month = int(
        LANDMARK_MONTHS[
            int(
                landmark_index
            )
        ]
    )

    fold_id = int(
        bundle.fold_id
    )

    checkpoint_dir = (
        landmark_checkpoint_dir(
            fold_id,
            landmark_month,
        )
    )

    patient_path = (
        checkpoint_dir
        / "patient_local.npy"
    )

    prediction_path = (
        group_prediction_path(
            fold_id=fold_id,
            landmark_month=(
                landmark_month
            ),
            group_key=(
                None
                if group is None
                else group.group_key
            ),
        )
    )

    if (
        RESUME
        and prediction_path.exists()
    ):
        if not patient_path.exists():
            raise FileNotFoundError(
                "已有预测但缺少患者顺序文件："
                f"{patient_path}"
            )

        saved_patient = np.load(
            patient_path
        ).astype(
            np.int32
        )

        if not np.array_equal(
            saved_patient,
            patient_local.astype(
                np.int32
            ),
        ):
            raise ValueError(
                f"Fold {fold_id}, "
                f"Landmark {landmark_month}月："
                "已有checkpoint患者顺序"
                "与当前不一致。"
            )

        loaded = np.load(
            prediction_path
        ).astype(
            np.float32
        )

        if loaded.shape != (
            len(
                patient_local
            ),
            EXPECTED_FUTURE_INTERVAL_N,
        ):
            raise ValueError(
                "已有预测形状错误："
                f"{prediction_path}"
            )

        return loaded

    risk = (
        predict_fold_group_fp32(
            bundle=bundle,
            common=common,
            fold_arrays=(
                fold_arrays
            ),
            sample_pairs=(
                sample_pairs
            ),
            landmark_index=(
                landmark_index
            ),
            group=group,
            calibration_alpha=(
                calibration_alpha
            ),
            calibration_beta=(
                calibration_beta
            ),
            device=device,
        )
    )

    if not patient_path.exists():
        np.save(
            patient_path,
            patient_local.astype(
                np.int32
            ),
        )

    np.save(
        prediction_path,
        risk,
    )

    return risk


# =============================================================================
# 15. 生存评价
# =============================================================================

def build_survival_array(
    event: np.ndarray,
    time_month: np.ndarray,
) -> np.ndarray:
    event = np.asarray(
        event,
        dtype=bool,
    )
    time_month = np.asarray(
        time_month,
        dtype=np.float64,
    )
    if event.shape != time_month.shape:
        raise ValueError(
            "事件和随访时间形状不一致。"
        )
    if not np.isfinite(
        time_month
    ).all():
        raise ValueError(
            "随访时间存在NaN或无穷值。"
        )
    if np.any(
        time_month <= 0
    ):
        raise ValueError(
            "Landmark后随访时间必须>0。"
        )
    return Surv.from_arrays(
        event=event,
        time=time_month,
    )

def evaluate_survival_predictions(
    survival_reference: np.ndarray,
    risk_matrix: np.ndarray,
) -> dict[str, Any]:
    risk_matrix = np.asarray(
        risk_matrix,
        dtype=np.float64,
    )
    if risk_matrix.ndim != 2:
        raise ValueError(
            "risk_matrix必须为二维。"
        )
    if risk_matrix.shape[1] != EXPECTED_FUTURE_INTERVAL_N:
        raise ValueError(
            "risk_matrix未来区间数不是10。"
        )

    survival_matrix = 1.0 - risk_matrix

    c_index = float(
        concordance_index_ipcw(
            survival_reference,
            survival_reference,
            risk_matrix[:, -1],
            tau=float(METRIC_TIMES[-1]),
        )[0]
    )

    dynamic_auc, mean_auc = (
        cumulative_dynamic_auc(
            survival_reference,
            survival_reference,
            risk_matrix,
            METRIC_TIMES,
        )
    )

    _, brier_values = brier_score(
        survival_reference,
        survival_reference,
        survival_matrix,
        METRIC_TIMES,
    )

    ibs = float(
        integrated_brier_score(
            survival_reference,
            survival_reference,
            survival_matrix,
            METRIC_TIMES,
        )
    )

    return {
        "uno_c_index_5y": c_index,
        "integrated_dynamic_auc": float(
            mean_auc
        ),
        "integrated_brier": ibs,
        "dynamic_auc": np.asarray(
            dynamic_auc,
            dtype=np.float64,
        ),
        "brier": np.asarray(
            brier_values,
            dtype=np.float64,
        ),
    }


# =============================================================================
# 16. 结果表
# =============================================================================

def group_metric_row(
    group: ResolvedGroupSpec,
    landmark_index: int,
    landmark_month: int,
    risk_set_n: int,
    event_n: int,
    baseline_metrics: dict[str, Any],
    occluded_metrics: dict[str, Any],
    baseline_saved_metrics: dict[str, Any],
    patient_delta_risk5: np.ndarray,
) -> dict[str, Any]:
    baseline_c = float(
        baseline_metrics[
            "uno_c_index_5y"
        ]
    )
    baseline_iauc = float(
        baseline_metrics[
            "integrated_dynamic_auc"
        ]
    )
    baseline_ibs = float(
        baseline_metrics[
            "integrated_brier"
        ]
    )

    occ_c = float(
        occluded_metrics[
            "uno_c_index_5y"
        ]
    )
    occ_iauc = float(
        occluded_metrics[
            "integrated_dynamic_auc"
        ]
    )
    occ_ibs = float(
        occluded_metrics[
            "integrated_brier"
        ]
    )

    delta_risk5 = np.asarray(
        patient_delta_risk5,
        dtype=np.float64,
    )

    return {
        "group_key": (
            group.group_key
        ),
        "group_label": (
            group.group_label
        ),
        "landmark_index": int(
            landmark_index
        ),
        "landmark_month": int(
            landmark_month
        ),
        "landmark_year": float(
            landmark_month
            / 12.0
        ),
        "risk_set_n": int(
            risk_set_n
        ),
        "future_5y_event_n": int(
            event_n
        ),
        "occluded_dynamic_channel_n": int(
            len(
                group.dynamic_indices
            )
        ),
        "occluded_static_channel_n": int(
            len(
                group.static_indices
            )
        ),
        "age_occluded": bool(
            group.include_age
        ),
        "total_occluded_input_channel_n": int(
            len(
                group.dynamic_indices
            )
            + len(
                group.static_indices
            )
            + int(
                group.include_age
            )
        ),
        "clinical_features": "|".join(
            group.clinical_features
        ),
        "saved_step11_baseline_uno_c": float(
            baseline_saved_metrics[
                "uno_c_index_5y"
            ]
        ),
        "saved_step11_baseline_iAUC": float(
            baseline_saved_metrics[
                "integrated_dynamic_auc"
            ]
        ),
        "saved_step11_baseline_IBS": float(
            baseline_saved_metrics[
                "integrated_brier"
            ]
        ),
        "fp32_baseline_uno_c": (
            baseline_c
        ),
        "fp32_baseline_iAUC": (
            baseline_iauc
        ),
        "fp32_baseline_IBS": (
            baseline_ibs
        ),
        "occluded_uno_c": (
            occ_c
        ),
        "occluded_iAUC": (
            occ_iauc
        ),
        "occluded_IBS": (
            occ_ibs
        ),
        "uno_c_loss": (
            baseline_c
            - occ_c
        ),
        "iAUC_loss": (
            baseline_iauc
            - occ_iauc
        ),
        "IBS_increase": (
            occ_ibs
            - baseline_ibs
        ),
        "occluded_minus_baseline_uno_c": (
            occ_c
            - baseline_c
        ),
        "occluded_minus_baseline_iAUC": (
            occ_iauc
            - baseline_iauc
        ),
        "occluded_minus_baseline_IBS": (
            occ_ibs
            - baseline_ibs
        ),
        "mean_delta_risk5": float(
            np.mean(
                delta_risk5
            )
        ),
        "median_delta_risk5": float(
            np.median(
                delta_risk5
            )
        ),
        "mean_abs_delta_risk5": float(
            np.mean(
                np.abs(
                    delta_risk5
                )
            )
        ),
        "median_abs_delta_risk5": float(
            np.median(
                np.abs(
                    delta_risk5
                )
            )
        ),
        "p95_abs_delta_risk5": float(
            np.quantile(
                np.abs(
                    delta_risk5
                ),
                0.95,
            )
        ),
    }


def group_horizon_rows(
    group: ResolvedGroupSpec,
    landmark_index: int,
    landmark_month: int,
    baseline_metrics: dict[str, Any],
    occluded_metrics: dict[str, Any],
) -> list[
    dict[str, Any]
]:
    rows: list[
        dict[str, Any]
    ] = []

    for horizon_index, horizon_month in enumerate(
        FUTURE_END_MONTHS
    ):
        baseline_auc = float(
            baseline_metrics[
                "dynamic_auc"
            ][
                horizon_index
            ]
        )

        occluded_auc = float(
            occluded_metrics[
                "dynamic_auc"
            ][
                horizon_index
            ]
        )

        baseline_brier = float(
            baseline_metrics[
                "brier"
            ][
                horizon_index
            ]
        )

        occluded_brier = float(
            occluded_metrics[
                "brier"
            ][
                horizon_index
            ]
        )

        rows.append(
            {
                "group_key": (
                    group.group_key
                ),
                "group_label": (
                    group.group_label
                ),
                "landmark_index": int(
                    landmark_index
                ),
                "landmark_month": int(
                    landmark_month
                ),
                "landmark_year": float(
                    landmark_month
                    / 12.0
                ),
                "horizon_position": int(
                    horizon_index
                ),
                "horizon_month": float(
                    horizon_month
                ),
                "horizon_year": float(
                    horizon_month
                    / 12.0
                ),
                "baseline_dynamic_auc": (
                    baseline_auc
                ),
                "occluded_dynamic_auc": (
                    occluded_auc
                ),
                "dynamic_auc_loss": (
                    baseline_auc
                    - occluded_auc
                ),
                "baseline_brier": (
                    baseline_brier
                ),
                "occluded_brier": (
                    occluded_brier
                ),
                "brier_increase": (
                    occluded_brier
                    - baseline_brier
                ),
            }
        )

    return rows


# =============================================================================
# 17. 图形
# =============================================================================

def configure_plot_style() -> None:
    plt.rcParams.update(
        {
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "savefig.facecolor": "white",
            "text.color": "black",
            "axes.labelcolor": "black",
            "axes.edgecolor": "black",
            "axes.titlecolor": "black",
            "xtick.color": "black",
            "ytick.color": "black",
            "font.size": 10,
            "axes.titlesize": 11,
            "axes.labelsize": 10,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )

def save_figure(
    fig: plt.Figure,
    stem: str,
) -> None:
    for extension in [
        "png",
        "pdf",
        "svg",
    ]:
        kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
        }
        if extension == "png":
            kwargs["dpi"] = 600
        fig.savefig(
            FIGURE_DIR
            / f"{stem}.{extension}",
            **kwargs,
        )
    plt.close(fig)


GROUP_PLOT_ORDER = [
    spec.group_key
    for spec in GROUP_SPECS
]

GROUP_LABEL_MAP = {
    spec.group_key: spec.group_label
    for spec in GROUP_SPECS
}


def plot_group_metric(
    metrics_df: pd.DataFrame,
    y_column: str,
    y_label: str,
    stem: str,
) -> None:
    fig, ax = plt.subplots(
        figsize=(
            10.5,
            6.8,
        )
    )

    x = np.arange(
        len(
            GROUP_PLOT_ORDER
        ),
        dtype=float,
    )

    offsets = np.linspace(
        -0.24,
        0.24,
        len(
            PRIMARY_LANDMARK_INDICES
        ),
    )

    for offset, landmark_index in zip(
        offsets,
        PRIMARY_LANDMARK_INDICES,
    ):
        landmark_index = int(
            landmark_index
        )

        sub = (
            metrics_df.loc[
                metrics_df[
                    "landmark_index"
                ]
                == landmark_index
            ]
            .set_index(
                "group_key"
            )
            .reindex(
                GROUP_PLOT_ORDER
            )
        )

        if sub[
            y_column
        ].isna().any():
            raise ValueError(
                f"绘图缺少{y_column}结果："
                f"Landmark {landmark_index}"
            )

        landmark_year = float(
            LANDMARK_MONTHS[
                landmark_index
            ]
            / 12.0
        )

        label = (
            "Baseline"
            if landmark_year
            == 0.0
            else (
                f"ART year "
                f"{landmark_year:.0f}"
            )
        )

        ax.scatter(
            x
            + offset,
            sub[
                y_column
            ].to_numpy(
                dtype=float
            ),
            s=34,
            label=label,
        )

    ax.axhline(
        0.0,
        linewidth=1.0,
        linestyle="--",
    )

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        [
            GROUP_LABEL_MAP[
                key
            ]
            for key in (
                GROUP_PLOT_ORDER
            )
        ],
        rotation=35,
        ha="right",
    )

    ax.set_ylabel(
        y_label
    )

    ax.set_title(
        "Grouped clinical-domain reference occlusion"
    )

    ax.grid(
        axis="y",
        alpha=0.20,
    )

    ax.legend(
        frameon=False,
        ncol=2,
    )

    fig.tight_layout()

    save_figure(
        fig,
        stem,
    )


# =============================================================================
# 18. 主流程
# =============================================================================

def run() -> None:
    total_start = time.time()

    set_random_seed(
        RANDOM_SEED
    )

    configure_plot_style()

    device = default_device()

    require_files(
        [
            STEP10E_DIR
            / "lstm_v2_summary.json",
            CALIBRATION_FILE,
            CALIBRATED_RISK_FILE,
        ]
    )

    lstm_summary = json.loads(
        (
            STEP10E_DIR
            / "lstm_v2_summary.json"
        ).read_text(
            encoding="utf-8"
        )
    )

    if bool(
        lstm_summary.get(
            "locked_test_used",
            False,
        )
    ):
        raise ValueError(
            "Step10E摘要显示读取过锁定测试集。"
        )

    calibration_table = pd.read_csv(
        CALIBRATION_FILE,
        encoding="utf-8-sig",
    )

    common = load_common_data()

    evaluation = (
        load_evaluation_data(
            common
        )
    )

    initialize_checkpoint_metadata()

    print(
        "\n"
        + "=" * 118
    )
    print(
        "Step 12C：LSTM-v2 Grouped Clinical-Domain Reference Occlusion 五折OOF分析"
    )
    print(
        "=" * 118
    )
    print(
        "运行设备：",
        device,
    )
    print(
        "主Landmark（月）：",
        PRIMARY_LANDMARK_MONTHS.tolist(),
    )
    print(
        "Group数：",
        len(
            GROUP_SPECS
        ),
    )
    print(
        "动态遮挡：该组通道在所有有效历史时间点 -> fold训练风险集step-specific reference"
    )
    print(
        "实验室变量：value/observed/time-since/delta四类表示一起遮挡"
    )
    print(
        "静态/Age：若属于该组 -> Landmark-specific训练风险集reference"
    )
    print(
        "row_mask/history length/landmark context：保持不变"
    )
    print(
        "calibrator：原Step11 cross-fit calibrator冻结，不重新校准"
    )
    print(
        "perturbation前向：FP32；baseline与group occlusion使用完全相同数值路径"
    )
    print(
        "锁定测试集：未读取"
    )
    print(
        "输出目录：",
        OUTPUT_DIR,
    )
    print(
        "=" * 118
    )

    baseline_fp32_long = np.full(
        (
            EXPECTED_DEVELOPMENT_VALID_ORIGIN_N,
            EXPECTED_FUTURE_INTERVAL_N,
        ),
        np.nan,
        dtype=np.float32,
    )

    group_risk_long: dict[
        tuple[
            int,
            str,
        ],
        np.ndarray,
    ] = {}

    for landmark_index in (
        PRIMARY_LANDMARK_INDICES
    ):
        for spec in GROUP_SPECS:
            group_risk_long[
                (
                    int(
                        landmark_index
                    ),
                    spec.group_key,
                )
            ] = np.full(
                (
                    EXPECTED_DEVELOPMENT_VALID_ORIGIN_N,
                    EXPECTED_FUTURE_INTERVAL_N,
                ),
                np.nan,
                dtype=np.float32,
            )

    audit_rows: list[
        dict[str, Any]
    ] = []

    mapping_frames: list[
        pd.DataFrame
    ] = []

    resolved_group_reference: (
        tuple[
            ResolvedGroupSpec,
            ...,
        ]
        | None
    ) = None

    for fold_id in range(
        N_SPLITS
    ):
        progress_print(
            f"Fold {fold_id}：准备fold-specific输入和Landmark风险集reference。"
        )

        fold_arrays = (
            prepare_fold_arrays(
                common,
                fold_id,
            )
        )

        bundle = (
            load_fold_checkpoint_bundle(
                fold_id,
                fold_arrays,
            )
        )

        resolved_groups = (
            resolve_group_specs(
                fold_arrays
            )
        )

        current_mapping = (
            group_mapping_frame(
                resolved_groups
            )
        )
        current_mapping.insert(
            0,
            "fold_id",
            int(
                fold_id
            ),
        )
        mapping_frames.append(
            current_mapping
        )

        if resolved_group_reference is None:
            resolved_group_reference = (
                resolved_groups
            )

        progress_print(
            f"Fold {fold_id}：group partition审计通过 | "
            f"dynamic={EXPECTED_ENHANCED_DYNAMIC_N}, "
            f"static={EXPECTED_STATIC_N}, age=1。"
        )

        for landmark_index in (
            PRIMARY_LANDMARK_INDICES
        ):
            landmark_index = int(
                landmark_index
            )

            landmark_month = int(
                LANDMARK_MONTHS[
                    landmark_index
                ]
            )

            patient_mask = (
                (
                    common.development_fold_id
                    == int(
                        fold_id
                    )
                )
                & (
                    common.prediction_origin_mask[
                        :,
                        landmark_index,
                    ]
                )
            )

            patients = np.where(
                patient_mask
            )[0].astype(
                np.int32
            )

            if len(
                patients
            ) == 0:
                raise ValueError(
                    f"Fold {fold_id}, "
                    f"Landmark {landmark_month}月"
                    "没有OOF origins。"
                )

            sample_pairs = np.column_stack(
                [
                    patients,
                    np.full(
                        len(
                            patients
                        ),
                        landmark_index,
                        dtype=np.int32,
                    ),
                ]
            ).astype(
                np.int32
            )

            long_rows = (
                common.long_row_index_map[
                    patients,
                    landmark_index,
                ].astype(
                    np.int64
                )
            )

            if np.any(
                long_rows
                < 0
            ):
                raise ValueError(
                    f"Fold {fold_id}, "
                    f"Landmark {landmark_month}月"
                    "存在非法long row。"
                )

            if not np.all(
                evaluation.landmark_index_long[
                    long_rows
                ]
                == landmark_index
            ):
                raise ValueError(
                    "patient-landmark到Step6"
                    "长表映射错误。"
                )

            alpha, beta = (
                load_crossfit_calibrator(
                    calibration_table,
                    fold_id=fold_id,
                    landmark_month=(
                        landmark_month
                    ),
                )
            )

            progress_print(
                f"Fold {fold_id} | "
                f"Landmark {landmark_month}月："
                f"baseline FP32前向，"
                f"OOF origins={len(patients)}。"
            )

            baseline_risk = (
                get_or_predict_group(
                    bundle=bundle,
                    common=common,
                    fold_arrays=(
                        fold_arrays
                    ),
                    sample_pairs=(
                        sample_pairs
                    ),
                    patient_local=(
                        patients
                    ),
                    landmark_index=(
                        landmark_index
                    ),
                    group=None,
                    calibration_alpha=(
                        alpha
                    ),
                    calibration_beta=(
                        beta
                    ),
                    device=device,
                )
            )

            baseline_fp32_long[
                long_rows,
                :,
            ] = baseline_risk

            saved_baseline = np.asarray(
                evaluation.calibrated_risk_saved[
                    long_rows,
                    :,
                ],
                dtype=np.float32,
            )

            baseline_diff = np.abs(
                baseline_risk.astype(
                    np.float64
                )
                - saved_baseline.astype(
                    np.float64
                )
            )

            mean_diff = float(
                np.mean(
                    baseline_diff
                )
            )

            p99_diff = float(
                np.quantile(
                    baseline_diff,
                    0.99,
                )
            )

            max_diff = float(
                np.max(
                    baseline_diff
                )
            )

            if (
                mean_diff
                > FP32_BASELINE_MEAN_HARD_TOL
                or p99_diff
                > FP32_BASELINE_P99_HARD_TOL
            ):
                raise ValueError(
                    f"Fold {fold_id}, "
                    f"Landmark {landmark_month}月："
                    "FP32 baseline与Step11正式风险漂移异常，"
                    f"mean={mean_diff:.3e}, "
                    f"p99={p99_diff:.3e}, "
                    f"max={max_diff:.3e}。"
                )

            audit_rows.append(
                {
                    "fold_id": int(
                        fold_id
                    ),
                    "landmark_index": int(
                        landmark_index
                    ),
                    "landmark_month": int(
                        landmark_month
                    ),
                    "origin_n": int(
                        len(
                            patients
                        )
                    ),
                    "mean_abs_risk_diff_vs_saved_step11": (
                        mean_diff
                    ),
                    "p99_abs_risk_diff_vs_saved_step11": (
                        p99_diff
                    ),
                    "max_abs_risk_diff_vs_saved_step11": (
                        max_diff
                    ),
                }
            )

            progress_print(
                f"Fold {fold_id} | "
                f"Landmark {landmark_month}月："
                f"baseline审计通过 | "
                f"mean={mean_diff:.3e} | "
                f"p99={p99_diff:.3e} | "
                f"max={max_diff:.3e}"
            )

            for group_position, group in enumerate(
                resolved_groups,
                start=1,
            ):
                progress_print(
                    f"Fold {fold_id} | "
                    f"Landmark {landmark_month}月 | "
                    f"group {group_position}/"
                    f"{len(resolved_groups)} | "
                    f"{group.group_label}"
                )

                group_risk = (
                    get_or_predict_group(
                        bundle=bundle,
                        common=common,
                        fold_arrays=(
                            fold_arrays
                        ),
                        sample_pairs=(
                            sample_pairs
                        ),
                        patient_local=(
                            patients
                        ),
                        landmark_index=(
                            landmark_index
                        ),
                        group=group,
                        calibration_alpha=(
                            alpha
                        ),
                        calibration_beta=(
                            beta
                        ),
                        device=device,
                    )
                )

                group_risk_long[
                    (
                        landmark_index,
                        group.group_key,
                    )
                ][
                    long_rows,
                    :,
                ] = group_risk

        del bundle
        del fold_arrays
        gc.collect()
        torch.cuda.empty_cache()

    if (
        not mapping_frames
        or resolved_group_reference
        is None
    ):
        raise RuntimeError(
            "没有生成group mapping。"
        )

    all_mapping_df = pd.concat(
        mapping_frames,
        axis=0,
        ignore_index=True,
    )

    all_mapping_df.to_csv(
        OUTPUT_DIR
        / "grouped_occlusion_channel_mapping.csv",
        index=False,
        encoding="utf-8-sig",
    )

    group_definition_rows = []

    for group in (
        resolved_group_reference
    ):
        group_definition_rows.append(
            {
                "group_key": (
                    group.group_key
                ),
                "group_label": (
                    group.group_label
                ),
                "clinical_features": (
                    "|".join(
                        group.clinical_features
                    )
                ),
                "dynamic_channel_n": int(
                    len(
                        group.dynamic_indices
                    )
                ),
                "static_channel_n": int(
                    len(
                        group.static_indices
                    )
                ),
                "age_included": bool(
                    group.include_age
                ),
                "total_input_channel_n": int(
                    len(
                        group.dynamic_indices
                    )
                    + len(
                        group.static_indices
                    )
                    + int(
                        group.include_age
                    )
                ),
            }
        )

    group_definition_df = (
        pd.DataFrame(
            group_definition_rows
        )
    )

    group_definition_df.to_csv(
        OUTPUT_DIR
        / "grouped_occlusion_group_definition.csv",
        index=False,
        encoding="utf-8-sig",
    )

    primary_long_mask = np.isin(
        evaluation.landmark_index_long,
        PRIMARY_LANDMARK_INDICES,
    )

    if not np.isfinite(
        baseline_fp32_long[
            primary_long_mask
        ]
    ).all():
        raise ValueError(
            "主Landmark FP32 baseline"
            "存在未填充预测。"
        )

    for (
        landmark_index,
        group_key,
    ), risk_array in (
        group_risk_long.items()
    ):
        landmark_rows = (
            evaluation.landmark_index_long
            == int(
                landmark_index
            )
        )

        if not np.isfinite(
            risk_array[
                landmark_rows
            ]
        ).all():
            raise ValueError(
                f"Landmark "
                f"{LANDMARK_MONTHS[landmark_index]}月，"
                f"group={group_key}"
                "存在未填充OOF预测。"
            )

    audit_df = pd.DataFrame(
        audit_rows
    )

    audit_df.to_csv(
        OUTPUT_DIR
        / "fp32_baseline_vs_saved_step11_audit.csv",
        index=False,
        encoding="utf-8-sig",
    )

    metric_rows: list[
        dict[str, Any]
    ] = []

    horizon_metric_rows: list[
        dict[str, Any]
    ] = []

    patient_risk_rows: list[
        dict[str, Any]
    ] = []

    baseline_metric_rows: list[
        dict[str, Any]
    ] = []

    group_lookup = {
        group.group_key: group
        for group in (
            resolved_group_reference
        )
    }

    for landmark_index in (
        PRIMARY_LANDMARK_INDICES
    ):
        landmark_index = int(
            landmark_index
        )

        landmark_month = int(
            LANDMARK_MONTHS[
                landmark_index
            ]
        )

        rows = np.where(
            evaluation.landmark_index_long
            == landmark_index
        )[0]

        survival_reference = (
            build_survival_array(
                evaluation.event_within_60m[
                    rows
                ],
                evaluation.analysis_time_month[
                    rows
                ],
            )
        )

        saved_baseline_matrix = np.asarray(
            evaluation.calibrated_risk_saved[
                rows,
                :,
            ],
            dtype=np.float32,
        )

        fp32_baseline_matrix = (
            baseline_fp32_long[
                rows,
                :,
            ]
        )

        saved_baseline_metrics = (
            evaluate_survival_predictions(
                survival_reference,
                saved_baseline_matrix,
            )
        )

        baseline_metrics = (
            evaluate_survival_predictions(
                survival_reference,
                fp32_baseline_matrix,
            )
        )

        baseline_metric_rows.append(
            {
                "landmark_index": int(
                    landmark_index
                ),
                "landmark_month": int(
                    landmark_month
                ),
                "landmark_year": float(
                    landmark_month
                    / 12.0
                ),
                "risk_set_n": int(
                    len(
                        rows
                    )
                ),
                "future_5y_event_n": int(
                    evaluation.event_within_60m[
                        rows
                    ].sum()
                ),
                "saved_step11_uno_c": float(
                    saved_baseline_metrics[
                        "uno_c_index_5y"
                    ]
                ),
                "fp32_baseline_uno_c": float(
                    baseline_metrics[
                        "uno_c_index_5y"
                    ]
                ),
                "saved_step11_iAUC": float(
                    saved_baseline_metrics[
                        "integrated_dynamic_auc"
                    ]
                ),
                "fp32_baseline_iAUC": float(
                    baseline_metrics[
                        "integrated_dynamic_auc"
                    ]
                ),
                "saved_step11_IBS": float(
                    saved_baseline_metrics[
                        "integrated_brier"
                    ]
                ),
                "fp32_baseline_IBS": float(
                    baseline_metrics[
                        "integrated_brier"
                    ]
                ),
            }
        )

        patient_local_for_rows = (
            evaluation.local_patient_idx_long[
                rows
            ].astype(
                np.int32
            )
        )

        for group_key in (
            GROUP_PLOT_ORDER
        ):
            group = group_lookup[
                group_key
            ]

            occluded_matrix = (
                group_risk_long[
                    (
                        landmark_index,
                        group_key,
                    )
                ][
                    rows,
                    :,
                ]
            )

            occluded_metrics = (
                evaluate_survival_predictions(
                    survival_reference,
                    occluded_matrix,
                )
            )

            delta_risk5 = (
                occluded_matrix[
                    :,
                    -1,
                ].astype(
                    np.float64
                )
                - fp32_baseline_matrix[
                    :,
                    -1,
                ].astype(
                    np.float64
                )
            )

            metric_rows.append(
                group_metric_row(
                    group=group,
                    landmark_index=(
                        landmark_index
                    ),
                    landmark_month=(
                        landmark_month
                    ),
                    risk_set_n=len(
                        rows
                    ),
                    event_n=int(
                        evaluation.event_within_60m[
                            rows
                        ].sum()
                    ),
                    baseline_metrics=(
                        baseline_metrics
                    ),
                    occluded_metrics=(
                        occluded_metrics
                    ),
                    baseline_saved_metrics=(
                        saved_baseline_metrics
                    ),
                    patient_delta_risk5=(
                        delta_risk5
                    ),
                )
            )

            horizon_metric_rows.extend(
                group_horizon_rows(
                    group=group,
                    landmark_index=(
                        landmark_index
                    ),
                    landmark_month=(
                        landmark_month
                    ),
                    baseline_metrics=(
                        baseline_metrics
                    ),
                    occluded_metrics=(
                        occluded_metrics
                    ),
                )
            )

            for i, long_row in enumerate(
                rows
            ):
                patient_risk_rows.append(
                    {
                        "long_row": int(
                            long_row
                        ),
                        "patient_local": int(
                            patient_local_for_rows[
                                i
                            ]
                        ),
                        "fold_id": int(
                            common.development_fold_id[
                                patient_local_for_rows[
                                    i
                                ]
                            ]
                        ),
                        "landmark_index": int(
                            landmark_index
                        ),
                        "landmark_month": int(
                            landmark_month
                        ),
                        "landmark_year": float(
                            landmark_month
                            / 12.0
                        ),
                        "group_key": (
                            group.group_key
                        ),
                        "group_label": (
                            group.group_label
                        ),
                        "baseline_risk_5y": float(
                            fp32_baseline_matrix[
                                i,
                                -1,
                            ]
                        ),
                        "occluded_risk_5y": float(
                            occluded_matrix[
                                i,
                                -1,
                            ]
                        ),
                        "delta_risk_5y": float(
                            delta_risk5[
                                i
                            ]
                        ),
                        "abs_delta_risk_5y": float(
                            abs(
                                delta_risk5[
                                    i
                                ]
                            )
                        ),
                    }
                )

    metrics_df = pd.DataFrame(
        metric_rows
    )

    horizon_df = pd.DataFrame(
        horizon_metric_rows
    )

    patient_df = pd.DataFrame(
        patient_risk_rows
    )

    baseline_metrics_df = (
        pd.DataFrame(
            baseline_metric_rows
        )
    )

    for metric_name, rank_name in [
        (
            "iAUC_loss",
            "iAUC_loss_rank",
        ),
        (
            "IBS_increase",
            "IBS_increase_rank",
        ),
        (
            "uno_c_loss",
            "uno_c_loss_rank",
        ),
        (
            "mean_abs_delta_risk5",
            "risk_change_rank",
        ),
    ]:
        metrics_df[
            rank_name
        ] = (
            metrics_df.groupby(
                "landmark_index"
            )[
                metric_name
            ]
            .rank(
                method="first",
                ascending=False,
            )
            .astype(
                int
            )
        )

    baseline_metrics_df.to_csv(
        OUTPUT_DIR
        / "grouped_occlusion_baseline_metric_audit.csv",
        index=False,
        encoding="utf-8-sig",
    )

    metrics_df.to_csv(
        OUTPUT_DIR
        / "grouped_occlusion_metrics.csv",
        index=False,
        encoding="utf-8-sig",
    )

    horizon_df.to_csv(
        OUTPUT_DIR
        / "grouped_occlusion_horizon_metrics.csv",
        index=False,
        encoding="utf-8-sig",
    )

    patient_df.to_csv(
        OUTPUT_DIR
        / "grouped_occlusion_patient_level_5y_risk_change.csv",
        index=False,
        encoding="utf-8-sig",
    )

    plot_group_metric(
        metrics_df=metrics_df,
        y_column=(
            "iAUC_loss"
        ),
        y_label=(
            "iAUC loss after group occlusion"
        ),
        stem=(
            "Figure12C1_group_iAUC_loss"
        ),
    )

    plot_group_metric(
        metrics_df=metrics_df,
        y_column=(
            "IBS_increase"
        ),
        y_label=(
            "IBS increase after group occlusion"
        ),
        stem=(
            "Figure12C2_group_IBS_increase"
        ),
    )

    plot_group_metric(
        metrics_df=metrics_df,
        y_column=(
            "uno_c_loss"
        ),
        y_label=(
            "Uno C-index loss after group occlusion"
        ),
        stem=(
            "Figure12C3_group_uno_c_loss"
        ),
    )

    plot_group_metric(
        metrics_df=metrics_df,
        y_column=(
            "mean_abs_delta_risk5"
        ),
        y_label=(
            "Mean absolute change in 5-year CKD risk"
        ),
        stem=(
            "Figure12C4_group_5y_risk_change"
        ),
    )

    print(
        "\n"
        + "=" * 118
    )
    print(
        "Step 12C Grouped Clinical-Domain Reference Occlusion完成"
    )
    print(
        "=" * 118
    )
    print(
        "锁定测试集：未读取"
    )
    print(
        "总耗时：",
        format_duration(
            time.time()
            - total_start
        ),
    )

    print(
        "\nFP32 baseline vs saved Step11 metric audit："
    )
    print(
        baseline_metrics_df.to_string(
            index=False
        )
    )

    print(
        "\n各Landmark按iAUC loss排序的临床域："
    )

    for landmark_index in (
        PRIMARY_LANDMARK_INDICES
    ):
        landmark_index = int(
            landmark_index
        )

        sub = metrics_df.loc[
            metrics_df[
                "landmark_index"
            ]
            == landmark_index
        ].sort_values(
            "iAUC_loss",
            ascending=False,
        )

        landmark_year = float(
            LANDMARK_MONTHS[
                landmark_index
            ]
            / 12.0
        )

        print(
            "\n"
            + "-" * 100
        )
        print(
            f"Landmark {landmark_year:.0f}年"
        )
        print(
            sub[
                [
                    "iAUC_loss_rank",
                    "group_label",
                    "iAUC_loss",
                    "IBS_increase",
                    "uno_c_loss",
                    "mean_abs_delta_risk5",
                    "occluded_dynamic_channel_n",
                    "occluded_static_channel_n",
                    "age_occluded",
                ]
            ].to_string(
                index=False
            )
        )

    summary = {
        "stage": (
            "Step12C_LSTM_v2_grouped_clinical_domain_reference_occlusion"
        ),
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "primary_landmark_months": (
            PRIMARY_LANDMARK_MONTHS.tolist()
        ),
        "future_end_months": (
            FUTURE_END_MONTHS.tolist()
        ),
        "metric_times": (
            METRIC_TIMES.tolist()
        ),
        "group_keys": (
            GROUP_PLOT_ORDER
        ),
        "group_labels": (
            GROUP_LABEL_MAP
        ),
        "occlusion_strategy": {
            "dynamic": (
                "replace all selected group dynamic channels "
                "at each active pre-landmark history step with "
                "heldout-fold-excluded training-risk-set "
                "landmark/step-specific reference"
            ),
            "laboratory_representation": (
                "value + observed + time_since_last + "
                "delta_last_observed occluded together"
            ),
            "static": (
                "selected static channels replaced by "
                "landmark-specific training-risk-set reference"
            ),
            "age": (
                "if included in group, replaced by "
                "landmark-specific training-risk-set "
                "standardized age reference"
            ),
            "row_mask": (
                "kept unchanged"
            ),
            "landmark_context": (
                "kept unchanged"
            ),
            "calibrator": (
                "original Step11 cross-fitted calibrator; "
                "no recalibration after occlusion"
            ),
            "inference_precision": (
                "FP32 for baseline and occlusion"
            ),
        },
        "interpretation_scope": (
            "conditional reliance of the frozen LSTM on each "
            "predefined input domain; not a retrained ablation; "
            "group effects are not assumed additive"
        ),
        "locked_test_used": False,
        "elapsed_seconds": float(
            time.time()
            - total_start
        ),
        "output_dir": str(
            OUTPUT_DIR
        ),
    }

    save_json(
        summary,
        OUTPUT_DIR
        / "step12c_grouped_occlusion_summary.json",
    )

    save_json(
        {
            "completed": True,
            "completed_at": (
                datetime.now().isoformat(
                    timespec="seconds"
                )
            ),
            "locked_test_used": False,
        },
        OUTPUT_DIR
        / "completed.json",
    )

    print(
        "\n最重要结果文件："
    )

    for filename in [
        "grouped_occlusion_metrics.csv",
        "grouped_occlusion_group_definition.csv",
        "grouped_occlusion_channel_mapping.csv",
        "grouped_occlusion_baseline_metric_audit.csv",
        "grouped_occlusion_horizon_metrics.csv",
        "grouped_occlusion_patient_level_5y_risk_change.csv",
    ]:
        print(
            " -",
            OUTPUT_DIR
            / filename,
        )

    print(
        "\n输出目录：",
        OUTPUT_DIR,
    )
    print(
        "=" * 118
    )


if __name__ == "__main__":
    run()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Step 12C-2：LSTM-v2 Grouped Clinical-Domain Occlusion 患者级 paired bootstrap 95% CI

目的
----
1. 不重新训练模型，不重新运行LSTM前向，不读取锁定测试集。
2. 直接读取Step12C已经保存的baseline与10个临床域group occlusion累计风险预测。
3. 完全沿用Step11/Step12B-2患者级paired bootstrap框架：
   - 每个replicate从全部22,337名开发患者中有放回抽样22,337次；
   - 同一批bootstrap患者同时用于4个主Landmark与全部10个group；
   - 某Landmark无有效prediction origin的患者自动排除；
   - IPCW survival_train/reference始终使用该Landmark完整原始OOF风险集。
4. 对每个Landmark × group计算：
   - Uno C-index loss = baseline - occluded；
   - iAUC loss = baseline - occluded；
   - IBS increase = occluded - baseline；
   - mean absolute change in 5-year cumulative CKD risk；
   并给出患者级paired bootstrap percentile 95% CI。
5. 正值统一表示“遮挡该临床域后预测表现变差”。
6. 支持断点续跑。正式分析默认1000次；测试时可临时：
      CKD_GROUP_BOOTSTRAP_REPS=50 python Step12C2_....py

解释边界
--------
- 这是冻结LSTM的post-hoc grouped input reference occlusion，不是重新训练后的ablation。
- 不同group包含的输入通道数不同，结果表示“整个临床域被替换为训练风险集reference后的性能变化”，
  不应除以通道数解释成单通道平均重要性。
- group效应不假定可加。
"""

from __future__ import annotations

import hashlib
import json
import os
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sksurv.metrics import (
    concordance_index_ipcw,
    cumulative_dynamic_auc,
    integrated_brier_score,
)
from sksurv.util import Surv


# =============================================================================
# 1. 路径与固定配置
# =============================================================================

PROJECT_DIR = Path(
    os.getenv(
        "CKD_LSTM_PROJECT_DIR",
        "__CKD_WORKDIR__",
    )
)

STEP4_DIR = (
    PROJECT_DIR
    / "rolling_5y_step4_preprocessed"
)

STEP6_DIR = (
    PROJECT_DIR
    / "rolling_5y_step6_super_landmark_data"
)

STEP12C_DIR = (
    PROJECT_DIR
    / "rolling_5y_step12c_lstm_v2_grouped_reference_occlusion_v1"
)

STEP12C_PREDICTION_DIR = (
    STEP12C_DIR
    / "prediction_checkpoints"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "rolling_5y_step12c2_grouped_occlusion_paired_bootstrap_v1"
)

BOOTSTRAP_DIR = (
    OUTPUT_DIR
    / "bootstrap_checkpoints"
)

FIGURE_DIR = (
    OUTPUT_DIR
    / "figures"
)

for directory in [
    OUTPUT_DIR,
    BOOTSTRAP_DIR,
    FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

EXPECTED_DEVELOPMENT_N = 22337
EXPECTED_LANDMARK_N = 6
EXPECTED_INTERVAL_N = 10
EXPECTED_LONG_N = 100122
EXPECTED_GROUP_N = 10
N_SPLITS = 5

LANDMARK_MONTHS = np.asarray(
    [0, 12, 24, 36, 48, 60],
    dtype=np.int32,
)

PRIMARY_LANDMARK_INDICES = np.asarray(
    [0, 1, 3, 5],
    dtype=np.int64,
)

PRIMARY_LANDMARK_MONTHS = (
    LANDMARK_MONTHS[
        PRIMARY_LANDMARK_INDICES
    ]
)

FUTURE_END_MONTHS = np.arange(
    6,
    61,
    6,
    dtype=np.float64,
)

# 与Step11、Step12B-2完全一致。
# 第10个风险仍表示未来60月风险；数值评价时用59.999月满足sksurv要求。
METRIC_TIMES = np.asarray(
    [
        6,
        12,
        18,
        24,
        30,
        36,
        42,
        48,
        54,
        59.999,
    ],
    dtype=np.float64,
)

BOOTSTRAP_REPS = int(
    os.getenv(
        "CKD_GROUP_BOOTSTRAP_REPS",
        "1000",
    )
)

RANDOM_SEED = int(
    os.getenv(
        "CKD_GROUP_BOOTSTRAP_SEED",
        "20260812",
    )
)

BOOTSTRAP_SAVE_EVERY = int(
    os.getenv(
        "CKD_GROUP_BOOTSTRAP_SAVE_EVERY",
        "10",
    )
)

PROGRESS_EVERY = int(
    os.getenv(
        "CKD_GROUP_BOOTSTRAP_PROGRESS_EVERY",
        "25",
    )
)

MINIMUM_VALID_BOOTSTRAP_RATE = 0.80
POINT_AUDIT_TOL = 5e-8

PIPELINE_VERSION = (
    "step12c2_grouped_occlusion_"
    "patient_paired_bootstrap_v1"
)

BOOTSTRAP_METRICS = [
    "uno_c_loss",
    "iAUC_loss",
    "IBS_increase",
    "mean_abs_delta_risk5",
]

METRIC_INDEX = {
    metric_name: metric_index
    for metric_index, metric_name
    in enumerate(
        BOOTSTRAP_METRICS
    )
}

STEP12C_METRICS_FILE = (
    STEP12C_DIR
    / "grouped_occlusion_metrics.csv"
)

STEP12C_GROUP_DEFINITION_FILE = (
    STEP12C_DIR
    / "grouped_occlusion_group_definition.csv"
)

STEP12C_CHANNEL_MAPPING_FILE = (
    STEP12C_DIR
    / "grouped_occlusion_channel_mapping.csv"
)

STEP12C_BASELINE_AUDIT_FILE = (
    STEP12C_DIR
    / "grouped_occlusion_baseline_metric_audit.csv"
)

STEP12C_CHECKPOINT_METADATA_FILE = (
    STEP12C_PREDICTION_DIR
    / "checkpoint_metadata.json"
)


# =============================================================================
# 2. 通用工具
# =============================================================================

def require_files(
    paths: list[Path],
) -> None:
    missing = [
        str(path)
        for path in paths
        if not path.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "以下必要输入文件不存在：\n"
            + "\n".join(
                missing
            )
        )


def save_json(
    value: Any,
    path: Path,
) -> None:
    path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )


def sha256_file(
    path: Path,
) -> str:
    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as file:
        while True:
            block = file.read(
                1024 * 1024
            )
            if not block:
                break
            digest.update(
                block
            )

    return digest.hexdigest()


def format_duration(
    seconds: float,
) -> str:
    seconds = max(
        0,
        int(
            round(
                float(
                    seconds
                )
            )
        ),
    )

    hours, remainder = divmod(
        seconds,
        3600,
    )

    minutes, seconds = divmod(
        remainder,
        60,
    )

    if hours:
        return (
            f"{hours}小时"
            f"{minutes:02d}分"
            f"{seconds:02d}秒"
        )

    if minutes:
        return (
            f"{minutes}分"
            f"{seconds:02d}秒"
        )

    return f"{seconds}秒"


def percentile_interval(
    values: np.ndarray,
    expected_n: int,
) -> tuple[
    float,
    float,
    float,
    int,
]:
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    valid = values[
        np.isfinite(
            values
        )
    ]

    required_n = int(
        np.ceil(
            expected_n
            * MINIMUM_VALID_BOOTSTRAP_RATE
        )
    )

    if len(valid) < required_n:
        raise RuntimeError(
            f"仅获得{len(valid)}/{expected_n}"
            "个有效bootstrap估计。"
        )

    lower = float(
        np.percentile(
            valid,
            2.5,
        )
    )

    upper = float(
        np.percentile(
            valid,
            97.5,
        )
    )

    median = float(
        np.median(
            valid
        )
    )

    return (
        lower,
        upper,
        median,
        int(
            len(
                valid
            )
        ),
    )


def positive_fraction(
    values: np.ndarray,
) -> float:
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    valid = values[
        np.isfinite(
            values
        )
    ]

    if len(valid) == 0:
        return np.nan

    return float(
        np.mean(
            valid > 0
        )
    )


def display_table(
    frame: pd.DataFrame,
    title: str,
) -> None:
    print(
        "\n"
        + "=" * 120
    )
    print(
        title
    )
    print(
        "=" * 120
    )

    try:
        from IPython.display import display
        display(
            frame
        )
    except ImportError:
        print(
            frame.to_string(
                index=False
            )
        )


# =============================================================================
# 3. 生存评价——与Step11/Step12B-2一致
# =============================================================================

def build_survival_array(
    event: np.ndarray,
    time_month: np.ndarray,
) -> np.ndarray:
    event = np.asarray(
        event,
        dtype=bool,
    )

    time_month = np.asarray(
        time_month,
        dtype=np.float64,
    )

    if event.shape != time_month.shape:
        raise ValueError(
            "事件和随访时间形状不一致。"
        )

    if not np.isfinite(
        time_month
    ).all():
        raise ValueError(
            "随访时间存在NaN或无穷值。"
        )

    if np.any(
        time_month <= 0
    ):
        raise ValueError(
            "Landmark后随访时间必须>0。"
        )

    return Surv.from_arrays(
        event=event,
        time=time_month,
    )


def evaluate_survival_predictions(
    survival_train: np.ndarray,
    survival_test: np.ndarray,
    risk_matrix: np.ndarray,
) -> dict[str, float]:
    risk_matrix = np.asarray(
        risk_matrix,
        dtype=np.float64,
    )

    if (
        risk_matrix.ndim != 2
        or risk_matrix.shape[1]
        != EXPECTED_INTERVAL_N
    ):
        raise ValueError(
            "risk_matrix形状非法。"
        )

    survival_matrix = (
        1.0
        - risk_matrix
    )

    c_index = float(
        concordance_index_ipcw(
            survival_train,
            survival_test,
            risk_matrix[
                :,
                -1,
            ],
            tau=float(
                METRIC_TIMES[
                    -1
                ]
            ),
        )[0]
    )

    _, mean_auc = (
        cumulative_dynamic_auc(
            survival_train,
            survival_test,
            risk_matrix,
            METRIC_TIMES,
        )
    )

    ibs = float(
        integrated_brier_score(
            survival_train,
            survival_test,
            survival_matrix,
            METRIC_TIMES,
        )
    )

    return {
        "uno_c_index_5y": (
            c_index
        ),
        "integrated_dynamic_auc": float(
            mean_auc
        ),
        "integrated_brier": (
            ibs
        ),
    }


def safe_evaluate_survival_predictions(
    survival_train: np.ndarray,
    survival_test: np.ndarray,
    risk_matrix: np.ndarray,
) -> dict[str, float] | None:
    try:
        return (
            evaluate_survival_predictions(
                survival_train,
                survival_test,
                risk_matrix,
            )
        )

    except (
        ValueError,
        ArithmeticError,
        ZeroDivisionError,
        FloatingPointError,
    ):
        return None


# =============================================================================
# 4. Step4 + Step6共同数组
# =============================================================================

def load_common_arrays() -> dict[str, np.ndarray]:
    required = [
        STEP4_DIR
        / "development_fold_id.npy",

        STEP6_DIR
        / "development_long_row_index_map.npy",

        STEP6_DIR
        / "development_local_patient_idx_long.npy",

        STEP6_DIR
        / "development_landmark_index_long.npy",

        STEP6_DIR
        / "development_analysis_time_month.npy",

        STEP6_DIR
        / "development_event_within_60m.npy",
    ]

    require_files(
        required
    )

    development_fold_id = np.load(
        STEP4_DIR
        / "development_fold_id.npy"
    ).astype(
        np.int8
    )

    row_index_map = np.load(
        STEP6_DIR
        / "development_long_row_index_map.npy"
    ).astype(
        np.int32
    )

    local_patient_idx_long = np.load(
        STEP6_DIR
        / "development_local_patient_idx_long.npy"
    ).astype(
        np.int32
    )

    landmark_index_long = np.load(
        STEP6_DIR
        / "development_landmark_index_long.npy"
    ).astype(
        np.int8
    )

    analysis_time_month = np.load(
        STEP6_DIR
        / "development_analysis_time_month.npy"
    ).astype(
        np.float64
    )

    event_within_60m = np.load(
        STEP6_DIR
        / "development_event_within_60m.npy"
    ).astype(
        bool
    )

    arrays = {
        "development_fold_id": (
            development_fold_id
        ),
        "row_index_map": (
            row_index_map
        ),
        "local_patient_idx_long": (
            local_patient_idx_long
        ),
        "landmark_index_long": (
            landmark_index_long
        ),
        "analysis_time_month": (
            analysis_time_month
        ),
        "event_within_60m": (
            event_within_60m
        ),
    }

    expected_shapes = {
        "development_fold_id": (
            EXPECTED_DEVELOPMENT_N,
        ),
        "row_index_map": (
            EXPECTED_DEVELOPMENT_N,
            EXPECTED_LANDMARK_N,
        ),
        "local_patient_idx_long": (
            EXPECTED_LONG_N,
        ),
        "landmark_index_long": (
            EXPECTED_LONG_N,
        ),
        "analysis_time_month": (
            EXPECTED_LONG_N,
        ),
        "event_within_60m": (
            EXPECTED_LONG_N,
        ),
    }

    for name, expected_shape in (
        expected_shapes.items()
    ):
        if (
            arrays[
                name
            ].shape
            != expected_shape
        ):
            raise ValueError(
                f"{name}形状="
                f"{arrays[name].shape}，"
                f"预期={expected_shape}。"
            )

    valid_map = (
        row_index_map
        >= 0
    )

    expected_valid_map = np.column_stack(
        [
            np.bincount(
                local_patient_idx_long[
                    landmark_index_long
                    == landmark_index
                ],
                minlength=(
                    EXPECTED_DEVELOPMENT_N
                ),
            )
            > 0
            for landmark_index
            in range(
                EXPECTED_LANDMARK_N
            )
        ]
    )

    if not np.array_equal(
        valid_map,
        expected_valid_map,
    ):
        raise ValueError(
            "Step6 row_index_map与"
            "long格式患者-Landmark映射不一致。"
        )

    return arrays


# =============================================================================
# 5. Step12C group定义/point-estimate结果
# =============================================================================

def load_group_tables() -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    require_files(
        [
            STEP12C_METRICS_FILE,
            STEP12C_GROUP_DEFINITION_FILE,
            STEP12C_CHANNEL_MAPPING_FILE,
            STEP12C_BASELINE_AUDIT_FILE,
            STEP12C_CHECKPOINT_METADATA_FILE,
        ]
    )

    metrics_df = pd.read_csv(
        STEP12C_METRICS_FILE,
        encoding="utf-8-sig",
    )

    group_definition_df = pd.read_csv(
        STEP12C_GROUP_DEFINITION_FILE,
        encoding="utf-8-sig",
    )

    required_metric_columns = {
        "group_key",
        "group_label",
        "landmark_index",
        "landmark_month",
        "landmark_year",
        "risk_set_n",
        "future_5y_event_n",
        "occluded_dynamic_channel_n",
        "occluded_static_channel_n",
        "age_occluded",
        "total_occluded_input_channel_n",
        "clinical_features",
        "fp32_baseline_uno_c",
        "fp32_baseline_iAUC",
        "fp32_baseline_IBS",
        "occluded_uno_c",
        "occluded_iAUC",
        "occluded_IBS",
        "uno_c_loss",
        "iAUC_loss",
        "IBS_increase",
        "mean_abs_delta_risk5",
    }

    missing = sorted(
        required_metric_columns
        - set(
            metrics_df.columns
        )
    )

    if missing:
        raise ValueError(
            "Step12C grouped_occlusion_metrics.csv缺少列："
            + ", ".join(
                missing
            )
        )

    required_definition_columns = {
        "group_key",
        "group_label",
        "clinical_features",
        "dynamic_channel_n",
        "static_channel_n",
        "age_included",
        "total_input_channel_n",
    }

    missing_definition = sorted(
        required_definition_columns
        - set(
            group_definition_df.columns
        )
    )

    if missing_definition:
        raise ValueError(
            "Step12C group definition缺少列："
            + ", ".join(
                missing_definition
            )
        )

    metrics_df = metrics_df.loc[
        metrics_df[
            "landmark_index"
        ].isin(
            PRIMARY_LANDMARK_INDICES
        )
    ].copy()

    metrics_df[
        "landmark_index"
    ] = metrics_df[
        "landmark_index"
    ].astype(
        int
    )

    metrics_df[
        "landmark_month"
    ] = metrics_df[
        "landmark_month"
    ].astype(
        int
    )

    group_definition_df = (
        group_definition_df.copy()
    )

    if len(
        group_definition_df
    ) != EXPECTED_GROUP_N:
        raise ValueError(
            "Step12C group definition"
            f"应为{EXPECTED_GROUP_N}组，"
            f"实际={len(group_definition_df)}。"
        )

    if group_definition_df[
        "group_key"
    ].duplicated().any():
        raise ValueError(
            "Step12C group_key存在重复。"
        )

    group_keys = (
        group_definition_df[
            "group_key"
        ]
        .astype(
            str
        )
        .tolist()
    )

    expected_row_n = (
        len(
            PRIMARY_LANDMARK_INDICES
        )
        * EXPECTED_GROUP_N
    )

    if len(
        metrics_df
    ) != expected_row_n:
        raise ValueError(
            "Step12C主Landmark metric行数异常："
            f"{len(metrics_df)}，预期={expected_row_n}。"
        )

    for landmark_index in (
        PRIMARY_LANDMARK_INDICES
    ):
        landmark_index = int(
            landmark_index
        )

        sub = metrics_df.loc[
            metrics_df[
                "landmark_index"
            ]
            == landmark_index
        ]

        observed_keys = set(
            sub[
                "group_key"
            ].astype(
                str
            )
        )

        if observed_keys != set(
            group_keys
        ):
            raise ValueError(
                f"Landmark {LANDMARK_MONTHS[landmark_index]}月"
                "group集合与定义文件不一致。"
            )

        if sub[
            "group_key"
        ].duplicated().any():
            raise ValueError(
                f"Landmark {LANDMARK_MONTHS[landmark_index]}月"
                "存在重复group。"
            )

    # 以定义文件顺序固定所有Landmark的group顺序，防止按结果排名后顺序改变。
    group_order = {
        group_key: position
        for position, group_key
        in enumerate(
            group_keys
        )
    }

    metrics_df[
        "group_position"
    ] = (
        metrics_df[
            "group_key"
        ]
        .map(
            group_order
        )
        .astype(
            int
        )
    )

    metrics_df = metrics_df.sort_values(
        [
            "landmark_index",
            "group_position",
        ]
    ).reset_index(
        drop=True
    )

    metrics_df[
        "condition_position"
    ] = np.arange(
        len(
            metrics_df
        ),
        dtype=np.int32,
    )

    return (
        metrics_df,
        group_definition_df,
    )


# =============================================================================
# 6. 读取Step12C prediction checkpoints并拼回完整OOF风险集
# =============================================================================

class PredictionStore:
    def __init__(
        self,
        common: dict[str, np.ndarray],
        metrics_df: pd.DataFrame,
        group_definition_df: pd.DataFrame,
    ) -> None:
        self.common = common
        self.metrics_df = metrics_df
        self.group_definition_df = (
            group_definition_df
        )

        self.group_keys = (
            group_definition_df[
                "group_key"
            ]
            .astype(
                str
            )
            .tolist()
        )

        self.rows_by_landmark: dict[
            int,
            np.ndarray,
        ] = {}

        self.position_lookup_by_landmark: dict[
            int,
            np.ndarray,
        ] = {}

        self.survival_reference_by_landmark: dict[
            int,
            np.ndarray,
        ] = {}

        self.risk_matrix_by_key: dict[
            tuple[
                int,
                str | None,
            ],
            np.ndarray,
        ] = {}

        self._checkpoint_patient_position: dict[
            tuple[
                int,
                int,
            ],
            np.ndarray,
        ] = {}

        self._prepare_landmark_maps()
        self._load_all_predictions()

    def _prepare_landmark_maps(
        self,
    ) -> None:
        landmark_index_long = (
            self.common[
                "landmark_index_long"
            ]
        )

        event = self.common[
            "event_within_60m"
        ]

        time_month = self.common[
            "analysis_time_month"
        ]

        for landmark_index in (
            PRIMARY_LANDMARK_INDICES
        ):
            landmark_index = int(
                landmark_index
            )

            rows = np.where(
                landmark_index_long
                == landmark_index
            )[0].astype(
                np.int64
            )

            if len(rows) == 0:
                raise ValueError(
                    f"Landmark {LANDMARK_MONTHS[landmark_index]}月"
                    "没有风险集。"
                )

            self.rows_by_landmark[
                landmark_index
            ] = rows

            position_lookup = np.full(
                EXPECTED_LONG_N,
                -1,
                dtype=np.int32,
            )

            position_lookup[
                rows
            ] = np.arange(
                len(rows),
                dtype=np.int32,
            )

            self.position_lookup_by_landmark[
                landmark_index
            ] = position_lookup

            self.survival_reference_by_landmark[
                landmark_index
            ] = build_survival_array(
                event[
                    rows
                ],
                time_month[
                    rows
                ],
            )

    def _checkpoint_patient_lookup(
        self,
        fold_id: int,
        landmark_month: int,
    ) -> np.ndarray:
        key = (
            int(
                fold_id
            ),
            int(
                landmark_month
            ),
        )

        if key in (
            self._checkpoint_patient_position
        ):
            return (
                self._checkpoint_patient_position[
                    key
                ]
            )

        checkpoint_dir = (
            STEP12C_PREDICTION_DIR
            / f"fold_{int(fold_id)}"
            / f"landmark_{int(landmark_month)}m"
        )

        patient_file = (
            checkpoint_dir
            / "patient_local.npy"
        )

        require_files(
            [
                patient_file
            ]
        )

        patient_local = np.load(
            patient_file
        ).astype(
            np.int32
        )

        if (
            len(
                np.unique(
                    patient_local
                )
            )
            != len(
                patient_local
            )
        ):
            raise ValueError(
                f"{patient_file}存在重复患者。"
            )

        if np.any(
            patient_local < 0
        ) or np.any(
            patient_local
            >= EXPECTED_DEVELOPMENT_N
        ):
            raise ValueError(
                f"{patient_file}存在非法患者索引。"
            )

        lookup = np.full(
            EXPECTED_DEVELOPMENT_N,
            -1,
            dtype=np.int32,
        )

        lookup[
            patient_local
        ] = np.arange(
            len(
                patient_local
            ),
            dtype=np.int32,
        )

        self._checkpoint_patient_position[
            key
        ] = lookup

        return lookup

    @staticmethod
    def _prediction_filename(
        group_key: str | None,
    ) -> str:
        if group_key is None:
            return (
                "baseline_calibrated_risk.npy"
            )

        return (
            f"group_{group_key}"
            "_calibrated_risk.npy"
        )

    def _assemble_landmark_prediction(
        self,
        landmark_index: int,
        group_key: str | None,
    ) -> np.ndarray:
        landmark_index = int(
            landmark_index
        )

        landmark_month = int(
            LANDMARK_MONTHS[
                landmark_index
            ]
        )

        rows = self.rows_by_landmark[
            landmark_index
        ]

        patient_for_rows = (
            self.common[
                "local_patient_idx_long"
            ][
                rows
            ]
        )

        fold_for_rows = (
            self.common[
                "development_fold_id"
            ][
                patient_for_rows
            ]
        )

        risk = np.full(
            (
                len(
                    rows
                ),
                EXPECTED_INTERVAL_N,
            ),
            np.nan,
            dtype=np.float32,
        )

        filename = (
            self._prediction_filename(
                group_key
            )
        )

        for fold_id in range(
            N_SPLITS
        ):
            target_position = np.where(
                fold_for_rows
                == fold_id
            )[0]

            if len(
                target_position
            ) == 0:
                continue

            checkpoint_dir = (
                STEP12C_PREDICTION_DIR
                / f"fold_{fold_id}"
                / f"landmark_{landmark_month}m"
            )

            prediction_file = (
                checkpoint_dir
                / filename
            )

            require_files(
                [
                    prediction_file
                ]
            )

            prediction = np.load(
                prediction_file,
                mmap_mode="r",
            )

            if (
                prediction.ndim != 2
                or prediction.shape[1]
                != EXPECTED_INTERVAL_N
            ):
                raise ValueError(
                    f"{prediction_file}形状非法："
                    f"{prediction.shape}"
                )

            lookup = (
                self._checkpoint_patient_lookup(
                    fold_id=fold_id,
                    landmark_month=landmark_month,
                )
            )

            target_patients = (
                patient_for_rows[
                    target_position
                ]
            )

            source_position = (
                lookup[
                    target_patients
                ]
            )

            if np.any(
                source_position
                < 0
            ):
                bad = target_patients[
                    source_position
                    < 0
                ][
                    :20
                ]

                raise ValueError(
                    f"Fold {fold_id}, "
                    f"Landmark {landmark_month}月"
                    "checkpoint缺少风险集患者："
                    f"{bad.tolist()}"
                )

            risk[
                target_position,
                :,
            ] = np.asarray(
                prediction[
                    source_position,
                    :,
                ],
                dtype=np.float32,
            )

        if not np.isfinite(
            risk
        ).all():
            raise ValueError(
                f"Landmark {landmark_month}月, "
                f"group={group_key}拼接后存在未填充预测。"
            )

        if np.any(
            risk < 0
        ) or np.any(
            risk > 1
        ):
            raise ValueError(
                "风险预测超出[0,1]。"
            )

        if np.any(
            np.diff(
                risk.astype(
                    np.float64
                ),
                axis=1,
            )
            < -1e-6
        ):
            raise ValueError(
                f"Landmark {landmark_month}月, "
                f"group={group_key}累计风险非单调。"
            )

        return risk

    def _load_all_predictions(
        self,
    ) -> None:
        for landmark_index in (
            PRIMARY_LANDMARK_INDICES
        ):
            landmark_index = int(
                landmark_index
            )

            self.risk_matrix_by_key[
                (
                    landmark_index,
                    None,
                )
            ] = (
                self._assemble_landmark_prediction(
                    landmark_index=landmark_index,
                    group_key=None,
                )
            )

            for group_key in (
                self.group_keys
            ):
                self.risk_matrix_by_key[
                    (
                        landmark_index,
                        group_key,
                    )
                ] = (
                    self._assemble_landmark_prediction(
                        landmark_index=landmark_index,
                        group_key=group_key,
                    )
                )

    def get(
        self,
        landmark_index: int,
        group_key: str | None,
    ) -> np.ndarray:
        key = (
            int(
                landmark_index
            ),
            group_key,
        )

        if key not in (
            self.risk_matrix_by_key
        ):
            raise KeyError(
                f"没有prediction matrix：{key}"
            )

        return (
            self.risk_matrix_by_key[
                key
            ]
        )


# =============================================================================
# 7. Step12C point-estimate重建审计
# =============================================================================

def audit_point_estimates(
    store: PredictionStore,
    metrics_df: pd.DataFrame,
) -> pd.DataFrame:
    baseline_audit = pd.read_csv(
        STEP12C_BASELINE_AUDIT_FILE,
        encoding="utf-8-sig",
    )

    audit_rows: list[
        dict[str, Any]
    ] = []

    for landmark_index in (
        PRIMARY_LANDMARK_INDICES
    ):
        landmark_index = int(
            landmark_index
        )

        landmark_month = int(
            LANDMARK_MONTHS[
                landmark_index
            ]
        )

        survival_reference = (
            store.survival_reference_by_landmark[
                landmark_index
            ]
        )

        baseline_risk = store.get(
            landmark_index,
            None,
        )

        baseline_metrics = (
            evaluate_survival_predictions(
                survival_reference,
                survival_reference,
                baseline_risk,
            )
        )

        saved_baseline = baseline_audit.loc[
            baseline_audit[
                "landmark_index"
            ].astype(
                int
            )
            == landmark_index
        ]

        if len(
            saved_baseline
        ) != 1:
            raise ValueError(
                f"Landmark {landmark_month}月"
                "baseline audit不是1行。"
            )

        saved_baseline = (
            saved_baseline.iloc[
                0
            ]
        )

        baseline_diffs = {
            "uno_c": abs(
                float(
                    baseline_metrics[
                        "uno_c_index_5y"
                    ]
                )
                - float(
                    saved_baseline[
                        "fp32_baseline_uno_c"
                    ]
                )
            ),
            "iAUC": abs(
                float(
                    baseline_metrics[
                        "integrated_dynamic_auc"
                    ]
                )
                - float(
                    saved_baseline[
                        "fp32_baseline_iAUC"
                    ]
                )
            ),
            "IBS": abs(
                float(
                    baseline_metrics[
                        "integrated_brier"
                    ]
                )
                - float(
                    saved_baseline[
                        "fp32_baseline_IBS"
                    ]
                )
            ),
        }

        if max(
            baseline_diffs.values()
        ) > POINT_AUDIT_TOL:
            raise ValueError(
                f"Landmark {landmark_month}月"
                "baseline point estimate重建不一致："
                f"{baseline_diffs}"
            )

        landmark_groups = metrics_df.loc[
            metrics_df[
                "landmark_index"
            ]
            == landmark_index
        ]

        for row in landmark_groups.itertuples(
            index=False
        ):
            group_key = str(
                row.group_key
            )

            occluded_risk = store.get(
                landmark_index,
                group_key,
            )

            occluded_metrics = (
                evaluate_survival_predictions(
                    survival_reference,
                    survival_reference,
                    occluded_risk,
                )
            )

            mean_abs_delta = float(
                np.mean(
                    np.abs(
                        occluded_risk[
                            :,
                            -1,
                        ].astype(
                            np.float64
                        )
                        - baseline_risk[
                            :,
                            -1,
                        ].astype(
                            np.float64
                        )
                    )
                )
            )

            differences = {
                "occluded_uno_c": abs(
                    float(
                        occluded_metrics[
                            "uno_c_index_5y"
                        ]
                    )
                    - float(
                        row.occluded_uno_c
                    )
                ),
                "occluded_iAUC": abs(
                    float(
                        occluded_metrics[
                            "integrated_dynamic_auc"
                        ]
                    )
                    - float(
                        row.occluded_iAUC
                    )
                ),
                "occluded_IBS": abs(
                    float(
                        occluded_metrics[
                            "integrated_brier"
                        ]
                    )
                    - float(
                        row.occluded_IBS
                    )
                ),
                "mean_abs_delta_risk5": abs(
                    mean_abs_delta
                    - float(
                        row.mean_abs_delta_risk5
                    )
                ),
            }

            max_difference = max(
                differences.values()
            )

            if (
                max_difference
                > POINT_AUDIT_TOL
            ):
                raise ValueError(
                    "Point estimate重建失败："
                    f"Landmark={landmark_month}月, "
                    f"group={group_key}, "
                    f"diff={differences}"
                )

            audit_rows.append(
                {
                    "landmark_index": (
                        landmark_index
                    ),
                    "landmark_month": (
                        landmark_month
                    ),
                    "group_key": (
                        group_key
                    ),
                    "group_label": (
                        row.group_label
                    ),
                    "max_abs_metric_difference": (
                        max_difference
                    ),
                }
            )

    audit_df = pd.DataFrame(
        audit_rows
    )

    audit_df.to_csv(
        OUTPUT_DIR
        / "step12c_point_estimate_reconstruction_audit.csv",
        index=False,
        encoding="utf-8-sig",
    )

    return audit_df


# =============================================================================
# 8. Bootstrap checkpoint与版本签名
# =============================================================================

def build_source_signature(
    metrics_df: pd.DataFrame,
    group_definition_df: pd.DataFrame,
) -> dict[str, Any]:
    return {
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "step12c_checkpoint_metadata_sha256": (
            sha256_file(
                STEP12C_CHECKPOINT_METADATA_FILE
            )
        ),
        "step12c_metrics_sha256": (
            sha256_file(
                STEP12C_METRICS_FILE
            )
        ),
        "step12c_group_definition_sha256": (
            sha256_file(
                STEP12C_GROUP_DEFINITION_FILE
            )
        ),
        "step12c_channel_mapping_sha256": (
            sha256_file(
                STEP12C_CHANNEL_MAPPING_FILE
            )
        ),
        "bootstrap_reps": int(
            BOOTSTRAP_REPS
        ),
        "random_seed": int(
            RANDOM_SEED
        ),
        "bootstrap_unit": (
            "development patient"
        ),
        "bootstrap_pairing": (
            "same sampled patients across all primary landmarks and all groups"
        ),
        "ipcw_reference": (
            "complete original OOF risk set within each landmark"
        ),
        "primary_landmark_indices": (
            PRIMARY_LANDMARK_INDICES.tolist()
        ),
        "primary_landmark_months": (
            PRIMARY_LANDMARK_MONTHS.tolist()
        ),
        "metric_times": (
            METRIC_TIMES.tolist()
        ),
        "group_keys": (
            group_definition_df[
                "group_key"
            ].astype(
                str
            ).tolist()
        ),
        "condition_order": (
            metrics_df[
                [
                    "landmark_index",
                    "group_key",
                ]
            ].to_dict(
                orient="records"
            )
        ),
        "bootstrap_metrics": (
            BOOTSTRAP_METRICS
        ),
        "locked_test_used": False,
    }


def initialize_bootstrap_storage(
    metrics_df: pd.DataFrame,
    group_definition_df: pd.DataFrame,
) -> tuple[
    np.ndarray,
    np.ndarray,
    Path,
]:
    source_signature = (
        build_source_signature(
            metrics_df=metrics_df,
            group_definition_df=(
                group_definition_df
            ),
        )
    )

    metadata_file = (
        BOOTSTRAP_DIR
        / (
            f"paired_bootstrap_"
            f"{BOOTSTRAP_REPS}_metadata.json"
        )
    )

    checkpoint_file = (
        BOOTSTRAP_DIR
        / (
            f"paired_bootstrap_"
            f"{BOOTSTRAP_REPS}_replicates.npz"
        )
    )

    if metadata_file.exists():
        observed = json.loads(
            metadata_file.read_text(
                encoding="utf-8"
            )
        )

        if observed != source_signature:
            raise ValueError(
                "已有bootstrap metadata"
                "与当前Step12C结果/配置不一致。"
                "请不要混用不同版本结果。"
            )

    else:
        if checkpoint_file.exists():
            raise ValueError(
                "检测到bootstrap checkpoint"
                "但缺少metadata。"
            )

        save_json(
            source_signature,
            metadata_file,
        )

    expected_shape = (
        BOOTSTRAP_REPS,
        len(
            metrics_df
        ),
        len(
            BOOTSTRAP_METRICS
        ),
    )

    if checkpoint_file.exists():
        checkpoint = np.load(
            checkpoint_file,
            allow_pickle=False,
        )

        values = checkpoint[
            "bootstrap_values"
        ].astype(
            np.float64
        )

        completed = checkpoint[
            "completed_replicates"
        ].astype(
            bool
        )

        if values.shape != expected_shape:
            raise ValueError(
                "已有bootstrap values"
                "形状与当前配置不一致。"
            )

        if completed.shape != (
            BOOTSTRAP_REPS,
        ):
            raise ValueError(
                "已有completed_replicates"
                "形状不一致。"
            )

        print(
            "\n检测到bootstrap断点："
            f"{int(completed.sum())}/"
            f"{BOOTSTRAP_REPS}已完成。"
        )

    else:
        values = np.full(
            expected_shape,
            np.nan,
            dtype=np.float64,
        )

        completed = np.zeros(
            BOOTSTRAP_REPS,
            dtype=bool,
        )

    return (
        values,
        completed,
        checkpoint_file,
    )


# =============================================================================
# 9. 患者级 paired bootstrap
# =============================================================================

def run_bootstrap(
    common: dict[str, np.ndarray],
    store: PredictionStore,
    metrics_df: pd.DataFrame,
    bootstrap_values: np.ndarray,
    completed_replicates: np.ndarray,
    checkpoint_file: Path,
) -> None:
    row_index_map = (
        common[
            "row_index_map"
        ]
    )

    event_within_60m = (
        common[
            "event_within_60m"
        ]
    )

    analysis_time_month = (
        common[
            "analysis_time_month"
        ]
    )

    bootstrap_start = time.time()

    for bootstrap_index in range(
        BOOTSTRAP_REPS
    ):
        if completed_replicates[
            bootstrap_index
        ]:
            continue

        # 与Step11/Step12B-2一致：每个replicate有独立、可复现seed。
        rng = np.random.default_rng(
            RANDOM_SEED
            + bootstrap_index
            * 1009
        )

        sampled_patients = rng.integers(
            0,
            EXPECTED_DEVELOPMENT_N,
            size=EXPECTED_DEVELOPMENT_N,
            endpoint=False,
        )

        for landmark_index in (
            PRIMARY_LANDMARK_INDICES
        ):
            landmark_index = int(
                landmark_index
            )

            sampled_long_rows = (
                row_index_map[
                    sampled_patients,
                    landmark_index,
                ]
            )

            sampled_long_rows = sampled_long_rows[
                sampled_long_rows
                >= 0
            ].astype(
                np.int64
            )

            if len(
                sampled_long_rows
            ) < 50:
                continue

            sampled_event = (
                event_within_60m[
                    sampled_long_rows
                ]
            )

            if int(
                sampled_event.sum()
            ) < 5:
                continue

            survival_test = (
                build_survival_array(
                    sampled_event,
                    analysis_time_month[
                        sampled_long_rows
                    ],
                )
            )

            survival_train = (
                store.survival_reference_by_landmark[
                    landmark_index
                ]
            )

            position_lookup = (
                store.position_lookup_by_landmark[
                    landmark_index
                ]
            )

            sampled_position = (
                position_lookup[
                    sampled_long_rows
                ]
            )

            if np.any(
                sampled_position
                < 0
            ):
                raise RuntimeError(
                    "bootstrap sampled row"
                    "无法映射回Landmark位置。"
                )

            baseline_risk_full = store.get(
                landmark_index,
                None,
            )

            baseline_risk = (
                baseline_risk_full[
                    sampled_position,
                    :,
                ]
            )

            baseline_metrics = (
                safe_evaluate_survival_predictions(
                    survival_train,
                    survival_test,
                    baseline_risk,
                )
            )

            if baseline_metrics is None:
                continue

            landmark_groups = metrics_df.loc[
                metrics_df[
                    "landmark_index"
                ]
                == landmark_index
            ]

            for row in landmark_groups.itertuples(
                index=False
            ):
                group_key = str(
                    row.group_key
                )

                occluded_risk_full = store.get(
                    landmark_index,
                    group_key,
                )

                occluded_risk = (
                    occluded_risk_full[
                        sampled_position,
                        :,
                    ]
                )

                occluded_metrics = (
                    safe_evaluate_survival_predictions(
                        survival_train,
                        survival_test,
                        occluded_risk,
                    )
                )

                if occluded_metrics is None:
                    continue

                condition_position = int(
                    row.condition_position
                )

                bootstrap_values[
                    bootstrap_index,
                    condition_position,
                    METRIC_INDEX[
                        "uno_c_loss"
                    ],
                ] = (
                    float(
                        baseline_metrics[
                            "uno_c_index_5y"
                        ]
                    )
                    - float(
                        occluded_metrics[
                            "uno_c_index_5y"
                        ]
                    )
                )

                bootstrap_values[
                    bootstrap_index,
                    condition_position,
                    METRIC_INDEX[
                        "iAUC_loss"
                    ],
                ] = (
                    float(
                        baseline_metrics[
                            "integrated_dynamic_auc"
                        ]
                    )
                    - float(
                        occluded_metrics[
                            "integrated_dynamic_auc"
                        ]
                    )
                )

                bootstrap_values[
                    bootstrap_index,
                    condition_position,
                    METRIC_INDEX[
                        "IBS_increase"
                    ],
                ] = (
                    float(
                        occluded_metrics[
                            "integrated_brier"
                        ]
                    )
                    - float(
                        baseline_metrics[
                            "integrated_brier"
                        ]
                    )
                )

                bootstrap_values[
                    bootstrap_index,
                    condition_position,
                    METRIC_INDEX[
                        "mean_abs_delta_risk5"
                    ],
                ] = float(
                    np.mean(
                        np.abs(
                            occluded_risk[
                                :,
                                -1,
                            ].astype(
                                np.float64
                            )
                            - baseline_risk[
                                :,
                                -1,
                            ].astype(
                                np.float64
                            )
                        )
                    )
                )

        completed_replicates[
            bootstrap_index
        ] = True

        completed_n = int(
            completed_replicates.sum()
        )

        if (
            completed_n
            % BOOTSTRAP_SAVE_EVERY
            == 0
            or completed_n
            == BOOTSTRAP_REPS
        ):
            np.savez_compressed(
                checkpoint_file,
                bootstrap_values=(
                    bootstrap_values
                ),
                completed_replicates=(
                    completed_replicates
                ),
            )

        if (
            completed_n
            % PROGRESS_EVERY
            == 0
            or completed_n
            == BOOTSTRAP_REPS
        ):
            print(
                "Paired bootstrap："
                f"{completed_n}/"
                f"{BOOTSTRAP_REPS} | "
                "本次运行 "
                f"{format_duration(time.time() - bootstrap_start)}",
                flush=True,
            )


# =============================================================================
# 10. 汇总95% CI
# =============================================================================

def summarize_bootstrap(
    metrics_df: pd.DataFrame,
    bootstrap_values: np.ndarray,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    wide_rows: list[
        dict[str, Any]
    ] = []

    long_rows: list[
        dict[str, Any]
    ] = []

    point_column = {
        "uno_c_loss": (
            "uno_c_loss"
        ),
        "iAUC_loss": (
            "iAUC_loss"
        ),
        "IBS_increase": (
            "IBS_increase"
        ),
        "mean_abs_delta_risk5": (
            "mean_abs_delta_risk5"
        ),
    }

    for row in metrics_df.itertuples(
        index=False
    ):
        condition_position = int(
            row.condition_position
        )

        base = {
            "condition_position": (
                condition_position
            ),
            "group_position": int(
                row.group_position
            ),
            "group_key": (
                row.group_key
            ),
            "group_label": (
                row.group_label
            ),
            "landmark_index": int(
                row.landmark_index
            ),
            "landmark_month": int(
                row.landmark_month
            ),
            "landmark_year": float(
                row.landmark_year
            ),
            "risk_set_n": int(
                row.risk_set_n
            ),
            "future_5y_event_n": int(
                row.future_5y_event_n
            ),
            "occluded_dynamic_channel_n": int(
                row.occluded_dynamic_channel_n
            ),
            "occluded_static_channel_n": int(
                row.occluded_static_channel_n
            ),
            "age_occluded": bool(
                row.age_occluded
            ),
            "total_occluded_input_channel_n": int(
                row.total_occluded_input_channel_n
            ),
            "clinical_features": (
                row.clinical_features
            ),
        }

        wide = dict(
            base
        )

        for metric_name in (
            BOOTSTRAP_METRICS
        ):
            values = (
                bootstrap_values[
                    :,
                    condition_position,
                    METRIC_INDEX[
                        metric_name
                    ],
                ]
            )

            (
                lower,
                upper,
                median,
                valid_n,
            ) = percentile_interval(
                values,
                expected_n=(
                    BOOTSTRAP_REPS
                ),
            )

            point = float(
                getattr(
                    row,
                    point_column[
                        metric_name
                    ],
                )
            )

            favorable = (
                positive_fraction(
                    values
                )
                if metric_name
                != "mean_abs_delta_risk5"
                else np.nan
            )

            ci_excludes_zero = bool(
                lower > 0
                or upper < 0
            )

            wide[
                metric_name
            ] = point
            wide[
                f"{metric_name}_ci_lower"
            ] = lower
            wide[
                f"{metric_name}_ci_upper"
            ] = upper
            wide[
                f"{metric_name}_bootstrap_median"
            ] = median
            wide[
                f"{metric_name}_valid_n"
            ] = valid_n

            if metric_name != (
                "mean_abs_delta_risk5"
            ):
                wide[
                    f"{metric_name}_positive_fraction"
                ] = favorable
                wide[
                    f"{metric_name}_ci_excludes_zero"
                ] = (
                    ci_excludes_zero
                )

            long_rows.append(
                {
                    **base,
                    "metric": (
                        metric_name
                    ),
                    "point_estimate": (
                        point
                    ),
                    "ci_lower": (
                        lower
                    ),
                    "ci_upper": (
                        upper
                    ),
                    "bootstrap_median": (
                        median
                    ),
                    "valid_bootstrap_n": (
                        valid_n
                    ),
                    "positive_fraction": (
                        favorable
                    ),
                    "ci_excludes_zero": (
                        ci_excludes_zero
                        if metric_name
                        != "mean_abs_delta_risk5"
                        else np.nan
                    ),
                }
            )

        wide_rows.append(
            wide
        )

    wide_df = pd.DataFrame(
        wide_rows
    )

    long_df = pd.DataFrame(
        long_rows
    )

    # 在每个Landmark内按point estimate生成便于展示的rank。
    for metric_name, rank_name in [
        (
            "iAUC_loss",
            "iAUC_loss_rank",
        ),
        (
            "IBS_increase",
            "IBS_increase_rank",
        ),
        (
            "uno_c_loss",
            "uno_c_loss_rank",
        ),
        (
            "mean_abs_delta_risk5",
            "risk_change_rank",
        ),
    ]:
        wide_df[
            rank_name
        ] = (
            wide_df.groupby(
                "landmark_index"
            )[
                metric_name
            ]
            .rank(
                method="first",
                ascending=False,
            )
            .astype(
                int
            )
        )

    return (
        wide_df,
        long_df,
    )


# =============================================================================
# 11. 图形
# =============================================================================

def configure_plot_style() -> None:
    plt.rcParams.update(
        {
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "savefig.facecolor": "white",
            "text.color": "black",
            "axes.labelcolor": "black",
            "axes.edgecolor": "black",
            "axes.titlecolor": "black",
            "xtick.color": "black",
            "ytick.color": "black",
            "font.size": 10,
            "axes.titlesize": 11,
            "axes.labelsize": 10,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def save_figure(
    fig: plt.Figure,
    stem: str,
) -> None:
    for extension in [
        "png",
        "pdf",
        "svg",
    ]:
        kwargs = {
            "bbox_inches": "tight",
            "facecolor": "white",
        }

        if extension == "png":
            kwargs[
                "dpi"
            ] = 600

        fig.savefig(
            FIGURE_DIR
            / f"{stem}.{extension}",
            **kwargs,
        )

    plt.close(
        fig
    )


def plot_group_ci_by_landmark(
    wide_df: pd.DataFrame,
    metric_name: str,
    x_label: str,
    stem: str,
) -> None:
    group_order = (
        wide_df.loc[
            wide_df[
                "landmark_index"
            ]
            == 5
        ]
        .sort_values(
            metric_name,
            ascending=True,
        )[
            "group_label"
        ]
        .tolist()
    )

    if len(
        group_order
    ) != EXPECTED_GROUP_N:
        group_order = (
            wide_df[
                [
                    "group_position",
                    "group_label",
                ]
            ]
            .drop_duplicates()
            .sort_values(
                "group_position"
            )[
                "group_label"
            ]
            .tolist()
        )

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(
            18.0,
            6.5,
        ),
        sharey=True,
    )

    for ax, landmark_index in zip(
        axes,
        PRIMARY_LANDMARK_INDICES,
    ):
        landmark_index = int(
            landmark_index
        )

        sub = wide_df.loc[
            wide_df[
                "landmark_index"
            ]
            == landmark_index
        ].copy()

        sub[
            "_group_order"
        ] = sub[
            "group_label"
        ].apply(
            group_order.index
        )

        sub = sub.sort_values(
            "_group_order"
        )

        y = np.arange(
            len(
                sub
            )
        )

        point = sub[
            metric_name
        ].to_numpy(
            dtype=float
        )

        lower = sub[
            f"{metric_name}_ci_lower"
        ].to_numpy(
            dtype=float
        )

        upper = sub[
            f"{metric_name}_ci_upper"
        ].to_numpy(
            dtype=float
        )

        xerr = np.vstack(
            [
                point - lower,
                upper - point,
            ]
        )

        ax.errorbar(
            point,
            y,
            xerr=xerr,
            fmt="o",
            capsize=2.5,
            linewidth=1.2,
        )

        ax.axvline(
            0.0,
            linestyle="--",
            linewidth=1.0,
        )

        ax.set_yticks(
            y
        )

        ax.set_yticklabels(
            sub[
                "group_label"
            ]
        )

        landmark_year = (
            LANDMARK_MONTHS[
                landmark_index
            ]
            / 12.0
        )

        ax.set_title(
            "Baseline"
            if landmark_year == 0
            else f"ART year {landmark_year:.0f}"
        )

        ax.set_xlabel(
            x_label
        )

        ax.grid(
            axis="x",
            linestyle="--",
            alpha=0.20,
        )

    fig.suptitle(
        "Grouped clinical-domain occlusion | patient-level paired bootstrap 95% CI",
        y=1.02,
    )

    fig.tight_layout()

    save_figure(
        fig,
        stem,
    )


def plot_5y_group_ci(
    wide_df: pd.DataFrame,
    metric_name: str,
    x_label: str,
    stem: str,
) -> None:
    sub = wide_df.loc[
        wide_df[
            "landmark_index"
        ]
        == 5
    ].copy()

    sub = sub.sort_values(
        metric_name,
        ascending=True,
    )

    y = np.arange(
        len(
            sub
        )
    )

    point = sub[
        metric_name
    ].to_numpy(
        dtype=float
    )

    lower = sub[
        f"{metric_name}_ci_lower"
    ].to_numpy(
        dtype=float
    )

    upper = sub[
        f"{metric_name}_ci_upper"
    ].to_numpy(
        dtype=float
    )

    xerr = np.vstack(
        [
            point - lower,
            upper - point,
        ]
    )

    fig, ax = plt.subplots(
        figsize=(
            8.8,
            6.2,
        )
    )

    ax.errorbar(
        point,
        y,
        xerr=xerr,
        fmt="o",
        capsize=3,
        linewidth=1.3,
    )

    ax.axvline(
        0.0,
        linestyle="--",
        linewidth=1.0,
    )

    ax.set_yticks(
        y
    )

    ax.set_yticklabels(
        sub[
            "group_label"
        ]
    )

    ax.set_xlabel(
        x_label
    )

    ax.set_title(
        "ART year 5 grouped clinical-domain occlusion | paired bootstrap 95% CI"
    )

    ax.grid(
        axis="x",
        linestyle="--",
        alpha=0.20,
    )

    fig.tight_layout()

    save_figure(
        fig,
        stem,
    )


# =============================================================================
# 12. 主流程
# =============================================================================

def run() -> None:
    total_start = time.time()

    if BOOTSTRAP_REPS < 1:
        raise ValueError(
            "BOOTSTRAP_REPS必须为正整数。"
        )

    if BOOTSTRAP_SAVE_EVERY < 1:
        raise ValueError(
            "BOOTSTRAP_SAVE_EVERY必须>=1。"
        )

    configure_plot_style()

    print(
        "=" * 122
    )
    print(
        "Step 12C-2：Grouped Clinical-Domain Occlusion 患者级paired bootstrap 95% CI"
    )
    print(
        "=" * 122
    )
    print(
        "Bootstrap次数：",
        BOOTSTRAP_REPS,
    )
    print(
        "Bootstrap单位：development patient"
    )
    print(
        "同一bootstrap患者样本：同时用于全部4个主Landmark和全部10个临床域"
    )
    print(
        "IPCW reference：各Landmark完整原始OOF风险集"
    )
    print(
        "模型前向：不重新运行，直接读取Step12C保存预测"
    )
    print(
        "锁定测试集：未读取"
    )
    print(
        "输出目录：",
        OUTPUT_DIR,
    )
    print(
        "=" * 122
    )

    common = load_common_arrays()

    (
        metrics_df,
        group_definition_df,
    ) = load_group_tables()

    print(
        "Step12C group数：",
        len(
            group_definition_df
        ),
    )

    print(
        "主Landmark condition数：",
        len(
            metrics_df
        ),
    )

    print(
        "\n固定group顺序："
    )

    print(
        group_definition_df[
            [
                "group_key",
                "group_label",
                "dynamic_channel_n",
                "static_channel_n",
                "age_included",
            ]
        ].to_string(
            index=False
        )
    )

    print(
        "\n开始读取Step12C prediction checkpoints..."
    )

    store = PredictionStore(
        common=common,
        metrics_df=metrics_df,
        group_definition_df=(
            group_definition_df
        ),
    )

    print(
        "Prediction checkpoints读取完成。"
    )

    print(
        "\n开始point-estimate重建审计..."
    )

    audit_df = audit_point_estimates(
        store=store,
        metrics_df=metrics_df,
    )

    print(
        "Point-estimate审计通过 | "
        "worst abs difference="
        f"{audit_df['max_abs_metric_difference'].max():.3e}"
    )

    (
        bootstrap_values,
        completed_replicates,
        checkpoint_file,
    ) = initialize_bootstrap_storage(
        metrics_df=metrics_df,
        group_definition_df=(
            group_definition_df
        ),
    )

    if int(
        completed_replicates.sum()
    ) < BOOTSTRAP_REPS:
        run_bootstrap(
            common=common,
            store=store,
            metrics_df=metrics_df,
            bootstrap_values=(
                bootstrap_values
            ),
            completed_replicates=(
                completed_replicates
            ),
            checkpoint_file=(
                checkpoint_file
            ),
        )

    if int(
        completed_replicates.sum()
    ) != BOOTSTRAP_REPS:
        raise RuntimeError(
            "Bootstrap未全部完成。"
        )

    wide_df, long_df = summarize_bootstrap(
        metrics_df=metrics_df,
        bootstrap_values=(
            bootstrap_values
        ),
    )

    wide_df.to_csv(
        OUTPUT_DIR
        / "grouped_occlusion_paired_bootstrap_CI_all.csv",
        index=False,
        encoding="utf-8-sig",
    )

    long_df.to_csv(
        OUTPUT_DIR
        / "grouped_occlusion_paired_bootstrap_CI_long.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # 便于论文主结果直接读取：按Landmark、iAUC loss排序。
    ranked_df = wide_df.sort_values(
        [
            "landmark_index",
            "iAUC_loss",
        ],
        ascending=[
            True,
            False,
        ],
    ).copy()

    ranked_df.to_csv(
        OUTPUT_DIR
        / "grouped_occlusion_paired_bootstrap_CI_ranked_by_iAUC.csv",
        index=False,
        encoding="utf-8-sig",
    )

    plot_group_ci_by_landmark(
        wide_df=wide_df,
        metric_name=(
            "iAUC_loss"
        ),
        x_label=(
            "iAUC loss after group occlusion"
        ),
        stem=(
            "Figure12C2A_group_iAUC_loss_bootstrap_CI"
        ),
    )

    plot_group_ci_by_landmark(
        wide_df=wide_df,
        metric_name=(
            "IBS_increase"
        ),
        x_label=(
            "IBS increase after group occlusion"
        ),
        stem=(
            "Figure12C2B_group_IBS_increase_bootstrap_CI"
        ),
    )

    plot_group_ci_by_landmark(
        wide_df=wide_df,
        metric_name=(
            "uno_c_loss"
        ),
        x_label=(
            "Uno C-index loss after group occlusion"
        ),
        stem=(
            "Figure12C2C_group_UnoC_loss_bootstrap_CI"
        ),
    )

    plot_5y_group_ci(
        wide_df=wide_df,
        metric_name=(
            "iAUC_loss"
        ),
        x_label=(
            "iAUC loss after group occlusion"
        ),
        stem=(
            "Figure12C2D_ARTyear5_group_iAUC_loss_bootstrap_CI"
        ),
    )

    metadata = {
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "bootstrap_reps": int(
            BOOTSTRAP_REPS
        ),
        "bootstrap_unit": (
            "development patient"
        ),
        "bootstrap_pairing": (
            "same sampled patients across all primary landmarks and all groups"
        ),
        "ipcw_reference": (
            "complete original OOF risk set within each landmark"
        ),
        "primary_landmark_months": (
            PRIMARY_LANDMARK_MONTHS.tolist()
        ),
        "group_keys": (
            group_definition_df[
                "group_key"
            ].astype(
                str
            ).tolist()
        ),
        "metrics": (
            BOOTSTRAP_METRICS
        ),
        "ci_method": (
            "2.5th and 97.5th percentile of patient-level paired bootstrap distribution"
        ),
        "positive_direction": {
            "uno_c_loss": (
                "positive = group occlusion worsens performance"
            ),
            "iAUC_loss": (
                "positive = group occlusion worsens performance"
            ),
            "IBS_increase": (
                "positive = group occlusion worsens performance"
            ),
        },
        "interpretation_scope": (
            "post-hoc grouped input reference occlusion of frozen LSTM; not retrained ablation; group effects not assumed additive"
        ),
        "locked_test_used": False,
        "elapsed_seconds": float(
            time.time()
            - total_start
        ),
        "output_dir": str(
            OUTPUT_DIR
        ),
    }

    save_json(
        metadata,
        OUTPUT_DIR
        / "step12c2_bootstrap_summary.json",
    )

    print(
        "\n"
        + "=" * 122
    )
    print(
        "Step 12C-2 paired bootstrap完成"
    )
    print(
        "=" * 122
    )
    print(
        "Bootstrap次数：",
        BOOTSTRAP_REPS,
    )
    print(
        "锁定测试集：未读取"
    )
    print(
        "总耗时：",
        format_duration(
            time.time()
            - total_start
        ),
    )

    print(
        "\n各Landmark按iAUC loss排序的临床域（含95%CI）："
    )

    for landmark_index in (
        PRIMARY_LANDMARK_INDICES
    ):
        landmark_index = int(
            landmark_index
        )

        sub = wide_df.loc[
            wide_df[
                "landmark_index"
            ]
            == landmark_index
        ].sort_values(
            "iAUC_loss",
            ascending=False,
        )

        print(
            "\n"
            + "-" * 112
        )
        print(
            "Landmark "
            f"{LANDMARK_MONTHS[landmark_index] / 12:.0f}年"
        )
        print(
            sub[
                [
                    "iAUC_loss_rank",
                    "group_label",
                    "iAUC_loss",
                    "iAUC_loss_ci_lower",
                    "iAUC_loss_ci_upper",
                    "iAUC_loss_positive_fraction",
                    "IBS_increase",
                    "IBS_increase_ci_lower",
                    "IBS_increase_ci_upper",
                    "uno_c_loss",
                    "uno_c_loss_ci_lower",
                    "uno_c_loss_ci_upper",
                    "mean_abs_delta_risk5",
                ]
            ].to_string(
                index=False
            )
        )

    # 单独打印ART相关组，便于核查研究重点。
    art_sub = wide_df.loc[
        wide_df[
            "group_key"
        ].isin(
            [
                "current_art_regimen",
                "cumulative_art_exposure",
            ]
        )
    ].sort_values(
        [
            "landmark_index",
            "group_position",
        ]
    )

    print(
        "\nCurrent ART vs Cumulative ART（含95%CI）："
    )
    print(
        art_sub[
            [
                "landmark_year",
                "group_label",
                "iAUC_loss",
                "iAUC_loss_ci_lower",
                "iAUC_loss_ci_upper",
                "IBS_increase",
                "IBS_increase_ci_lower",
                "IBS_increase_ci_upper",
                "uno_c_loss",
                "uno_c_loss_ci_lower",
                "uno_c_loss_ci_upper",
                "mean_abs_delta_risk5",
            ]
        ].to_string(
            index=False
        )
    )

    print(
        "\n最重要结果文件："
    )

    for filename in [
        "grouped_occlusion_paired_bootstrap_CI_all.csv",
        "grouped_occlusion_paired_bootstrap_CI_ranked_by_iAUC.csv",
        "grouped_occlusion_paired_bootstrap_CI_long.csv",
        "step12c_point_estimate_reconstruction_audit.csv",
        "step12c2_bootstrap_summary.json",
    ]:
        print(
            " -",
            OUTPUT_DIR
            / filename,
        )

    print(
        "=" * 122
    )


if __name__ == "__main__":
    run()


Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

Source-reference placeholder; historical cell intentionally not distributed.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Step 13A：LSTM-v2 纵向信息增量价值分析——Baseline-only / Current-only重新训练OOF

研究目的
--------
在模型解释阶段冻结之后，单独验证“完整纵向历史是否提供增量预测价值”。

本脚本只训练一个history variant，由环境变量CKD_HISTORY_MODE控制：
- baseline_only：
    每个Landmark仅保留该患者Landmark前最早可用的动态临床状态；
- current_only：
    每个Landmark仅保留该患者Landmark前最近可用的动态临床状态；
- full_history：
    仅用于技术复现，不建议重新训练；正式Full-history结果直接使用既有Step10E。

为了避免Current-only仍通过工程特征携带既往历史：
- baseline_only/current_only仅保留前32个“当前临床状态”动态通道：
  15个实验室值 + 6个状态 + 3类代谢药物 + 8个current ART；
- 8个累计ART通道、15个observed、15个time-since-last、15个delta通道全部置0；
- static baseline、Age at landmark、landmark context保持与Full-history相同；
- 风险集、五折、标签、未来5年预测窗口、训练损失、超参数、早停、snapshot及seed ensemble
  均严格沿用Step10E冻结方案。

方法学边界
----------
1. 这是“重新训练后的ablation / longitudinal incremental value”分析，不属于模型解释。
2. 不重新调参：Baseline-only与Current-only固定使用Full-history冻结Trial参数，
   以保持模型容量和优化预算一致。
3. Current-only定义为“Landmark前最近可用临床状态”，因此若Landmark恰好没有有效时间行，
   使用最近的既往有效时间行；脚本会输出该时间滞后分布。
4. Baseline-only定义为“Landmark前最早可用临床状态”。脚本会审计其是否为0月时间行。
5. 锁定内部测试集不读取。
"""

from __future__ import annotations

import gc
import json
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import (
    DataLoader,
    Dataset,
    WeightedRandomSampler,
)

try:
    from sksurv.metrics import (
        concordance_index_ipcw,
        cumulative_dynamic_auc,
        integrated_brier_score,
    )
    from sksurv.util import Surv
except ImportError as exc:
    raise ImportError(
        "需要scikit-survival。请在当前环境运行：\n"
        "conda install -c conda-forge scikit-survival"
    ) from exc


# =============================================================================
# 1. 固定研究配置
# =============================================================================

PROJECT_DIR = Path(
    os.getenv("CKD_LSTM_PROJECT_DIR", "__CKD_WORKDIR__")
)

STEP1_DIR = PROJECT_DIR / "rolling_5y_step1_new_split"
STEP2_DIR = PROJECT_DIR / "rolling_5y_step2_folds"
STEP3_DIR = PROJECT_DIR / "rolling_5y_step3_raw_features"
STEP4_DIR = PROJECT_DIR / "rolling_5y_step4_preprocessed"
STEP6_DIR = PROJECT_DIR / "rolling_5y_step6_super_landmark_data"

EXPECTED_TOTAL_N = 31911
EXPECTED_DEVELOPMENT_N = 22337
EXPECTED_HISTORY_STEP_N = 11
EXPECTED_BASE_FEATURE_N = 57
EXPECTED_STATIC_N = 16
EXPECTED_BASE_DYNAMIC_N = 40
EXPECTED_LAB_N = 15
EXPECTED_EXTRA_DYNAMIC_N = EXPECTED_LAB_N * 3
EXPECTED_ENHANCED_DYNAMIC_N = (
    EXPECTED_BASE_DYNAMIC_N + EXPECTED_EXTRA_DYNAMIC_N
)
EXPECTED_LANDMARK_N = 6
EXPECTED_FUTURE_INTERVAL_N = 10
EXPECTED_DEVELOPMENT_VALID_ORIGIN_N = 100122
N_SPLITS = 5

LANDMARK_MONTHS = np.asarray(
    [0, 12, 24, 36, 48, 60],
    dtype=np.int32,
)
LANDMARK_BINS = (LANDMARK_MONTHS // 6).astype(np.int64)
FUTURE_END_MONTHS = np.arange(6, 61, 6, dtype=np.float64)
METRIC_TIMES = np.asarray(
    [6, 12, 18, 24, 30, 36, 42, 48, 54, 59.999],
    dtype=np.float64,
)
TIME_STEP_YEARS = (
    np.arange(EXPECTED_HISTORY_STEP_N, dtype=np.float32) * 0.5
)
EPS = 1e-7

# =============================================================================
# Step13A history-variant固定定义
# =============================================================================

HISTORY_MODE = os.getenv(
    "CKD_HISTORY_MODE",
    "current_only",
).strip().lower()

ALLOWED_HISTORY_MODES = {
    "baseline_only",
    "current_only",
    "full_history",
}

if HISTORY_MODE not in ALLOWED_HISTORY_MODES:
    raise ValueError(
        "CKD_HISTORY_MODE必须为baseline_only、current_only或full_history。"
    )


USE_AMP = True
REQUIRE_CUDA = True
NUM_WORKERS = 0
MAX_EPOCHS = 60
MIN_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MIN_DELTA = 1e-5
GRADIENT_CLIP_NORM = 1.0
TOP_SNAPSHOT_N = 3
LR_REDUCE_FACTOR = 0.5
LR_REDUCE_PATIENCE = 3
MIN_LEARNING_RATE = 1e-6

LAB_FEATURES = [
    "HIVRNA_log10",
    "CD4",
    "CD8",
    "Urea",
    "WBC",
    "PLT",
    "HB",
    "TC",
    "TG",
    "HDL",
    "LDL",
    "GLU",
    "ALT",
    "AST",
    "eGFR",
]

PERSISTENT_STATUS_FEATURES = [
    "CVD_status",
    "diabetes_status",
    "hypertension_status",
    "hypercholesterolemia_status",
    "HBV_status",
    "HCV_status",
]

METABOLIC_MED_FEATURES = [
    "antidiabetic_med",
    "antihypertensive_med",
    "antilipid_med",
]

CURRENT_ART_FEATURES = [
    "current_TDF_NNRTI_3TC_FTC",
    "current_TDF_PI_3TC_FTC",
    "current_nonTDF_PI",
    "current_BIC_FTC_TAF",
    "current_EVGc_FTC_TAF",
    "current_TDF_INSTI_3TC_FTC",
    "current_nonTDF_DTG",
    "current_nonTDF_traditional_NNRTI",
]

CUMULATIVE_ART_FEATURES = [
    "TDF_NNRTI_3TC_FTC_cum_month",
    "TDF_PI_3TC_FTC_cum_month",
    "nonTDF_PI_cum_month",
    "BIC_FTC_TAF_cum_month",
    "EVGc_FTC_TAF_cum_month",
    "TDF_INSTI_3TC_FTC_cum_month",
    "nonTDF_DTG_cum_month",
    "nonTDF_traditional_NNRTI_cum_month",
]

DYNAMIC_FEATURES = (
    LAB_FEATURES
    + PERSISTENT_STATUS_FEATURES
    + METABOLIC_MED_FEATURES
    + CURRENT_ART_FEATURES
    + CUMULATIVE_ART_FEATURES
)

# -----------------------------------------------------------------------------
# Current-only / Baseline-only允许保留的动态“当前状态”通道
#
# 必须在LAB/状态/药物/current ART等特征定义之后计算。
# -----------------------------------------------------------------------------

CURRENT_STATE_DYNAMIC_FEATURES = (
    LAB_FEATURES
    + PERSISTENT_STATUS_FEATURES
    + METABOLIC_MED_FEATURES
    + CURRENT_ART_FEATURES
)

CURRENT_STATE_DYNAMIC_N = len(
    CURRENT_STATE_DYNAMIC_FEATURES
)

if CURRENT_STATE_DYNAMIC_N != 32:
    raise RuntimeError(
        f"Current-state动态通道数={CURRENT_STATE_DYNAMIC_N}，预期32。"
    )

if DYNAMIC_FEATURES[
    :CURRENT_STATE_DYNAMIC_N
] != CURRENT_STATE_DYNAMIC_FEATURES:
    raise RuntimeError(
        "Current-state 32通道并非DYNAMIC_FEATURES前32项；"
        "不能安全使用位置切片构造current-only/baseline-only。"
    )

if len(DYNAMIC_FEATURES) != EXPECTED_BASE_DYNAMIC_N:
    raise RuntimeError(
        f"DYNAMIC_FEATURES数={len(DYNAMIC_FEATURES)}，"
        f"预期{EXPECTED_BASE_DYNAMIC_N}。"
    )



# =============================================================================
# 2. 通用函数
# =============================================================================

def set_random_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def format_duration(seconds: float) -> str:
    seconds = max(0, int(round(float(seconds))))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    if hours:
        return f"{hours}小时{minutes:02d}分{seconds:02d}秒"
    if minutes:
        return f"{minutes}分{seconds:02d}秒"
    return f"{seconds}秒"


def save_json(value: Any, path: Path) -> None:
    path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )


def require_files(paths: list[Path]) -> None:
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "以下必要文件不存在：\n" + "\n".join(missing)
        )


def build_survival_array(
    event: np.ndarray,
    time_month: np.ndarray,
) -> np.ndarray:
    event = np.asarray(event, dtype=bool)
    time_month = np.asarray(time_month, dtype=np.float64)
    if event.shape != time_month.shape:
        raise ValueError("事件与时间形状不一致。")
    if not np.isfinite(time_month).all() or np.any(time_month <= 0):
        raise ValueError("生存时间必须为有限正数。")
    return Surv.from_arrays(event=event, time=time_month)


def autocast_context(device: torch.device, enabled: bool):
    try:
        return torch.amp.autocast(
            device_type=device.type,
            enabled=enabled,
        )
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)


def make_grad_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=enabled,
        )
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


# =============================================================================
# 3. 共同数据
# =============================================================================

@dataclass
class CommonData:
    development_idx: np.ndarray
    development_fold_id: np.ndarray
    sequence_row_mask: np.ndarray
    prediction_origin_mask: np.ndarray
    future_event_matrix: np.ndarray
    future_at_risk_mask: np.ndarray
    continuous_raw: np.ndarray
    continuous_vars: list[str]
    age_continuous_index: int
    lab_continuous_indices: np.ndarray
    long_row_index_map: np.ndarray
    fold_id_long: np.ndarray
    landmark_month_long: np.ndarray
    analysis_time_month: np.ndarray
    event_within_60m: np.ndarray
    y_long_all: np.ndarray


def load_common_data() -> CommonData:
    required = [
        STEP1_DIR / "development_idx.npy",
        STEP1_DIR / "prediction_origin_mask.npy",
        STEP1_DIR / "future_event_matrix.npy",
        STEP1_DIR / "future_at_risk_mask.npy",
        STEP1_DIR / "landmark_months.npy",
        STEP1_DIR / "landmark_bins.npy",
        STEP2_DIR / "development_fold_id.npy",
        STEP3_DIR / "continuous_raw_0_60.npy",
        STEP3_DIR / "feature_groups.json",
        STEP4_DIR / "development_idx.npy",
        STEP4_DIR / "development_fold_id.npy",
        STEP4_DIR / "sequence_row_mask_development.npy",
        STEP6_DIR / "development_long_row_index_map.npy",
        STEP6_DIR / "development_fold_id_long.npy",
        STEP6_DIR / "development_landmark_month_long.npy",
        STEP6_DIR / "development_analysis_time_month.npy",
        STEP6_DIR / "development_event_within_60m.npy",
    ]
    for fold_id in range(N_SPLITS):
        fold_dir = STEP4_DIR / f"fold_{fold_id}"
        required.extend(
            [
                fold_dir / "X_development.npy",
                fold_dir / "preprocessor.joblib",
                fold_dir / "feature_names.csv",
            ]
        )
    require_files(required)

    development_idx_step1 = np.load(
        STEP1_DIR / "development_idx.npy"
    ).astype(np.int32)
    development_idx = np.load(
        STEP4_DIR / "development_idx.npy"
    ).astype(np.int32)
    fold_step2 = np.load(
        STEP2_DIR / "development_fold_id.npy"
    ).astype(np.int8)
    development_fold_id = np.load(
        STEP4_DIR / "development_fold_id.npy"
    ).astype(np.int8)

    if not np.array_equal(development_idx_step1, development_idx):
        raise ValueError("Step 1与Step 4开发集顺序不一致。")
    if not np.array_equal(fold_step2, development_fold_id):
        raise ValueError("Step 2与Step 4固定五折不一致。")

    sequence_row_mask = np.load(
        STEP4_DIR / "sequence_row_mask_development.npy"
    ).astype(bool)

    prediction_origin_mask_all = np.load(
        STEP1_DIR / "prediction_origin_mask.npy"
    ).astype(bool)
    future_event_all = np.load(
        STEP1_DIR / "future_event_matrix.npy"
    ).astype(np.float32)
    future_at_risk_all = np.load(
        STEP1_DIR / "future_at_risk_mask.npy"
    ).astype(bool)

    landmark_months_file = np.load(
        STEP1_DIR / "landmark_months.npy"
    ).astype(np.int32)
    landmark_bins_file = np.load(
        STEP1_DIR / "landmark_bins.npy"
    ).astype(np.int64)

    if not np.array_equal(landmark_months_file, LANDMARK_MONTHS):
        raise ValueError("Landmark月份配置不一致。")
    if not np.array_equal(landmark_bins_file, LANDMARK_BINS):
        raise ValueError("Landmark时间行配置不一致。")

    continuous_raw = np.load(
        STEP3_DIR / "continuous_raw_0_60.npy",
        mmap_mode="r",
    )
    feature_groups = json.loads(
        (STEP3_DIR / "feature_groups.json").read_text(
            encoding="utf-8"
        )
    )
    continuous_vars = list(feature_groups["continuous_vars"])

    if "Age" not in continuous_vars:
        raise ValueError("连续变量中缺少Age。")
    missing_labs = sorted(set(LAB_FEATURES) - set(continuous_vars))
    if missing_labs:
        raise ValueError(f"连续变量中缺少实验室指标：{missing_labs}")

    age_continuous_index = continuous_vars.index("Age")
    lab_continuous_indices = np.asarray(
        [continuous_vars.index(name) for name in LAB_FEATURES],
        dtype=np.int64,
    )

    long_row_index_map = np.load(
        STEP6_DIR / "development_long_row_index_map.npy"
    ).astype(np.int32)
    fold_id_long = np.load(
        STEP6_DIR / "development_fold_id_long.npy"
    ).astype(np.int8)
    landmark_month_long = np.load(
        STEP6_DIR / "development_landmark_month_long.npy"
    ).astype(np.int32)
    analysis_time_month = np.load(
        STEP6_DIR / "development_analysis_time_month.npy"
    ).astype(np.float64)
    event_within_60m = np.load(
        STEP6_DIR / "development_event_within_60m.npy"
    ).astype(bool)

    prediction_origin_mask = prediction_origin_mask_all[development_idx]
    future_event_matrix = future_event_all[development_idx]
    future_at_risk_mask = future_at_risk_all[development_idx]

    expected_checks = [
        (
            development_idx.shape,
            (EXPECTED_DEVELOPMENT_N,),
            "development_idx",
        ),
        (
            development_fold_id.shape,
            (EXPECTED_DEVELOPMENT_N,),
            "development_fold_id",
        ),
        (
            sequence_row_mask.shape,
            (EXPECTED_DEVELOPMENT_N, EXPECTED_HISTORY_STEP_N),
            "sequence_row_mask",
        ),
        (
            prediction_origin_mask.shape,
            (EXPECTED_DEVELOPMENT_N, EXPECTED_LANDMARK_N),
            "prediction_origin_mask",
        ),
        (
            future_event_matrix.shape,
            (
                EXPECTED_DEVELOPMENT_N,
                EXPECTED_LANDMARK_N,
                EXPECTED_FUTURE_INTERVAL_N,
            ),
            "future_event_matrix",
        ),
        (
            future_at_risk_mask.shape,
            (
                EXPECTED_DEVELOPMENT_N,
                EXPECTED_LANDMARK_N,
                EXPECTED_FUTURE_INTERVAL_N,
            ),
            "future_at_risk_mask",
        ),
        (
            long_row_index_map.shape,
            (EXPECTED_DEVELOPMENT_N, EXPECTED_LANDMARK_N),
            "long_row_index_map",
        ),
    ]
    for actual, expected, name in expected_checks:
        if actual != expected:
            raise ValueError(f"{name}形状={actual}，预期={expected}。")

    if int(prediction_origin_mask.sum()) != EXPECTED_DEVELOPMENT_VALID_ORIGIN_N:
        raise ValueError("有效患者-Landmark记录数不是100122。")
    if not np.array_equal(
        prediction_origin_mask,
        long_row_index_map >= 0,
    ):
        raise ValueError("标签有效起点与长格式映射不一致。")
    if np.any(future_event_matrix.astype(bool) & ~future_at_risk_mask):
        raise ValueError("存在事件标签为1但风险掩码为0。")
    if np.any(future_event_matrix.sum(axis=2) > 1):
        raise ValueError("同一患者-Landmark存在多个事件区间。")
    if not np.isin(development_fold_id, np.arange(N_SPLITS)).all():
        raise ValueError("开发集固定折编号必须为0～4。")
    if len(analysis_time_month) != EXPECTED_DEVELOPMENT_VALID_ORIGIN_N:
        raise ValueError("Step 6生存时间记录数错误。")

    y_long_all = build_survival_array(
        event_within_60m,
        analysis_time_month,
    )

    return CommonData(
        development_idx=development_idx,
        development_fold_id=development_fold_id,
        sequence_row_mask=sequence_row_mask,
        prediction_origin_mask=prediction_origin_mask,
        future_event_matrix=future_event_matrix,
        future_at_risk_mask=future_at_risk_mask,
        continuous_raw=continuous_raw,
        continuous_vars=continuous_vars,
        age_continuous_index=age_continuous_index,
        lab_continuous_indices=lab_continuous_indices,
        long_row_index_map=long_row_index_map,
        fold_id_long=fold_id_long,
        landmark_month_long=landmark_month_long,
        analysis_time_month=analysis_time_month,
        event_within_60m=event_within_60m,
        y_long_all=y_long_all,
    )


# =============================================================================
# 4. 强化纵向特征
# =============================================================================

def build_enhanced_dynamic_array(
    base_dynamic: np.ndarray,
    lab_raw_development: np.ndarray,
    sequence_row_mask: np.ndarray,
) -> tuple[np.ndarray, list[str]]:
    """
    追加：
    - 15项实验室是否真实测量；
    - 15项实验室距上次真实测量时间（除以5年）；
    - 15项实验室与上次真实测量的标准化变化量。
    """
    base_dynamic = np.asarray(base_dynamic, dtype=np.float32)
    lab_raw_development = np.asarray(
        lab_raw_development,
        dtype=np.float32,
    )
    sequence_row_mask = np.asarray(sequence_row_mask, dtype=bool)

    expected_base_shape = (
        EXPECTED_DEVELOPMENT_N,
        EXPECTED_HISTORY_STEP_N,
        EXPECTED_BASE_DYNAMIC_N,
    )
    expected_lab_shape = (
        EXPECTED_DEVELOPMENT_N,
        EXPECTED_HISTORY_STEP_N,
        EXPECTED_LAB_N,
    )
    if base_dynamic.shape != expected_base_shape:
        raise ValueError(
            f"基础动态特征形状={base_dynamic.shape}，"
            f"预期={expected_base_shape}。"
        )
    if lab_raw_development.shape != expected_lab_shape:
        raise ValueError(
            f"原始实验室形状={lab_raw_development.shape}，"
            f"预期={expected_lab_shape}。"
        )

    lab_observed = (
        np.isfinite(lab_raw_development)
        & sequence_row_mask[:, :, None]
    )
    lab_observed_float = lab_observed.astype(np.float32)

    time_since = np.zeros_like(lab_observed_float, dtype=np.float32)
    lab_delta = np.zeros_like(lab_observed_float, dtype=np.float32)

    standardized_labs = base_dynamic[:, :, :EXPECTED_LAB_N]
    last_seen_step = np.full(
        (EXPECTED_DEVELOPMENT_N, EXPECTED_LAB_N),
        -1,
        dtype=np.int16,
    )
    last_seen_value = np.zeros(
        (EXPECTED_DEVELOPMENT_N, EXPECTED_LAB_N),
        dtype=np.float32,
    )
    has_seen = np.zeros(
        (EXPECTED_DEVELOPMENT_N, EXPECTED_LAB_N),
        dtype=bool,
    )

    for step_index in range(EXPECTED_HISTORY_STEP_N):
        active_row = sequence_row_mask[:, step_index][:, None]
        observed_now = lab_observed[:, step_index, :]
        current_value = standardized_labs[:, step_index, :]

        elapsed_years = (
            step_index - last_seen_step
        ).astype(np.float32) * 0.5
        elapsed_years = np.clip(elapsed_years, 0.0, 5.0)
        elapsed_years[~has_seen] = min(
            (step_index + 1) * 0.5,
            5.0,
        )
        time_since[:, step_index, :] = np.where(
            active_row,
            np.where(observed_now, 0.0, elapsed_years / 5.0),
            0.0,
        )

        delta_now = current_value - last_seen_value
        lab_delta[:, step_index, :] = np.where(
            observed_now & has_seen,
            delta_now,
            0.0,
        )

        last_seen_value = np.where(
            observed_now,
            current_value,
            last_seen_value,
        )
        last_seen_step = np.where(
            observed_now,
            step_index,
            last_seen_step,
        )
        has_seen |= observed_now

    enhanced = np.concatenate(
        [
            base_dynamic,
            lab_observed_float,
            time_since,
            lab_delta,
        ],
        axis=2,
    ).astype(np.float32)

    enhanced[~sequence_row_mask] = 0.0

    if enhanced.shape != (
        EXPECTED_DEVELOPMENT_N,
        EXPECTED_HISTORY_STEP_N,
        EXPECTED_ENHANCED_DYNAMIC_N,
    ):
        raise ValueError("强化动态特征形状错误。")
    if not np.isfinite(enhanced).all():
        raise ValueError("强化动态特征存在NaN或无穷值。")

    names = (
        DYNAMIC_FEATURES
        + [f"{name}_observed" for name in LAB_FEATURES]
        + [f"{name}_time_since_last" for name in LAB_FEATURES]
        + [f"{name}_delta_last_observed" for name in LAB_FEATURES]
    )
    return enhanced, names


# =============================================================================
# 5. Dataset与Fold数据
# =============================================================================

class EnhancedPatientLandmarkDataset(Dataset):
    def __init__(
        self,
        sample_pairs: np.ndarray,
        enhanced_dynamic: np.ndarray,
        sequence_mask: np.ndarray,
        static_baseline: np.ndarray,
        baseline_age_raw: np.ndarray,
        event_matrix: np.ndarray,
        at_risk_mask: np.ndarray,
        age_mean: float,
        age_scale: float,
        history_mode: str,
    ):
        self.sample_pairs = np.asarray(sample_pairs, dtype=np.int32)
        self.enhanced_dynamic = enhanced_dynamic
        self.sequence_mask = sequence_mask
        self.static_baseline = static_baseline
        self.baseline_age_raw = baseline_age_raw
        self.event_matrix = event_matrix
        self.at_risk_mask = at_risk_mask
        self.age_mean = float(age_mean)
        self.age_scale = float(age_scale)
        self.history_mode = str(history_mode)

        if self.history_mode not in ALLOWED_HISTORY_MODES:
            raise ValueError(
                f"非法history_mode：{self.history_mode}"
            )

    def __len__(self) -> int:
        return len(self.sample_pairs)

    def __getitem__(self, index: int) -> dict[str, np.ndarray]:
        patient_local, landmark_index = self.sample_pairs[index]
        landmark_month = float(LANDMARK_MONTHS[landmark_index])

        age_raw = (
            float(self.baseline_age_raw[patient_local])
            + landmark_month / 12.0
        )
        age_standardized = (
            age_raw - self.age_mean
        ) / self.age_scale

        original_dynamic = np.asarray(
            self.enhanced_dynamic[patient_local],
            dtype=np.float32,
        )
        original_mask = np.asarray(
            self.sequence_mask[patient_local],
            dtype=np.bool_,
        )

        landmark_bin = int(
            LANDMARK_BINS[
                landmark_index
            ]
        )

        if self.history_mode == "full_history":
            dynamic_sequence = original_dynamic
            row_mask = original_mask
        else:
            eligible_steps = np.where(
                original_mask
                & (
                    np.arange(
                        EXPECTED_HISTORY_STEP_N
                    )
                    <= landmark_bin
                )
            )[0]

            if len(eligible_steps) == 0:
                raise ValueError(
                    "部分患者在Landmark前没有可用于"
                    f"{self.history_mode}的有效动态时间行。"
                )

            if self.history_mode == "baseline_only":
                selected_step = int(
                    eligible_steps[
                        0
                    ]
                )
            elif self.history_mode == "current_only":
                selected_step = int(
                    eligible_steps[
                        -1
                    ]
                )
            else:
                raise RuntimeError(
                    "未知history_mode。"
                )

            dynamic_sequence = np.zeros_like(
                original_dynamic,
                dtype=np.float32,
            )
            row_mask = np.zeros_like(
                original_mask,
                dtype=np.bool_,
            )

            # 只保留当前状态32通道，显式删除：
            # cumulative ART + observed/time-since/delta。
            dynamic_sequence[
                selected_step,
                :CURRENT_STATE_DYNAMIC_N,
            ] = original_dynamic[
                selected_step,
                :CURRENT_STATE_DYNAMIC_N,
            ]

            row_mask[
                selected_step
            ] = True

        return {
            "dynamic_sequence": np.asarray(
                dynamic_sequence,
                dtype=np.float32,
            ),
            "row_mask": np.asarray(
                row_mask,
                dtype=np.bool_,
            ),
            "static_baseline": np.asarray(
                self.static_baseline[patient_local],
                dtype=np.float32,
            ),
            "age_at_landmark": np.float32(age_standardized),
            "landmark_normalized": np.float32(landmark_month / 60.0),
            "landmark_bin": np.int64(LANDMARK_BINS[landmark_index]),
            "event_target": np.asarray(
                self.event_matrix[patient_local, landmark_index],
                dtype=np.float32,
            ),
            "at_risk_mask": np.asarray(
                self.at_risk_mask[patient_local, landmark_index],
                dtype=np.float32,
            ),
            "patient_local": np.int64(patient_local),
            "landmark_index": np.int64(landmark_index),
        }


def prepare_fold_data(
    common: CommonData,
    fold_id: int,
) -> dict[str, Any]:
    fold_dir = STEP4_DIR / f"fold_{fold_id}"

    x_development = np.load(
        fold_dir / "X_development.npy",
        mmap_mode="r",
    )
    feature_names = (
        pd.read_csv(
            fold_dir / "feature_names.csv",
            encoding="utf-8-sig",
        )["feature_name"]
        .astype(str)
        .tolist()
    )
    preprocessor = joblib.load(
        fold_dir / "preprocessor.joblib"
    )

    if x_development.shape != (
        EXPECTED_DEVELOPMENT_N,
        EXPECTED_HISTORY_STEP_N,
        EXPECTED_BASE_FEATURE_N,
    ):
        raise ValueError(f"第{fold_id}折预处理张量形状错误。")
    if len(feature_names) != EXPECTED_BASE_FEATURE_N:
        raise ValueError(f"第{fold_id}折预处理特征数不是57。")

    x_array = np.asarray(x_development)
    if not np.isfinite(x_array).all():
        raise ValueError(f"第{fold_id}折预处理张量存在非法值。")
    if not np.all(x_array[~common.sequence_row_mask] == 0.0):
        raise ValueError(f"第{fold_id}折空缺时间行不是全0。")

    feature_to_index = {
        name: index for index, name in enumerate(feature_names)
    }

    onehot_static = [
        name
        for name in feature_names
        if (
            name.startswith("Sex_")
            or name.startswith("Marriage_")
            or name.startswith("Course_")
            or name.startswith("WHOstage_")
        )
    ]
    static_features = ["BMI", "Oppinfection", *onehot_static]
    if len(static_features) != EXPECTED_STATIC_N:
        raise ValueError(
            f"第{fold_id}折静态特征数={len(static_features)}，应为16。"
        )

    required_names = {"Age", *static_features, *DYNAMIC_FEATURES}
    missing = sorted(required_names - set(feature_names))
    if missing:
        raise ValueError(f"第{fold_id}折缺少特征：{missing}")

    static_indices = np.asarray(
        [feature_to_index[name] for name in static_features],
        dtype=np.int64,
    )
    dynamic_indices = np.asarray(
        [feature_to_index[name] for name in DYNAMIC_FEATURES],
        dtype=np.int64,
    )

    first_observed_step = np.argmax(
        common.sequence_row_mask,
        axis=1,
    ).astype(np.int64)
    if (~common.sequence_row_mask.any(axis=1)).any():
        raise ValueError("部分开发集患者没有任何历史时间行。")

    patient_local = np.arange(
        EXPECTED_DEVELOPMENT_N,
        dtype=np.int64,
    )
    static_baseline = np.asarray(
        x_development[
            patient_local,
            first_observed_step,
            :,
        ][:, static_indices],
        dtype=np.float32,
    )

    age_first = np.asarray(
        common.continuous_raw[
            common.development_idx,
            first_observed_step,
            common.age_continuous_index,
        ],
        dtype=np.float32,
    )
    baseline_age_raw = (
        age_first - TIME_STEP_YEARS[first_observed_step]
    ).astype(np.float32)
    if not np.isfinite(baseline_age_raw).all():
        raise ValueError("基线年龄存在缺失或非法值。")

    age_mean = float(
        preprocessor["scaler"].mean_[common.age_continuous_index]
    )
    age_scale = float(
        preprocessor["scaler"].scale_[common.age_continuous_index]
    )
    if (
        not np.isfinite(age_mean)
        or not np.isfinite(age_scale)
        or age_scale <= 0
    ):
        raise ValueError("年龄标准化参数无效。")

    base_dynamic = np.asarray(
        x_development[:, :, dynamic_indices],
        dtype=np.float32,
    )
    lab_raw_dev = np.asarray(
        common.continuous_raw[
            common.development_idx,
            :,
            :,
        ][:, :, common.lab_continuous_indices],
        dtype=np.float32,
    )
    enhanced_dynamic, enhanced_names = build_enhanced_dynamic_array(
        base_dynamic=base_dynamic,
        lab_raw_development=lab_raw_dev,
        sequence_row_mask=common.sequence_row_mask,
    )

    train_patient_mask = common.development_fold_id != fold_id
    validation_patient_mask = common.development_fold_id == fold_id

    train_sample_pairs = np.argwhere(
        common.prediction_origin_mask
        & train_patient_mask[:, None]
    ).astype(np.int32)
    validation_sample_pairs = np.argwhere(
        common.prediction_origin_mask
        & validation_patient_mask[:, None]
    ).astype(np.int32)

    train_long_idx = common.long_row_index_map[
        train_sample_pairs[:, 0],
        train_sample_pairs[:, 1],
    ].astype(np.int32)
    validation_long_idx = common.long_row_index_map[
        validation_sample_pairs[:, 0],
        validation_sample_pairs[:, 1],
    ].astype(np.int32)

    if np.any(train_long_idx < 0) or np.any(validation_long_idx < 0):
        raise ValueError(f"第{fold_id}折存在无效长格式行号。")
    if np.intersect1d(train_long_idx, validation_long_idx).size:
        raise ValueError(f"第{fold_id}折训练和验证记录重叠。")

    train_dataset = EnhancedPatientLandmarkDataset(
        sample_pairs=train_sample_pairs,
        enhanced_dynamic=enhanced_dynamic,
        sequence_mask=common.sequence_row_mask,
        static_baseline=static_baseline,
        baseline_age_raw=baseline_age_raw,
        event_matrix=common.future_event_matrix,
        at_risk_mask=common.future_at_risk_mask,
        age_mean=age_mean,
        age_scale=age_scale,
        history_mode=HISTORY_MODE,
    )
    validation_dataset = EnhancedPatientLandmarkDataset(
        sample_pairs=validation_sample_pairs,
        enhanced_dynamic=enhanced_dynamic,
        sequence_mask=common.sequence_row_mask,
        static_baseline=static_baseline,
        baseline_age_raw=baseline_age_raw,
        event_matrix=common.future_event_matrix,
        at_risk_mask=common.future_at_risk_mask,
        age_mean=age_mean,
        age_scale=age_scale,
        history_mode=HISTORY_MODE,
    )

    return {
        "fold_id": int(fold_id),
        "train_dataset": train_dataset,
        "validation_dataset": validation_dataset,
        "train_sample_pairs": train_sample_pairs,
        "validation_sample_pairs": validation_sample_pairs,
        "train_long_idx": train_long_idx,
        "validation_long_idx": validation_long_idx,
        "static_features": static_features,
        "enhanced_dynamic_features": enhanced_names,
    }



def build_history_variant_audit(
    common: CommonData,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    对所有development有效patient-landmark origin审计history variant实际使用的时间行。
    不读取锁定测试集。
    """

    rows = []

    valid_pairs = np.argwhere(
        common.prediction_origin_mask
    ).astype(
        np.int32
    )

    for (
        patient_local,
        landmark_index,
    ) in valid_pairs:
        patient_local = int(
            patient_local
        )
        landmark_index = int(
            landmark_index
        )
        landmark_bin = int(
            LANDMARK_BINS[
                landmark_index
            ]
        )

        original_mask = np.asarray(
            common.sequence_row_mask[
                patient_local
            ],
            dtype=bool,
        )

        eligible = np.where(
            original_mask
            & (
                np.arange(
                    EXPECTED_HISTORY_STEP_N
                )
                <= landmark_bin
            )
        )[0]

        if len(
            eligible
        ) == 0:
            raise ValueError(
                "有效origin在Landmark前没有历史行："
                f"patient={patient_local}, "
                f"landmark={LANDMARK_MONTHS[landmark_index]}"
            )

        first_step = int(
            eligible[
                0
            ]
        )
        last_step = int(
            eligible[
                -1
            ]
        )

        if HISTORY_MODE == "baseline_only":
            selected_step = first_step
        elif HISTORY_MODE == "current_only":
            selected_step = last_step
        else:
            selected_step = last_step

        rows.append(
            {
                "patient_local": (
                    patient_local
                ),
                "landmark_index": (
                    landmark_index
                ),
                "landmark_month": int(
                    LANDMARK_MONTHS[
                        landmark_index
                    ]
                ),
                "history_mode": (
                    HISTORY_MODE
                ),
                "first_available_step": (
                    first_step
                ),
                "first_available_month": int(
                    first_step
                    * 6
                ),
                "last_available_step": (
                    last_step
                ),
                "last_available_month": int(
                    last_step
                    * 6
                ),
                "selected_step": (
                    selected_step
                ),
                "selected_history_month": int(
                    selected_step
                    * 6
                ),
                "lag_from_landmark_month": int(
                    LANDMARK_MONTHS[
                        landmark_index
                    ]
                    - selected_step
                    * 6
                ),
                "exact_landmark_row_available": bool(
                    last_step
                    == landmark_bin
                ),
                "baseline_0m_row_available": bool(
                    original_mask[
                        0
                    ]
                ),
                "active_history_row_n_full": int(
                    len(
                        eligible
                    )
                ),
                "active_history_row_n_variant": (
                    1
                    if HISTORY_MODE
                    != "full_history"
                    else int(
                        len(
                            eligible
                        )
                    )
                ),
            }
        )

    origin_df = pd.DataFrame(
        rows
    )

    summary_df = (
        origin_df.groupby(
            [
                "history_mode",
                "landmark_index",
                "landmark_month",
            ],
            as_index=False,
        )
        .agg(
            origin_n=(
                "patient_local",
                "size",
            ),
            baseline_0m_available_fraction=(
                "baseline_0m_row_available",
                "mean",
            ),
            exact_landmark_row_available_fraction=(
                "exact_landmark_row_available",
                "mean",
            ),
            selected_history_month_mean=(
                "selected_history_month",
                "mean",
            ),
            selected_history_month_median=(
                "selected_history_month",
                "median",
            ),
            lag_from_landmark_month_mean=(
                "lag_from_landmark_month",
                "mean",
            ),
            lag_from_landmark_month_median=(
                "lag_from_landmark_month",
                "median",
            ),
            lag_from_landmark_month_max=(
                "lag_from_landmark_month",
                "max",
            ),
            full_history_active_row_n_mean=(
                "active_history_row_n_full",
                "mean",
            ),
        )
    )

    return (
        origin_df,
        summary_df,
    )


# =============================================================================
# 6. 强化LSTM模型
# =============================================================================

class HybridAttentionLSTMSurvival(nn.Module):
    def __init__(
        self,
        dynamic_n: int,
        static_n: int,
        hidden_size: int,
        num_layers: int,
        dropout: float,
        projection_size: int,
        bidirectional: bool,
        pooling_mode: str,
        static_hidden: int,
        summary_hidden: int,
        horizon_embed_dim: int,
        future_n: int,
    ):
        super().__init__()

        self.dynamic_n = int(dynamic_n)
        self.static_n = int(static_n)
        self.hidden_size = int(hidden_size)
        self.num_layers = int(num_layers)
        self.bidirectional = bool(bidirectional)
        self.pooling_mode = str(pooling_mode)
        self.future_n = int(future_n)
        self.direction_n = 2 if self.bidirectional else 1
        self.representation_n = self.hidden_size * self.direction_n

        if self.pooling_mode not in {
            "last_attention",
            "last_attention_mean",
        }:
            raise ValueError("pooling_mode无效。")

        self.input_encoder = nn.Sequential(
            nn.LayerNorm(self.dynamic_n + 2),
            nn.Linear(self.dynamic_n + 2, int(projection_size)),
            nn.SiLU(),
            nn.Dropout(float(dropout)),
        )

        self.forward_cells = nn.ModuleList(
            [
                nn.LSTMCell(
                    input_size=(
                        int(projection_size)
                        if layer_index == 0
                        else self.hidden_size
                    ),
                    hidden_size=self.hidden_size,
                )
                for layer_index in range(self.num_layers)
            ]
        )
        if self.bidirectional:
            self.backward_cells = nn.ModuleList(
                [
                    nn.LSTMCell(
                        input_size=(
                            int(projection_size)
                            if layer_index == 0
                            else self.hidden_size
                        ),
                        hidden_size=self.hidden_size,
                    )
                    for layer_index in range(self.num_layers)
                ]
            )
        else:
            self.backward_cells = None

        self.recurrent_dropout = nn.Dropout(float(dropout))

        self.attention_hidden = nn.Linear(
            self.representation_n,
            self.representation_n,
            bias=False,
        )
        self.attention_query = nn.Linear(
            self.representation_n,
            self.representation_n,
            bias=False,
        )
        self.attention_score = nn.Linear(
            self.representation_n,
            1,
            bias=False,
        )

        self.static_encoder = nn.Sequential(
            nn.LayerNorm(self.static_n),
            nn.Linear(self.static_n, int(static_hidden)),
            nn.SiLU(),
            nn.Dropout(float(dropout)),
        )

        self.dynamic_summary_encoder = nn.Sequential(
            nn.LayerNorm(self.dynamic_n * 2),
            nn.Linear(self.dynamic_n * 2, int(summary_hidden)),
            nn.SiLU(),
            nn.Dropout(float(dropout)),
        )

        recurrent_context_n = self.representation_n * 2
        if self.pooling_mode == "last_attention_mean":
            recurrent_context_n += self.representation_n

        context_n = (
            recurrent_context_n
            + int(static_hidden)
            + int(summary_hidden)
            + 2
        )
        self.context_encoder = nn.Sequential(
            nn.LayerNorm(context_n),
            nn.Linear(context_n, self.hidden_size),
            nn.SiLU(),
            nn.Dropout(float(dropout)),
        )

        self.horizon_embedding = nn.Embedding(
            self.future_n,
            int(horizon_embed_dim),
        )
        head_input_n = self.hidden_size + int(horizon_embed_dim)
        head_hidden_n = max(32, self.hidden_size // 2)
        self.hazard_head = nn.Sequential(
            nn.LayerNorm(head_input_n),
            nn.Linear(head_input_n, head_hidden_n),
            nn.SiLU(),
            nn.Dropout(float(dropout)),
            nn.Linear(head_hidden_n, 1),
        )
        self.interval_bias = nn.Parameter(
            torch.zeros(self.future_n, dtype=torch.float32)
        )

    def _run_direction(
        self,
        encoded_sequence: torch.Tensor,
        active_mask: torch.Tensor,
        cells: nn.ModuleList,
        reverse: bool,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch_n, time_n, _ = encoded_sequence.shape
        hidden = [
            torch.zeros(
                batch_n,
                self.hidden_size,
                dtype=encoded_sequence.dtype,
                device=encoded_sequence.device,
            )
            for _ in range(self.num_layers)
        ]
        cell = [torch.zeros_like(hidden[0]) for _ in range(self.num_layers)]
        history = [None] * time_n

        indices = range(time_n - 1, -1, -1) if reverse else range(time_n)
        for step_index in indices:
            step_active = active_mask[:, step_index].unsqueeze(1)
            layer_input = encoded_sequence[:, step_index, :]

            for layer_index, recurrent_cell in enumerate(cells):
                candidate_hidden, candidate_cell = recurrent_cell(
                    layer_input,
                    (hidden[layer_index], cell[layer_index]),
                )
                hidden[layer_index] = torch.where(
                    step_active,
                    candidate_hidden,
                    hidden[layer_index],
                )
                cell[layer_index] = torch.where(
                    step_active,
                    candidate_cell,
                    cell[layer_index],
                )
                layer_input = hidden[layer_index]
                if layer_index < self.num_layers - 1:
                    layer_input = self.recurrent_dropout(layer_input)

            history[step_index] = hidden[-1]

        return torch.stack(history, dim=1), hidden[-1]

    def forward(
        self,
        dynamic_sequence: torch.Tensor,
        row_mask: torch.Tensor,
        static_baseline: torch.Tensor,
        age_at_landmark: torch.Tensor,
        landmark_normalized: torch.Tensor,
        landmark_bin: torch.Tensor,
    ) -> torch.Tensor:
        batch_n, time_n, dynamic_n = dynamic_sequence.shape
        if dynamic_n != self.dynamic_n:
            raise ValueError("模型收到的动态特征数不正确。")

        step_indices = torch.arange(
            time_n,
            device=dynamic_sequence.device,
        )
        active_mask = (
            row_mask
            & (
                step_indices[None, :]
                <= landmark_bin[:, None]
            )
        )
        if (~active_mask.any(dim=1)).any():
            raise ValueError("部分样本在Landmark前没有有效历史行。")

        time_normalized = (
            step_indices.float()
            / float(max(time_n - 1, 1))
        ).expand(batch_n, time_n)

        previous_active = torch.full(
            (batch_n,),
            -1,
            dtype=torch.long,
            device=dynamic_sequence.device,
        )
        gap_values = []
        for step_index in range(time_n):
            current_active = active_mask[:, step_index]
            gap = torch.where(
                current_active & (previous_active >= 0),
                (
                    step_index - previous_active
                ).float()
                / float(max(time_n - 1, 1)),
                torch.zeros(
                    batch_n,
                    dtype=dynamic_sequence.dtype,
                    device=dynamic_sequence.device,
                ),
            )
            gap_values.append(gap)
            previous_active = torch.where(
                current_active,
                torch.full_like(previous_active, step_index),
                previous_active,
            )
        gap_normalized = torch.stack(gap_values, dim=1)

        encoded_sequence = self.input_encoder(
            torch.cat(
                [
                    dynamic_sequence,
                    time_normalized.unsqueeze(2),
                    gap_normalized.unsqueeze(2),
                ],
                dim=2,
            )
        )

        forward_history, forward_final = self._run_direction(
            encoded_sequence,
            active_mask,
            self.forward_cells,
            reverse=False,
        )

        if self.bidirectional:
            backward_history, backward_final = self._run_direction(
                encoded_sequence,
                active_mask,
                self.backward_cells,
                reverse=True,
            )
            recurrent_history = torch.cat(
                [forward_history, backward_history],
                dim=2,
            )
            final_hidden = torch.cat(
                [forward_final, backward_final],
                dim=1,
            )
        else:
            recurrent_history = forward_history
            final_hidden = forward_final

        attention_logits = self.attention_score(
            torch.tanh(
                self.attention_hidden(recurrent_history)
                + self.attention_query(final_hidden).unsqueeze(1)
            )
        ).squeeze(2)
        attention_logits = attention_logits.masked_fill(
            ~active_mask,
            -1e4,
        )
        attention_weight = torch.softmax(attention_logits, dim=1)
        attention_pool = torch.sum(
            recurrent_history * attention_weight.unsqueeze(2),
            dim=1,
        )

        active_float = active_mask.unsqueeze(2).to(dynamic_sequence.dtype)
        active_count = active_float.sum(dim=1).clamp_min(1.0)
        recurrent_mean = (
            recurrent_history * active_float
        ).sum(dim=1) / active_count

        dynamic_mean = (
            dynamic_sequence * active_float
        ).sum(dim=1) / active_count

        last_position = (
            active_mask.long()
            * (step_indices[None, :] + 1)
        ).argmax(dim=1)
        batch_index = torch.arange(
            batch_n,
            device=dynamic_sequence.device,
        )
        dynamic_last = dynamic_sequence[
            batch_index,
            last_position,
            :,
        ]

        static_encoded = self.static_encoder(static_baseline)
        summary_encoded = self.dynamic_summary_encoder(
            torch.cat([dynamic_last, dynamic_mean], dim=1)
        )

        recurrent_parts = [final_hidden, attention_pool]
        if self.pooling_mode == "last_attention_mean":
            recurrent_parts.append(recurrent_mean)

        context = self.context_encoder(
            torch.cat(
                [
                    *recurrent_parts,
                    static_encoded,
                    summary_encoded,
                    age_at_landmark.unsqueeze(1),
                    landmark_normalized.unsqueeze(1),
                ],
                dim=1,
            )
        )

        horizon_index = torch.arange(
            self.future_n,
            device=dynamic_sequence.device,
        )
        horizon_embedding = self.horizon_embedding(
            horizon_index
        ).unsqueeze(0).expand(batch_n, -1, -1)
        context_expanded = context.unsqueeze(1).expand(
            -1,
            self.future_n,
            -1,
        )
        head_input = torch.cat(
            [context_expanded, horizon_embedding],
            dim=2,
        )
        logits = self.hazard_head(head_input).squeeze(2)
        return logits + self.interval_bias.unsqueeze(0)


# =============================================================================
# 7. Landmark平衡生存损失
# =============================================================================

class LandmarkBalancedSurvivalLoss(nn.Module):
    def __init__(
        self,
        positive_weight: float,
        focal_gamma: float,
        auxiliary_5y_weight: float,
        ranking_weight: float,
        smoothness_weight: float,
    ):
        super().__init__()
        self.register_buffer(
            "positive_weight",
            torch.tensor(float(positive_weight), dtype=torch.float32),
        )
        self.focal_gamma = float(focal_gamma)
        self.auxiliary_5y_weight = float(auxiliary_5y_weight)
        self.ranking_weight = float(ranking_weight)
        self.smoothness_weight = float(smoothness_weight)

    @staticmethod
    def _landmark_equal_mean(
        sample_loss: torch.Tensor,
        landmark_index: torch.Tensor,
    ) -> torch.Tensor:
        landmark_losses = []
        for landmark_value in range(EXPECTED_LANDMARK_N):
            mask = landmark_index == landmark_value
            if mask.any():
                landmark_losses.append(sample_loss[mask].mean())
        if not landmark_losses:
            raise ValueError("当前批次没有有效Landmark。")
        return torch.stack(landmark_losses).mean()

    def forward(
        self,
        logits: torch.Tensor,
        event_target: torch.Tensor,
        at_risk_mask: torch.Tensor,
        landmark_index: torch.Tensor,
    ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        logits_float = logits.float()
        target = event_target.float()
        risk_mask = at_risk_mask.float()

        interval_loss = torch.nn.functional.binary_cross_entropy_with_logits(
            logits_float,
            target,
            reduction="none",
            pos_weight=self.positive_weight,
        )

        if self.focal_gamma > 0:
            probability = torch.sigmoid(logits_float)
            p_t = (
                target * probability
                + (1.0 - target) * (1.0 - probability)
            )
            interval_loss = interval_loss * (
                1.0 - p_t
            ).pow(self.focal_gamma)

        sample_interval_n = risk_mask.sum(dim=1).clamp_min(1.0)
        sample_survival_loss = (
            interval_loss * risk_mask
        ).sum(dim=1) / sample_interval_n
        survival_loss = self._landmark_equal_mean(
            sample_survival_loss,
            landmark_index,
        )

        hazard = torch.sigmoid(logits_float)
        risk_5y = 1.0 - torch.prod(1.0 - hazard, dim=1)
        target_5y = target.sum(dim=1).clamp(0.0, 1.0)
        known_5y = (
            (target_5y > 0.5)
            | (risk_mask[:, -1] > 0.5)
        )

        auxiliary_loss = torch.zeros(
            (),
            dtype=logits_float.dtype,
            device=logits_float.device,
        )
        if self.auxiliary_5y_weight > 0 and known_5y.any():
            # AMP安全的未来5年累计风险BCE。
            #
            # 对离散条件风险h_j：
            #   S_5y = Π_j(1-h_j)
            #        = exp[Σ_j log sigmoid(-logit_j)]
            #   R_5y = 1-S_5y
            #
            # 先计算累计风险的logit，再使用
            # binary_cross_entropy_with_logits，避免AMP下直接对概率
            # 调用binary_cross_entropy所产生的RuntimeError。
            log_survival_5y = torch.nn.functional.logsigmoid(
                -logits_float
            ).sum(dim=1)
            log_risk_5y = torch.log(
                torch.clamp(
                    -torch.expm1(log_survival_5y),
                    min=EPS,
                )
            )
            cumulative_risk_logit_5y = (
                log_risk_5y - log_survival_5y
            )

            auxiliary_sample = (
                torch.nn.functional.binary_cross_entropy_with_logits(
                    cumulative_risk_logit_5y[known_5y],
                    target_5y[known_5y],
                    reduction="none",
                )
            )
            auxiliary_loss = self._landmark_equal_mean(
                auxiliary_sample,
                landmark_index[known_5y],
            )

        ranking_loss = torch.zeros_like(auxiliary_loss)
        if self.ranking_weight > 0:
            case_mask = target_5y > 0.5
            control_mask = (
                (target_5y <= 0.5)
                & (risk_mask[:, -1] > 0.5)
            )
            if case_mask.any() and control_mask.any():
                case_risk = risk_5y[case_mask]
                control_risk = risk_5y[control_mask]
                pairwise_margin = (
                    case_risk[:, None]
                    - control_risk[None, :]
                )
                ranking_loss = torch.nn.functional.softplus(
                    -pairwise_margin / 0.10
                ).mean()

        smoothness_loss = torch.zeros_like(auxiliary_loss)
        if self.smoothness_weight > 0:
            smoothness_loss = (
                logits_float[:, 1:]
                - logits_float[:, :-1]
            ).pow(2).mean()

        total = (
            survival_loss
            + self.auxiliary_5y_weight * auxiliary_loss
            + self.ranking_weight * ranking_loss
            + self.smoothness_weight * smoothness_loss
        )
        parts = {
            "survival_loss": survival_loss.detach(),
            "auxiliary_5y_loss": auxiliary_loss.detach(),
            "ranking_loss": ranking_loss.detach(),
            "smoothness_loss": smoothness_loss.detach(),
        }
        return total, parts


# =============================================================================
# 8. DataLoader
# =============================================================================

def move_batch_to_device(
    batch: dict[str, torch.Tensor],
    device: torch.device,
) -> dict[str, torch.Tensor]:
    return {
        "dynamic_sequence": batch["dynamic_sequence"].to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        ),
        "row_mask": batch["row_mask"].to(
            device=device,
            dtype=torch.bool,
            non_blocking=True,
        ),
        "static_baseline": batch["static_baseline"].to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        ),
        "age_at_landmark": batch["age_at_landmark"].to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        ),
        "landmark_normalized": batch["landmark_normalized"].to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        ),
        "landmark_bin": batch["landmark_bin"].to(
            device=device,
            dtype=torch.long,
            non_blocking=True,
        ),
        "event_target": batch["event_target"].to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        ),
        "at_risk_mask": batch["at_risk_mask"].to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        ),
        "patient_local": batch["patient_local"],
        "landmark_index": batch["landmark_index"].to(
            device=device,
            dtype=torch.long,
            non_blocking=True,
        ),
    }


def create_data_loaders(
    fold_data: dict[str, Any],
    batch_size: int,
    seed: int,
    device: torch.device,
) -> tuple[DataLoader, DataLoader]:
    generator = torch.Generator()
    generator.manual_seed(int(seed))

    train_landmarks = fold_data["train_sample_pairs"][:, 1]
    landmark_counts = np.bincount(
        train_landmarks,
        minlength=EXPECTED_LANDMARK_N,
    ).astype(np.float64)
    if np.any(landmark_counts <= 0):
        raise ValueError("训练集中存在没有样本的Landmark。")

    sample_weights = (
        1.0 / landmark_counts[train_landmarks]
    )
    sample_weights = (
        sample_weights / sample_weights.mean()
    )

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(
            sample_weights,
            dtype=torch.double,
        ),
        num_samples=len(sample_weights),
        replacement=True,
        generator=generator,
    )

    train_loader = DataLoader(
        fold_data["train_dataset"],
        batch_size=int(batch_size),
        sampler=sampler,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )
    validation_loader = DataLoader(
        fold_data["validation_dataset"],
        batch_size=int(batch_size),
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )
    return train_loader, validation_loader


# =============================================================================
# 9. 指标
# =============================================================================

def hazards_to_survival(
    hazard: np.ndarray,
) -> np.ndarray:
    hazard = np.clip(
        np.asarray(hazard, dtype=np.float64),
        EPS,
        1.0 - EPS,
    )
    return np.cumprod(1.0 - hazard, axis=1).astype(np.float32)


def compute_equal_weight_landmark_metrics(
    common: CommonData,
    train_long_idx: np.ndarray,
    validation_long_idx: np.ndarray,
    validation_hazard: np.ndarray,
) -> tuple[dict[str, float], pd.DataFrame]:
    validation_survival = hazards_to_survival(validation_hazard)
    validation_risk = 1.0 - validation_survival

    train_landmarks = common.landmark_month_long[train_long_idx]
    validation_landmarks = common.landmark_month_long[validation_long_idx]

    rows = []
    for landmark_month in LANDMARK_MONTHS:
        train_mask = train_landmarks == landmark_month
        validation_mask = validation_landmarks == landmark_month

        y_train = common.y_long_all[train_long_idx[train_mask]]
        y_validation = common.y_long_all[
            validation_long_idx[validation_mask]
        ]
        survival_estimate = np.asarray(
            validation_survival[validation_mask],
            dtype=np.float64,
        )
        risk_estimate = np.asarray(
            validation_risk[validation_mask],
            dtype=np.float64,
        )

        if len(y_train) == 0 or len(y_validation) == 0:
            raise ValueError(
                f"Landmark {landmark_month}个月缺少训练或验证记录。"
            )
        max_supported = min(
            float(np.max(y_train["time"])),
            float(np.max(y_validation["time"])),
        )
        if max_supported <= METRIC_TIMES[-1]:
            raise ValueError(
                f"Landmark {landmark_month}个月不足以评价至5年。"
            )

        ibs = float(
            integrated_brier_score(
                y_train,
                y_validation,
                survival_estimate,
                METRIC_TIMES,
            )
        )
        auc_values, iauc = cumulative_dynamic_auc(
            y_train,
            y_validation,
            risk_estimate,
            METRIC_TIMES,
        )
        cindex = float(
            concordance_index_ipcw(
                y_train,
                y_validation,
                risk_estimate[:, -1],
                tau=float(METRIC_TIMES[-1]),
            )[0]
        )

        rows.append(
            {
                "landmark_month": int(landmark_month),
                "train_record_n": int(train_mask.sum()),
                "validation_record_n": int(validation_mask.sum()),
                "validation_event_n": int(y_validation["event"].sum()),
                "ibs": ibs,
                "iauc": float(iauc),
                "uno_c_index_5y": cindex,
            }
        )

    table = pd.DataFrame(rows)
    summary = {
        "mean_ibs": float(table["ibs"].mean()),
        "mean_iauc": float(table["iauc"].mean()),
        "mean_uno_c": float(table["uno_c_index_5y"].mean()),
    }
    return summary, table


# =============================================================================
# 10. 训练、快照与集成
# =============================================================================

def build_model(
    parameters: dict[str, Any],
    device: torch.device,
) -> HybridAttentionLSTMSurvival:
    return HybridAttentionLSTMSurvival(
        dynamic_n=EXPECTED_ENHANCED_DYNAMIC_N,
        static_n=EXPECTED_STATIC_N,
        hidden_size=int(parameters["hidden_size"]),
        num_layers=int(parameters["num_layers"]),
        dropout=float(parameters["dropout"]),
        projection_size=int(parameters["projection_size"]),
        bidirectional=bool(parameters["bidirectional"]),
        pooling_mode=str(parameters["pooling_mode"]),
        static_hidden=int(parameters["static_hidden"]),
        summary_hidden=int(parameters["summary_hidden"]),
        horizon_embed_dim=int(parameters["horizon_embed_dim"]),
        future_n=EXPECTED_FUTURE_INTERVAL_N,
    ).to(device)


def predict_loader(
    model: nn.Module,
    loader: DataLoader,
    loss_function: LandmarkBalancedSurvivalLoss,
    device: torch.device,
    use_amp: bool,
) -> tuple[float, dict[str, np.ndarray]]:
    model.eval()
    total_loss = 0.0
    total_sample_n = 0
    hazards = []
    patient_local = []
    landmark_index = []

    with torch.no_grad():
        for raw_batch in loader:
            batch = move_batch_to_device(raw_batch, device)
            with autocast_context(device, use_amp):
                logits = model(
                    dynamic_sequence=batch["dynamic_sequence"],
                    row_mask=batch["row_mask"],
                    static_baseline=batch["static_baseline"],
                    age_at_landmark=batch["age_at_landmark"],
                    landmark_normalized=batch["landmark_normalized"],
                    landmark_bin=batch["landmark_bin"],
                )
                loss, _ = loss_function(
                    logits,
                    batch["event_target"],
                    batch["at_risk_mask"],
                    batch["landmark_index"],
                )

            batch_n = int(logits.shape[0])
            total_loss += float(loss.item()) * batch_n
            total_sample_n += batch_n
            hazards.append(
                torch.sigmoid(logits.float())
                .cpu()
                .numpy()
                .astype(np.float32)
            )
            patient_local.append(
                batch["patient_local"].numpy().astype(np.int32)
            )
            landmark_index.append(
                batch["landmark_index"]
                .cpu()
                .numpy()
                .astype(np.int8)
            )

    return (
        total_loss / max(total_sample_n, 1),
        {
            "hazard": np.concatenate(hazards, axis=0),
            "patient_local": np.concatenate(patient_local, axis=0),
            "landmark_index": np.concatenate(landmark_index, axis=0),
        },
    )


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_function: LandmarkBalancedSurvivalLoss,
    grad_scaler: Any,
    device: torch.device,
    use_amp: bool,
) -> float:
    model.train()
    total_loss = 0.0
    total_sample_n = 0

    for raw_batch in loader:
        batch = move_batch_to_device(raw_batch, device)
        optimizer.zero_grad(set_to_none=True)

        with autocast_context(device, use_amp):
            logits = model(
                dynamic_sequence=batch["dynamic_sequence"],
                row_mask=batch["row_mask"],
                static_baseline=batch["static_baseline"],
                age_at_landmark=batch["age_at_landmark"],
                landmark_normalized=batch["landmark_normalized"],
                landmark_bin=batch["landmark_bin"],
            )
            loss, _ = loss_function(
                logits,
                batch["event_target"],
                batch["at_risk_mask"],
                batch["landmark_index"],
            )

        if not torch.isfinite(loss):
            raise FloatingPointError("训练损失不是有限数。")

        grad_scaler.scale(loss).backward()
        grad_scaler.unscale_(optimizer)
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRADIENT_CLIP_NORM,
        )
        if not torch.isfinite(gradient_norm):
            raise FloatingPointError("梯度范数不是有限数。")

        grad_scaler.step(optimizer)
        grad_scaler.update()

        batch_n = int(logits.shape[0])
        total_loss += float(loss.item()) * batch_n
        total_sample_n += batch_n

    return total_loss / max(total_sample_n, 1)


def clone_state_dict(model: nn.Module) -> dict[str, torch.Tensor]:
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def ensemble_snapshot_predictions(
    model: nn.Module,
    snapshot_states: list[dict[str, torch.Tensor]],
    validation_loader: DataLoader,
    loss_function: LandmarkBalancedSurvivalLoss,
    device: torch.device,
    use_amp: bool,
) -> np.ndarray:
    hazard_predictions = []
    for state in snapshot_states:
        model.load_state_dict(state)
        _, prediction = predict_loader(
            model,
            validation_loader,
            loss_function,
            device,
            use_amp,
        )
        hazard_predictions.append(prediction["hazard"])
    return np.mean(
        np.stack(hazard_predictions, axis=0),
        axis=0,
    ).astype(np.float32)


def fit_fold_model(
    common: CommonData,
    fold_data: dict[str, Any],
    parameters: dict[str, Any],
    seed: int,
    device: torch.device,
    use_amp: bool,
    progress_callback=None,
) -> dict[str, Any]:
    set_random_seed(seed)
    train_loader, validation_loader = create_data_loaders(
        fold_data=fold_data,
        batch_size=int(parameters["batch_size"]),
        seed=seed,
        device=device,
    )

    model = build_model(parameters, device)
    loss_function = LandmarkBalancedSurvivalLoss(
        positive_weight=float(parameters["positive_weight"]),
        focal_gamma=float(parameters["focal_gamma"]),
        auxiliary_5y_weight=float(parameters["auxiliary_5y_weight"]),
        ranking_weight=float(parameters["ranking_weight"]),
        smoothness_weight=float(parameters["smoothness_weight"]),
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(parameters["learning_rate"]),
        weight_decay=float(parameters["weight_decay"]),
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_REDUCE_FACTOR,
        patience=LR_REDUCE_PATIENCE,
        min_lr=MIN_LEARNING_RATE,
    )
    grad_scaler = make_grad_scaler(enabled=use_amp)

    top_snapshots: list[dict[str, Any]] = []
    best_monitor_ibs = math.inf
    no_improvement_n = 0
    history_rows = []
    start_time = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            loss_function,
            grad_scaler,
            device,
            use_amp,
        )
        validation_loss, prediction = predict_loader(
            model,
            validation_loader,
            loss_function,
            device,
            use_amp,
        )
        metric_summary, _ = compute_equal_weight_landmark_metrics(
            common=common,
            train_long_idx=fold_data["train_long_idx"],
            validation_long_idx=fold_data["validation_long_idx"],
            validation_hazard=prediction["hazard"],
        )

        validation_ibs = metric_summary["mean_ibs"]
        validation_iauc = metric_summary["mean_iauc"]
        validation_c = metric_summary["mean_uno_c"]
        scheduler.step(validation_ibs)

        snapshot = {
            "epoch": int(epoch),
            "ibs": float(validation_ibs),
            "iauc": float(validation_iauc),
            "uno_c": float(validation_c),
            "state_dict": clone_state_dict(model),
        }
        top_snapshots.append(snapshot)
        top_snapshots.sort(
            key=lambda item: (
                item["ibs"],
                -item["iauc"],
                -item["uno_c"],
            )
        )
        top_snapshots = top_snapshots[:TOP_SNAPSHOT_N]

        current_lr = float(optimizer.param_groups[0]["lr"])
        history_rows.append(
            {
                "epoch": int(epoch),
                "train_loss": float(train_loss),
                "validation_loss": float(validation_loss),
                "validation_mean_ibs": float(validation_ibs),
                "validation_mean_iauc": float(validation_iauc),
                "validation_mean_uno_c": float(validation_c),
                "learning_rate": current_lr,
                "elapsed_seconds": float(time.time() - start_time),
            }
        )

        if progress_callback is not None:
            progress_callback(
                f"Epoch {epoch}/{MAX_EPOCHS} | "
                f"train={train_loss:.6f} | "
                f"val={validation_loss:.6f} | "
                f"IBS={validation_ibs:.6f} | "
                f"iAUC={validation_iauc:.6f} | "
                f"UnoC={validation_c:.6f} | "
                f"lr={current_lr:.2e}"
            )

        improved = (
            validation_ibs
            < best_monitor_ibs - EARLY_STOPPING_MIN_DELTA
        )
        if improved:
            best_monitor_ibs = float(validation_ibs)
            no_improvement_n = 0
        else:
            no_improvement_n += 1

        if (
            epoch >= MIN_EPOCHS
            and no_improvement_n >= EARLY_STOPPING_PATIENCE
        ):
            break

    if not top_snapshots:
        raise RuntimeError("没有保存任何有效模型快照。")

    ensemble_candidates = []
    sorted_snapshots = sorted(
        top_snapshots,
        key=lambda item: (
            item["ibs"],
            -item["iauc"],
            -item["uno_c"],
        ),
    )
    for snapshot_n in range(1, len(sorted_snapshots) + 1):
        selected_states = [
            item["state_dict"]
            for item in sorted_snapshots[:snapshot_n]
        ]
        hazard = ensemble_snapshot_predictions(
            model=model,
            snapshot_states=selected_states,
            validation_loader=validation_loader,
            loss_function=loss_function,
            device=device,
            use_amp=use_amp,
        )
        summary, table = compute_equal_weight_landmark_metrics(
            common=common,
            train_long_idx=fold_data["train_long_idx"],
            validation_long_idx=fold_data["validation_long_idx"],
            validation_hazard=hazard,
        )
        ensemble_candidates.append(
            {
                "snapshot_n": int(snapshot_n),
                "hazard": hazard,
                "summary": summary,
                "table": table,
                "states": selected_states,
                "epochs": [
                    int(item["epoch"])
                    for item in sorted_snapshots[:snapshot_n]
                ],
            }
        )

    ensemble_candidates.sort(
        key=lambda item: (
            item["summary"]["mean_ibs"],
            -item["summary"]["mean_iauc"],
            -item["summary"]["mean_uno_c"],
        )
    )
    best_ensemble = ensemble_candidates[0]
    hazard = best_ensemble["hazard"]
    survival = hazards_to_survival(hazard)
    risk = (1.0 - survival).astype(np.float32)

    if not np.isfinite(hazard).all():
        raise ValueError("验证条件风险存在NaN或无穷值。")
    if np.any((hazard < 0) | (hazard > 1)):
        raise ValueError("验证条件风险超出0～1。")
    if int(np.sum(np.diff(risk, axis=1) < -1e-7)) != 0:
        raise ValueError("验证累计风险不单调。")

    model_parameter_n = int(
        sum(parameter.numel() for parameter in model.parameters())
    )

    result = {
        "model": model,
        "snapshot_state_dicts": best_ensemble["states"],
        "snapshot_epochs": best_ensemble["epochs"],
        "selected_snapshot_n": best_ensemble["snapshot_n"],
        "history": pd.DataFrame(history_rows),
        "hazard": hazard,
        "survival": survival,
        "risk": risk,
        "metric_summary": best_ensemble["summary"],
        "landmark_metrics": best_ensemble["table"],
        "patient_local": fold_data["validation_sample_pairs"][:, 0].astype(
            np.int32
        ),
        "landmark_index": fold_data["validation_sample_pairs"][:, 1].astype(
            np.int8
        ),
        "parameter_n": model_parameter_n,
        "elapsed_seconds": float(time.time() - start_time),
    }

    del train_loader
    del validation_loader
    del optimizer
    del scheduler
    del grad_scaler
    del loss_function
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return result


# =============================================================================
# 11. 参数标准化
# =============================================================================

def normalize_parameters(parameters: dict[str, Any]) -> dict[str, Any]:
    return {
        "hidden_size": int(parameters["hidden_size"]),
        "num_layers": int(parameters["num_layers"]),
        "dropout": float(parameters["dropout"]),
        "projection_size": int(parameters["projection_size"]),
        "bidirectional": bool(parameters["bidirectional"]),
        "pooling_mode": str(parameters["pooling_mode"]),
        "static_hidden": int(parameters["static_hidden"]),
        "summary_hidden": int(parameters["summary_hidden"]),
        "horizon_embed_dim": int(parameters["horizon_embed_dim"]),
        "learning_rate": float(parameters["learning_rate"]),
        "weight_decay": float(parameters["weight_decay"]),
        "batch_size": int(parameters["batch_size"]),
        "positive_weight": float(parameters["positive_weight"]),
        "focal_gamma": float(parameters["focal_gamma"]),
        "auxiliary_5y_weight": float(parameters["auxiliary_5y_weight"]),
        "ranking_weight": float(parameters["ranking_weight"]),
        "smoothness_weight": float(parameters["smoothness_weight"]),
    }


def default_device() -> tuple[torch.device, bool]:
    cuda_available = torch.cuda.is_available()
    if REQUIRE_CUDA and not cuda_available:
        raise RuntimeError("未检测到CUDA，本步骤要求GPU运行。")
    device = torch.device("cuda" if cuda_available else "cpu")
    use_amp = bool(USE_AMP and device.type == "cuda")
    torch.set_num_threads(min(8, os.cpu_count() or 1))
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return device, use_amp


# =============================================================================
# Step 10E专用依赖
# =============================================================================

from datetime import datetime

# =============================================================================
# 1. 路径与正式集成配置
# =============================================================================

TUNING_DIR = (
    PROJECT_DIR
    / "rolling_5y_step10d_lstm_v2_tune_resume"
    / "tuning"
)
TUNING_SUMMARY_FILE = (
    TUNING_DIR / "lstm_v2_tuning_summary.json"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / (
        "rolling_5y_step13a_lstm_v2_"
        f"{HISTORY_MODE}_oof_fixed_hyperparameters"
    )
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = (
    OUTPUT_DIR
    / f"step13a_{HISTORY_MODE}_live_progress.log"
)

FINAL_SEED_ENSEMBLE_N = int(
    os.getenv("CKD_LSTM_V2_FINAL_SEEDS", "2")
)
BASE_SEED = int(
    os.getenv("CKD_LSTM_V2_FINAL_BASE_SEED", "20260830")
)

if FINAL_SEED_ENSEMBLE_N < 1:
    raise ValueError("FINAL_SEED_ENSEMBLE_N必须为正整数。")
if not TUNING_SUMMARY_FILE.exists():
    raise FileNotFoundError(
        "未找到LSTM-v2调参摘要：\n"
        f"{TUNING_SUMMARY_FILE}\n"
        "请先运行Step 10D。"
    )


# =============================================================================
# 2. 日志与折级保存
# =============================================================================

def progress_print(message: str) -> None:
    line = (
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        f"{message}"
    )
    print(line, flush=True)
    with LOG_FILE.open("a", encoding="utf-8") as file:
        file.write(line + "\n")
        file.flush()


def fold_required_files(fold_dir: Path) -> list[Path]:
    return [
        fold_dir / "completed.json",
        fold_dir / "validation_hazard.npy",
        fold_dir / "validation_survival.npy",
        fold_dir / "validation_risk.npy",
        fold_dir / "validation_long_idx.npy",
        fold_dir / "validation_patient_local.npy",
        fold_dir / "validation_landmark_index.npy",
        fold_dir / "fold_summary.json",
        fold_dir / "landmark_metrics.csv",
    ]


def load_completed_fold(
    fold_dir: Path,
) -> dict[str, Any] | None:
    required = fold_required_files(fold_dir)
    if not all(path.exists() for path in required):
        return None

    marker = json.loads(
        (fold_dir / "completed.json").read_text(encoding="utf-8")
    )
    if not bool(marker.get("completed", False)):
        return None

    return {
        "hazard": np.load(
            fold_dir / "validation_hazard.npy"
        ).astype(np.float32),
        "survival": np.load(
            fold_dir / "validation_survival.npy"
        ).astype(np.float32),
        "risk": np.load(
            fold_dir / "validation_risk.npy"
        ).astype(np.float32),
        "validation_long_idx": np.load(
            fold_dir / "validation_long_idx.npy"
        ).astype(np.int32),
        "patient_local": np.load(
            fold_dir / "validation_patient_local.npy"
        ).astype(np.int32),
        "landmark_index": np.load(
            fold_dir / "validation_landmark_index.npy"
        ).astype(np.int8),
        "summary": json.loads(
            (fold_dir / "fold_summary.json").read_text(
                encoding="utf-8"
            )
        ),
        "metrics": pd.read_csv(
            fold_dir / "landmark_metrics.csv",
            encoding="utf-8-sig",
        ),
    }


# =============================================================================
# 3. 主流程
# =============================================================================

def run() -> None:
    tuning_summary = json.loads(
        TUNING_SUMMARY_FILE.read_text(encoding="utf-8")
    )
    if bool(tuning_summary.get("locked_test_read", False)):
        raise ValueError("调参摘要显示读取过锁定测试集。")

    selected_trial = int(
        tuning_summary["selected_trial_number"]
    )
    parameters = normalize_parameters(
        tuning_summary["selected_parameters"]
    )

    device, use_amp = default_device()
    common = load_common_data()

    (
        variant_origin_audit,
        variant_audit_summary,
    ) = build_history_variant_audit(
        common
    )

    variant_origin_audit.to_csv(
        OUTPUT_DIR
        / "history_variant_origin_audit.csv",
        index=False,
        encoding="utf-8-sig",
    )
    variant_audit_summary.to_csv(
        OUTPUT_DIR
        / "history_variant_audit_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print("\n" + "=" * 96)
    print(
        "Step 13A：LSTM-v2 longitudinal incremental value retraining | "
        f"{HISTORY_MODE}"
    )
    print("=" * 96)
    print("History variant：", HISTORY_MODE)
    print("Current-state动态通道数：", CURRENT_STATE_DYNAMIC_N)
    print("选中Full-history冻结Trial：", selected_trial)
    print(
        "调参平均IBS：",
        f"{float(tuning_summary['selected_mean_ibs']):.6f}",
    )
    print(
        "调参平均iAUC：",
        f"{float(tuning_summary['selected_mean_iauc']):.6f}",
    )
    print("最佳参数：", parameters)
    print("每折随机种子集成数：", FINAL_SEED_ENSEMBLE_N)
    print("运行设备：", device)
    if device.type == "cuda":
        print("GPU：", torch.cuda.get_device_name(0))
    print("锁定测试集：未读取")
    print("输出目录：", OUTPUT_DIR)
    print("\nHistory variant审计：")
    print(
        variant_audit_summary.to_string(
            index=False
        )
    )
    print("=" * 96)

    oof_hazard_long = np.full(
        (
            EXPECTED_DEVELOPMENT_VALID_ORIGIN_N,
            EXPECTED_FUTURE_INTERVAL_N,
        ),
        np.nan,
        dtype=np.float32,
    )
    assigned_long = np.zeros(
        EXPECTED_DEVELOPMENT_VALID_ORIGIN_N,
        dtype=bool,
    )

    fold_summaries = []
    fold_metric_frames = []
    total_start = time.time()

    final_static_features = None
    final_dynamic_features = None

    for fold_id in range(N_SPLITS):
        fold_dir = OUTPUT_DIR / f"fold_{fold_id}"
        fold_dir.mkdir(parents=True, exist_ok=True)

        completed = load_completed_fold(fold_dir)
        if completed is not None:
            progress_print(
                f"第{fold_id + 1}/5折已完成，直接读取检查点。"
            )
            fold_result = completed
        else:
            fold_start = time.time()
            progress_print(f"第{fold_id + 1}/5折开始。")
            fold_data = prepare_fold_data(common, fold_id)
            final_static_features = fold_data["static_features"]
            final_dynamic_features = fold_data[
                "enhanced_dynamic_features"
            ]

            seed_hazards = []
            seed_summaries = []

            for seed_position in range(FINAL_SEED_ENSEMBLE_N):
                seed = (
                    BASE_SEED
                    + fold_id * 1000
                    + seed_position * 100
                )
                progress_print(
                    f"第{fold_id + 1}/5折 | "
                    f"种子{seed_position + 1}/"
                    f"{FINAL_SEED_ENSEMBLE_N}开始 | seed={seed}"
                )

                def epoch_logger(message: str) -> None:
                    progress_print(
                        f"Fold {fold_id} | Seed {seed} | {message}"
                    )

                result = fit_fold_model(
                    common=common,
                    fold_data=fold_data,
                    parameters=parameters,
                    seed=seed,
                    device=device,
                    use_amp=use_amp,
                    progress_callback=epoch_logger,
                )

                seed_hazards.append(result["hazard"])
                seed_summary = {
                    "fold_id": int(fold_id),
                    "seed_position": int(seed_position),
                    "seed": int(seed),
                    "selected_snapshot_n": int(
                        result["selected_snapshot_n"]
                    ),
                    "snapshot_epochs": result["snapshot_epochs"],
                    "mean_ibs": float(
                        result["metric_summary"]["mean_ibs"]
                    ),
                    "mean_iauc": float(
                        result["metric_summary"]["mean_iauc"]
                    ),
                    "mean_uno_c": float(
                        result["metric_summary"]["mean_uno_c"]
                    ),
                    "parameter_n": int(result["parameter_n"]),
                    "elapsed_seconds": float(
                        result["elapsed_seconds"]
                    ),
                }
                seed_summaries.append(seed_summary)

                result["history"].to_csv(
                    fold_dir
                    / f"seed_{seed_position}_training_history.csv",
                    index=False,
                    encoding="utf-8-sig",
                )
                result["landmark_metrics"].to_csv(
                    fold_dir
                    / f"seed_{seed_position}_landmark_metrics.csv",
                    index=False,
                    encoding="utf-8-sig",
                )
                torch.save(
                    {
                        "stage": "Step13A_LSTM_v2_history_variant_OOF",
                        "history_mode": HISTORY_MODE,
                        "current_state_dynamic_n": CURRENT_STATE_DYNAMIC_N,
                        "fold_id": int(fold_id),
                        "seed": int(seed),
                        "selected_trial_number": selected_trial,
                        "parameters": parameters,
                        "snapshot_epochs": result["snapshot_epochs"],
                        "snapshot_state_dicts": (
                            result["snapshot_state_dicts"]
                        ),
                        "feature_names": {
                            "static": fold_data["static_features"],
                            "enhanced_dynamic": fold_data[
                                "enhanced_dynamic_features"
                            ],
                        },
                    },
                    fold_dir
                    / f"seed_{seed_position}_snapshot_ensemble.pt",
                )

                progress_print(
                    f"第{fold_id + 1}/5折 | "
                    f"种子{seed_position + 1}完成 | "
                    f"IBS={seed_summary['mean_ibs']:.6f} | "
                    f"iAUC={seed_summary['mean_iauc']:.6f} | "
                    f"UnoC={seed_summary['mean_uno_c']:.6f} | "
                    f"快照={seed_summary['snapshot_epochs']}"
                )

                del result
                gc.collect()
                if device.type == "cuda":
                    torch.cuda.empty_cache()

            final_hazard = np.mean(
                np.stack(seed_hazards, axis=0),
                axis=0,
            ).astype(np.float32)
            final_survival = hazards_to_survival(final_hazard)
            final_risk = (1.0 - final_survival).astype(np.float32)

            final_metric_summary, final_metric_table = (
                compute_equal_weight_landmark_metrics(
                    common=common,
                    train_long_idx=fold_data["train_long_idx"],
                    validation_long_idx=fold_data[
                        "validation_long_idx"
                    ],
                    validation_hazard=final_hazard,
                )
            )
            final_metric_table.insert(0, "fold_id", fold_id)

            fold_summary = {
                "fold_id": int(fold_id),
                "history_mode": HISTORY_MODE,
                "selected_trial_number": selected_trial,
                "parameters": parameters,
                "seed_ensemble_n": FINAL_SEED_ENSEMBLE_N,
                "seed_summaries": seed_summaries,
                "final_mean_ibs": float(
                    final_metric_summary["mean_ibs"]
                ),
                "final_mean_iauc": float(
                    final_metric_summary["mean_iauc"]
                ),
                "final_mean_uno_c": float(
                    final_metric_summary["mean_uno_c"]
                ),
                "validation_origin_n": int(len(final_hazard)),
                "elapsed_seconds": float(time.time() - fold_start),
                "locked_test_used": False,
            }

            np.save(
                fold_dir / "validation_hazard.npy",
                final_hazard,
            )
            np.save(
                fold_dir / "validation_survival.npy",
                final_survival,
            )
            np.save(
                fold_dir / "validation_risk.npy",
                final_risk,
            )
            np.save(
                fold_dir / "validation_long_idx.npy",
                fold_data["validation_long_idx"],
            )
            np.save(
                fold_dir / "validation_patient_local.npy",
                fold_data["validation_sample_pairs"][:, 0].astype(
                    np.int32
                ),
            )
            np.save(
                fold_dir / "validation_landmark_index.npy",
                fold_data["validation_sample_pairs"][:, 1].astype(
                    np.int8
                ),
            )
            final_metric_table.to_csv(
                fold_dir / "landmark_metrics.csv",
                index=False,
                encoding="utf-8-sig",
            )
            save_json(
                fold_summary,
                fold_dir / "fold_summary.json",
            )
            save_json(
                {
                    "completed": True,
                    "fold_id": int(fold_id),
                    "completed_at": datetime.now().isoformat(
                        timespec="seconds"
                    ),
                },
                fold_dir / "completed.json",
            )

            fold_result = {
                "hazard": final_hazard,
                "survival": final_survival,
                "risk": final_risk,
                "validation_long_idx": fold_data[
                    "validation_long_idx"
                ],
                "patient_local": fold_data[
                    "validation_sample_pairs"
                ][:, 0].astype(np.int32),
                "landmark_index": fold_data[
                    "validation_sample_pairs"
                ][:, 1].astype(np.int8),
                "summary": fold_summary,
                "metrics": final_metric_table,
            }

            progress_print(
                f"第{fold_id + 1}/5折完成 | "
                f"IBS={final_metric_summary['mean_ibs']:.6f} | "
                f"iAUC={final_metric_summary['mean_iauc']:.6f} | "
                f"UnoC={final_metric_summary['mean_uno_c']:.6f} | "
                f"耗时={format_duration(time.time() - fold_start)}"
            )

            del fold_data
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()

        long_idx = np.asarray(
            fold_result["validation_long_idx"],
            dtype=np.int64,
        )
        if assigned_long[long_idx].any():
            raise ValueError(f"第{fold_id}折OOF长格式位置重复赋值。")
        if fold_result["hazard"].shape != (
            len(long_idx),
            EXPECTED_FUTURE_INTERVAL_N,
        ):
            raise ValueError(f"第{fold_id}折预测形状错误。")

        oof_hazard_long[long_idx] = fold_result["hazard"]
        assigned_long[long_idx] = True
        fold_summaries.append(fold_result["summary"])

        metrics = fold_result["metrics"].copy()
        if "fold_id" not in metrics.columns:
            metrics.insert(0, "fold_id", fold_id)
        fold_metric_frames.append(metrics)

    if not assigned_long.all():
        raise ValueError(
            f"OOF仍缺少{int((~assigned_long).sum())}条长格式预测。"
        )
    if not np.isfinite(oof_hazard_long).all():
        raise ValueError("完整OOF条件风险存在NaN或无穷值。")

    oof_survival_long = hazards_to_survival(oof_hazard_long)
    oof_risk_long = (1.0 - oof_survival_long).astype(np.float32)

    patient_hazard = np.full(
        (
            EXPECTED_DEVELOPMENT_N,
            EXPECTED_LANDMARK_N,
            EXPECTED_FUTURE_INTERVAL_N,
        ),
        np.nan,
        dtype=np.float32,
    )
    patient_survival = np.full_like(patient_hazard, np.nan)
    patient_risk = np.full_like(patient_hazard, np.nan)

    valid_origin = common.long_row_index_map >= 0
    long_rows = common.long_row_index_map[valid_origin].astype(np.int64)
    patient_hazard[valid_origin] = oof_hazard_long[long_rows]
    patient_survival[valid_origin] = oof_survival_long[long_rows]
    patient_risk[valid_origin] = oof_risk_long[long_rows]

    np.save(
        OUTPUT_DIR / "lstm_v2_oof_hazard_long.npy",
        oof_hazard_long,
    )
    np.save(
        OUTPUT_DIR / "lstm_v2_oof_survival_long.npy",
        oof_survival_long,
    )
    np.save(
        OUTPUT_DIR / "lstm_v2_oof_risk_long.npy",
        oof_risk_long,
    )
    np.save(
        OUTPUT_DIR / "lstm_v2_oof_risk_score_long.npy",
        oof_risk_long[:, -1],
    )
    np.save(
        OUTPUT_DIR / "lstm_v2_oof_hazard.npy",
        patient_hazard,
    )
    np.save(
        OUTPUT_DIR / "lstm_v2_oof_survival.npy",
        patient_survival,
    )
    np.save(
        OUTPUT_DIR / "lstm_v2_oof_risk.npy",
        patient_risk,
    )
    np.save(
        OUTPUT_DIR / "lstm_v2_oof_risk_score.npy",
        patient_risk[:, :, -1],
    )

    fold_summary_table = pd.DataFrame(
        [
            {
                "fold_id": int(item["fold_id"]),
                "final_mean_ibs": float(item["final_mean_ibs"]),
                "final_mean_iauc": float(item["final_mean_iauc"]),
                "final_mean_uno_c": float(item["final_mean_uno_c"]),
                "seed_ensemble_n": int(item["seed_ensemble_n"]),
                "validation_origin_n": int(item["validation_origin_n"]),
                "elapsed_seconds": float(item["elapsed_seconds"]),
            }
            for item in fold_summaries
        ]
    ).sort_values("fold_id")
    fold_metric_table = pd.concat(
        fold_metric_frames,
        ignore_index=True,
    ).sort_values(["fold_id", "landmark_month"])

    fold_summary_table.to_csv(
        OUTPUT_DIR / "lstm_v2_fold_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )
    fold_metric_table.to_csv(
        OUTPUT_DIR / "lstm_v2_fold_landmark_metrics.csv",
        index=False,
        encoding="utf-8-sig",
    )

    if final_static_features is not None:
        feature_rows = (
            [
                {
                    "feature_group": "static",
                    "feature_name": name,
                    "history_mode": HISTORY_MODE,
                    "retained_in_single_state_variant": True,
                }
                for name in final_static_features
            ]
            + [
                {
                    "feature_group": "enhanced_dynamic",
                    "feature_name": name,
                    "history_mode": HISTORY_MODE,
                    "retained_in_single_state_variant": bool(
                        feature_index < CURRENT_STATE_DYNAMIC_N
                    ),
                }
                for feature_index, name in enumerate(
                    final_dynamic_features
                )
            ]
        )
        pd.DataFrame(feature_rows).to_csv(
            OUTPUT_DIR / "lstm_v2_feature_partition.csv",
            index=False,
            encoding="utf-8-sig",
        )

    survival_violation_n = int(
        np.sum(np.diff(oof_survival_long, axis=1) > 1e-7)
    )
    risk_violation_n = int(
        np.sum(np.diff(oof_risk_long, axis=1) < -1e-7)
    )

    final_summary = {
        "stage": "Step13A_LSTM_v2_history_variant_5fold_OOF",
        "history_mode": HISTORY_MODE,
        "history_variant_definition": (
            "full history" if HISTORY_MODE == "full_history"
            else (
                "earliest available dynamic state only; "
                "retain current-state 32 channels, zero cumulative/observed/recency/delta"
                if HISTORY_MODE == "baseline_only"
                else
                "most recent available dynamic state at/before landmark only; "
                "retain current-state 32 channels, zero cumulative/observed/recency/delta"
            )
        ),
        "hyperparameter_policy": (
            "Frozen Full-history Trial parameters; no retuning"
        ),
        "selected_trial_number": selected_trial,
        "selected_parameters": parameters,
        "seed_ensemble_n_per_fold": FINAL_SEED_ENSEMBLE_N,
        "five_fold_mean_ibs": float(
            fold_summary_table["final_mean_ibs"].mean()
        ),
        "five_fold_mean_iauc": float(
            fold_summary_table["final_mean_iauc"].mean()
        ),
        "five_fold_mean_uno_c": float(
            fold_summary_table["final_mean_uno_c"].mean()
        ),
        "oof_long_shape": list(oof_hazard_long.shape),
        "oof_patient_shape": list(patient_hazard.shape),
        "survival_monotonicity_violation_n": survival_violation_n,
        "risk_monotonicity_violation_n": risk_violation_n,
        "total_elapsed_seconds": float(time.time() - total_start),
        "locked_test_used": False,
        "output_dir": str(OUTPUT_DIR),
    }
    save_json(
        final_summary,
        OUTPUT_DIR / "lstm_v2_summary.json",
    )

    save_json(
        {
            "history_mode": HISTORY_MODE,
            "current_state_dynamic_n": CURRENT_STATE_DYNAMIC_N,
            "current_state_dynamic_features": (
                list(CURRENT_STATE_DYNAMIC_FEATURES)
                if HISTORY_MODE != "full_history"
                else list(DYNAMIC_FEATURES)
            ),
            "kept_dynamic_channel_positions": (
                list(range(CURRENT_STATE_DYNAMIC_N))
                if HISTORY_MODE != "full_history"
                else list(range(EXPECTED_ENHANCED_DYNAMIC_N))
            ),
            "zeroed_dynamic_channel_positions": (
                list(
                    range(
                        CURRENT_STATE_DYNAMIC_N,
                        EXPECTED_ENHANCED_DYNAMIC_N,
                    )
                )
                if HISTORY_MODE != "full_history"
                else []
            ),
            "same_development_risk_set_as_full_history": True,
            "same_fixed_five_folds_as_full_history": True,
            "same_frozen_hyperparameters_as_full_history": True,
            "locked_test_used": False,
        },
        OUTPUT_DIR
        / "history_variant_definition.json",
    )

    print("\n" + "=" * 96)
    print(
        "Step13A history variant五折OOF完成 | "
        f"{HISTORY_MODE}"
    )
    print("=" * 96)
    print(
        "五折平均IBS：",
        f"{final_summary['five_fold_mean_ibs']:.6f}",
    )
    print(
        "五折平均iAUC：",
        f"{final_summary['five_fold_mean_iauc']:.6f}",
    )
    print(
        "五折平均Uno C：",
        f"{final_summary['five_fold_mean_uno_c']:.6f}",
    )
    print("OOF长格式形状：", oof_hazard_long.shape)
    print("OOF患者级形状：", patient_hazard.shape)
    print("生存概率单调性违反数：", survival_violation_n)
    print("累积风险单调性违反数：", risk_violation_n)
    print("锁定测试集：未读取")
    print("总耗时：", format_duration(time.time() - total_start))
    print("输出目录：", OUTPUT_DIR)
    print("=" * 96)


if __name__ == "__main__":
    run()
